# 04. Análisis operativo y preparación de resultados del sistema de alerta temprana

El desarrollo del sistema predictivo ha permitido establecer una configuración definitiva para la detección anticipada de vuelos con riesgo de presentar un retraso de llegada igual o superior a 15 minutos. La fase de modelado concluyó con la selección de una regresión logística, la definición de una regla de decisión orientada a alcanzar una elevada cobertura de la clase positiva y la posterior evaluación del sistema sobre un período temporal externo correspondiente a enero–mayo de 2026. Este proceso proporciona el punto de partida para trasladar la evaluación desde una perspectiva estrictamente predictiva hacia una caracterización operativa de las alertas generadas.

El modelo definitivo fue entrenado sobre el período 2022–2025 después de fijar previamente sus decisiones de modelado mediante la validación temporal de 2025. La regla de decisión utiliza un umbral aproximado de 0.406588, seleccionado con un objetivo operativo de recall mínimo del 80 %. Su aplicación sobre 3,102,447 observaciones del período externo de 2026 produjo un recall de 0.7994 y una precision de 0.2527, junto con una proporción de falsas alertas de 0.7473. Estos resultados muestran la necesidad de complementar las métricas predictivas globales con un análisis que permita comprender cómo se materializan las decisiones del modelo sobre diferentes segmentos de la población.

La interpretación funcional adoptada corresponde a un sistema de alerta temprana de riesgo de retraso orientado al pasajero. En este contexto, una clasificación positiva no representa la certeza de que un vuelo sufrirá un retraso, sino la identificación anticipada de una situación de riesgo elevado de acuerdo con la información disponible antes de la salida. Esta perspectiva permite distinguir entre retrasos correctamente identificados, falsas alertas, ausencias correctas de alerta y retrasos reales que no fueron detectados anticipadamente, proporcionando una lectura operativa de los verdaderos positivos, falsos positivos, verdaderos negativos y falsos negativos.

La caracterización de estas situaciones permitirá examinar la distribución de las alertas y su comportamiento en diferentes dimensiones temporales y operativas, incluyendo meses, días de la semana, franjas horarias programadas, compañías aéreas y aeropuertos. El propósito de estas comparaciones será identificar patrones descriptivos que permitan comprender mejor las fortalezas y limitaciones del sistema, sin atribuir relaciones causales que no puedan demostrarse a partir del diseño observacional utilizado.

Esta fase mantiene una separación metodológica estricta respecto al proceso de selección del modelo. El período enero–mayo de 2026 ya fue utilizado como test externo independiente, por lo que cualquier análisis adicional realizado sobre sus observaciones tendrá carácter descriptivo y post-test. En consecuencia, los patrones identificados no se utilizarán para modificar el modelo definitivo, sus hiperparámetros, el tratamiento del desbalance, las variables predictoras ni el umbral de decisión conservando posteriormente 2026 como evidencia independiente. Los resultados podrán, en cambio, emplearse para interpretar el comportamiento del sistema y formular posibles líneas de mejora que requerirían una nueva evaluación temporal independiente.

El volumen de información disponible exige igualmente mantener los criterios de eficiencia computacional adoptados durante las fases anteriores. El procesamiento se concentrará principalmente en la población externa de 2026 y en los productos persistidos durante el modelado, priorizando la lectura selectiva de variables, la reutilización de resultados existentes y la generación de agregaciones específicamente diseñadas para responder a las preguntas analíticas planteadas. Esta estrategia permitirá evitar recomputaciones y materializaciones innecesarias y facilitará la construcción de productos compactos destinados a las fases posteriores del proyecto.

Como resultado, se generará una capa analítica situada entre el modelado predictivo y la explotación visual de sus resultados. Los productos obtenidos deberán permitir representar tanto el volumen de alertas como su calidad operativa, preservando simultáneamente información sobre el tamaño de las poblaciones analizadas. Esta consideración será especialmente relevante al comparar segmentos, dado que una tasa elevada calculada sobre un número reducido de vuelos no debe interpretarse de la misma manera que un comportamiento equivalente observado sobre una población considerablemente mayor.

La estructura propuesta mantiene una progresión desde la recuperación de los productos ya obtenidos hasta la construcción del dataset analítico, su caracterización global y segmentada y, finalmente, la preparación de productos destinados a la visualización y documentación del TFM.

## Estructura del notebook

1. Configuración y recuperación de productos del modelado.
2. Construcción del dataset analítico de alertas.
3. Caracterización global del sistema de alerta temprana.
4. Análisis temporal del comportamiento de las alertas.
5. Análisis operacional y geográfico.
6. Preparación de productos para visualización.
7. Conclusiones del análisis operativo.

## 1. Configuración y recuperación de productos del modelado

La transición desde la evaluación predictiva hacia el análisis operativo requiere establecer un entorno de trabajo capaz de recuperar de forma reproducible los productos definitivos obtenidos durante el modelado. El propósito de esta etapa no es reconstruir las decisiones adoptadas previamente, sino garantizar que los análisis posteriores utilicen exactamente el modelo, el preprocesamiento, la regla de decisión y los resultados que quedaron fijados antes de la evaluación externa.

La persistencia de estos productos permite mantener una separación efectiva entre notebooks y evita que la reproducibilidad dependa de objetos conservados temporalmente en memoria. El Notebook 04 deberá poder ejecutarse de manera independiente a partir de los archivos generados anteriormente, preservando simultáneamente la trazabilidad entre la configuración seleccionada, los resultados del test externo y las nuevas estructuras analíticas que se construirán a partir de ellos.

La configuración inicial se limitará a establecer las dependencias y rutas necesarias, mientras que la recuperación posterior incorporará el modelo definitivo, el encoder categórico, el escalador numérico, la configuración persistida y los resultados relevantes del proceso de evaluación. En esta fase no se realizará ningún nuevo ajuste del modelo, del preprocesamiento ni del umbral de decisión, ya que estos elementos constituyen entradas previamente fijadas para el análisis operativo.

La validación de estos productos se realizará de forma consolidada antes de procesar la población externa. Este criterio permitirá comprobar conjuntamente la disponibilidad y coherencia de los elementos necesarios sin fragmentar el notebook en verificaciones triviales o redundantes. Asimismo, la carga masiva de las observaciones de 2026 se reservará para la etapa específicamente destinada a construir el dataset analítico de alertas, evitando operaciones innecesarias durante la configuración.

En este bloque se abordarán progresivamente:

1. la configuración del entorno y de las rutas necesarias para recuperar los productos persistidos del proceso de modelado;
2. la recuperación del modelo definitivo, su configuración y los resultados necesarios para el análisis operativo;
3. la validación consolidada de los productos de entrada antes de construir el dataset analítico de alertas.

### 1.1 Configuración del entorno y rutas

El desarrollo del análisis operativo requiere establecer inicialmente un entorno de ejecución reproducible y localizar de forma explícita los productos generados durante las fases anteriores del TFM. Esta configuración permitirá que el Notebook 04 pueda ejecutarse de manera independiente, sin depender de objetos que permanezcan temporalmente en memoria tras la ejecución del notebook de modelado.

En este subbloque se definirán las dependencias y rutas necesarias para acceder tanto a la partición externa de datos correspondiente a enero–mayo de 2026 como a los productos persistidos del modelo definitivo. Se mantendrán las convenciones de nomenclatura y organización utilizadas previamente, evitando crear rutas alternativas para recursos que ya disponen de una localización establecida dentro del proyecto.

La configuración se limitará a preparar el entorno y comprobar posteriormente la disponibilidad de las ubicaciones necesarias. No se cargarán todavía los datos masivos del período externo ni se realizarán predicciones, transformaciones o ajustes de los componentes del modelo. Esta separación permite mantener una responsabilidad metodológica clara: la configuración establece las fuentes de entrada, mientras que su recuperación y utilización se desarrollarán en los subbloques posteriores.

In [1]:
# ---------------------------------------------------------
# 1. Importar las dependencias necesarias
# ---------------------------------------------------------

from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 2. Definir la raíz del proyecto
# ---------------------------------------------------------

project_root = Path(r"G:\My Drive\MASTER Big Data\TFM")


# ---------------------------------------------------------
# 3. Definir las rutas de los datos de modelado
# ---------------------------------------------------------

modeling_data_path = project_root / "data" / "processed" / "flights" / "modeling"

external_test_data_path = modeling_data_path / "test"


# ---------------------------------------------------------
# 4. Definir la ruta de resultados del modelado
# ---------------------------------------------------------

modeling_results_path = project_root / "results" / "modeling"


# ---------------------------------------------------------
# 5. Definir las rutas de los productos definitivos
# ---------------------------------------------------------

final_model_path = (
    modeling_results_path
    / "final_logistic_regression_2022_2025.joblib"
)

final_encoder_path = (
    modeling_results_path
    / "final_categorical_encoder_2022_2025.joblib"
)

final_scaler_path = (
    modeling_results_path
    / "final_numerical_scaler_2022_2025.joblib"
)

final_configuration_path = (
    modeling_results_path
    / "final_model_configuration.json"
)

threshold_results_path = (
    modeling_results_path
    / "threshold_selection_2025.csv"
)

validation_probabilities_path = (
    modeling_results_path
    / "selected_model_validation_probabilities_2025.npy"
)

external_test_results_path = (
    modeling_results_path
    / "external_test_results_2026.csv"
)

temporal_generalization_path = (
    modeling_results_path
    / "temporal_generalization_2025_2026.csv"
)


# ---------------------------------------------------------
# 6. Mostrar una salida mínima de configuración
# ---------------------------------------------------------

print(f"Raíz del proyecto: {project_root}")
print(f"Datos externos 2026: {external_test_data_path}")
print(f"Resultados de modelado: {modeling_results_path}")

Raíz del proyecto: G:\My Drive\MASTER Big Data\TFM
Datos externos 2026: G:\My Drive\MASTER Big Data\TFM\data\processed\flights\modeling\test
Resultados de modelado: G:\My Drive\MASTER Big Data\TFM\results\modeling


#### Interpretación

La configuración inicial del Notebook 04 se ha establecido correctamente a partir de la estructura persistida del proyecto. La raíz del TFM, la ubicación de los datos externos correspondientes al período de 2026 y el directorio que contiene los productos del modelado han quedado definidos de forma coherente y reutilizable.

Este resultado confirma que el notebook dispone de una base de rutas consistente para recuperar posteriormente el modelo definitivo, los componentes de preprocesamiento y los resultados persistidos del proceso de evaluación. La comprobación realizada en este punto es únicamente estructural y no implica todavía que todos los archivos requeridos existan o sean internamente coherentes, ya que esa validación se realizará de forma consolidada en un subbloque posterior.

La separación entre configuración, recuperación y validación evita mezclar responsabilidades y permite mantener la reproducibilidad del notebook sin depender de objetos previamente cargados en memoria.

### 1.2 Recuperación del modelo, configuración y resultados persistidos

Una vez establecidas las rutas de trabajo, el siguiente paso consiste en recuperar los productos persistidos que representan el estado definitivo del proceso de modelado. Esta recuperación garantiza que el análisis operativo se construya sobre los mismos componentes utilizados durante la evaluación externa, sin reconstruir manualmente configuraciones ni depender de objetos conservados en memoria desde el Notebook 03.

Se cargarán el modelo definitivo de regresión logística, el encoder categórico y el escalador numérico ajustados sobre el período 2022–2025, junto con la configuración final y los principales resultados persistidos relacionados con la selección del umbral, la evaluación externa de 2026 y la comparación temporal entre validación y test. Estos elementos constituyen las entradas necesarias para mantener la trazabilidad entre las decisiones metodológicas adoptadas durante el modelado y los análisis operativos posteriores.

La recuperación de estos productos no implica ningún nuevo proceso de aprendizaje. El modelo y los componentes de preprocesamiento se utilizarán exactamente en el estado en que fueron persistidos, mientras que el umbral definitivo se recuperará de la configuración almacenada en lugar de volver a optimizarse. De esta forma, el período externo de 2026 permanece separado de cualquier nueva decisión de selección o ajuste.

Las comprobaciones exhaustivas de coherencia entre los productos recuperados se reservarán para el subbloque siguiente. En esta etapa se verificará únicamente que la carga pueda realizarse correctamente y se presentará una salida compacta que permita identificar los objetos recuperados sin generar información redundante.

In [2]:
# ---------------------------------------------------------
# 1. Recuperar los componentes definitivos del modelo
# ---------------------------------------------------------

final_model = joblib.load(final_model_path)
final_encoder = joblib.load(final_encoder_path)
final_scaler = joblib.load(final_scaler_path)


# ---------------------------------------------------------
# 2. Recuperar la configuración definitiva
# ---------------------------------------------------------

with open(final_configuration_path, "r", encoding="utf-8") as file:
    final_model_configuration = json.load(file)


# ---------------------------------------------------------
# 3. Recuperar los resultados persistidos necesarios
# ---------------------------------------------------------

threshold_results = pd.read_csv(threshold_results_path)
external_test_results = pd.read_csv(external_test_results_path)
temporal_generalization_results = pd.read_csv(temporal_generalization_path)


# ---------------------------------------------------------
# 4. Recuperar las probabilidades de validación de 2025
# ---------------------------------------------------------

validation_probability = np.load(
    validation_probabilities_path,
    mmap_mode="r"
)


# ---------------------------------------------------------
# 5. Construir un resumen compacto de los productos recuperados
# ---------------------------------------------------------

recovered_products = pd.DataFrame(
    {
        "product": [
            "Modelo definitivo",
            "Encoder categórico",
            "Scaler numérico",
            "Configuración definitiva",
            "Selección del threshold 2025",
            "Resultados externos 2026",
            "Generalización temporal 2025-2026",
            "Scores de validación 2025",
        ],
        "object_type": [
            type(final_model).__name__,
            type(final_encoder).__name__,
            type(final_scaler).__name__,
            type(final_model_configuration).__name__,
            type(threshold_results).__name__,
            type(external_test_results).__name__,
            type(temporal_generalization_results).__name__,
            type(validation_probability).__name__,
        ],
        "size": [
            "—",
            "—",
            "—",
            len(final_model_configuration),
            len(threshold_results),
            len(external_test_results),
            len(temporal_generalization_results),
            validation_probability.shape[0],
        ],
    }
)

display(recovered_products)

,product,object_type,size
0,Modelo definitivo,LogisticRegression,—
1,Encoder categórico,OneHotEncoder,—
2,Scaler numérico,StandardScaler,—
3,Configuración definitiva,dict,15
4,Selección del threshold 2025,DataFrame,2
5,Resultados externos 2026,DataFrame,1
6,Generalización temporal 2025-2026,DataFrame,7
7,Scores de validación 2025,memmap,7597494


#### Interpretación

La recuperación de los productos persistidos del proceso de modelado se ha completado correctamente. El modelo definitivo se ha restaurado como un objeto LogisticRegression, acompañado por el OneHotEncoder y el StandardScaler utilizados durante el preprocesamiento final, lo que permite conservar exactamente la configuración empleada durante la evaluación externa.

La configuración definitiva se encuentra igualmente disponible, junto con los resultados correspondientes a la selección del umbral de decisión, la evaluación externa de 2026 y la comparación de generalización temporal entre 2025 y 2026. De este modo, el análisis operativo podrá construirse a partir de evidencia previamente persistida sin reconstruir manualmente las decisiones adoptadas durante el modelado.

Los 7,597,494 scores recuperados para la validación de 2025 coinciden con el número de observaciones que conformaron dicho período de validación. Su recuperación mediante una representación memmap permite, además, mantener disponible este producto sin requerir su carga completa inmediata en memoria.

La correcta recuperación de los objetos confirma que los principales productos necesarios se encuentran accesibles desde una ejecución independiente del Notebook 04. No obstante, esta comprobación acredita únicamente su disponibilidad y carga técnica; la coherencia entre la configuración, los parámetros del modelo, el umbral definitivo y los resultados persistidos deberá comprobarse conjuntamente antes de iniciar la construcción del dataset analítico de alertas.

### 1.3 Validación consolidada de los productos de entrada

Antes de utilizar los productos recuperados para construir el dataset analítico de alertas, es necesario comprobar de forma conjunta que los principales componentes del proceso de modelado mantienen la configuración y las propiedades esperadas. Esta validación constituye el último control de reproducibilidad previo al procesamiento de la población externa y permite detectar posibles inconsistencias entre los artefactos persistidos y las decisiones metodológicas fijadas en el Notebook 03.

La comprobación se realizará de manera consolidada para evitar verificaciones fragmentadas o redundantes. Se contrastarán la familia y los principales hiperparámetros del modelo definitivo, las características esenciales del preprocesamiento, el umbral de decisión persistido y la disponibilidad de la población externa de 2026. También se verificará la coherencia de los resultados recuperados con las dimensiones conocidas de los períodos de validación y test.

Estas comprobaciones no implican volver a evaluar ni seleccionar el modelo. Los valores esperados proceden de decisiones previamente cerradas y se utilizan exclusivamente como referencias de integridad. En particular, el umbral de decisión debe mantenerse como el seleccionado sobre 2025, sin realizar ninguna optimización adicional a partir de los resultados observados en 2026.

La validación consolidada permitirá cerrar la fase de recuperación únicamente si los componentes necesarios presentan una configuración compatible con el sistema definitivo. Una vez confirmada esta condición, podrá iniciarse la construcción del dataset analítico de alertas manteniendo la trazabilidad entre el modelo evaluado externamente y las observaciones que serán objeto del análisis operativo.

In [7]:
# ---------------------------------------------------------
# 1. Definir los valores de referencia del modelo definitivo
# ---------------------------------------------------------

expected_model_parameters = {
    "C": 0.1,
    "tol": 0.01,
    "solver": "saga",
    "penalty": "l2",
    "class_weight": "balanced",
    "max_iter": 500,
    "random_state": 42,
}

expected_validation_rows = 7_597_494
expected_external_test_rows = 3_102_447
expected_transformed_features = 853
expected_final_threshold = 0.406588

model_parameters = final_model.get_params()


# ---------------------------------------------------------
# 2. Recuperar el threshold definitivo desde la configuración persistida
# ---------------------------------------------------------

final_decision_threshold = float(
    final_model_configuration["decision_threshold"]
)


# ---------------------------------------------------------
# 3. Recuperar las propiedades del preprocesamiento definitivo
# ---------------------------------------------------------

encoder_feature_count = len(
    final_encoder.get_feature_names_out()
)

scaler_feature_count = int(
    final_scaler.n_features_in_
)

total_transformed_features = (
    encoder_feature_count + scaler_feature_count
)


# ---------------------------------------------------------
# 4. Localizar los archivos correspondientes al test externo
# ---------------------------------------------------------

test_parquet_files = sorted(
    external_test_data_path.rglob("*.parquet")
)


# ---------------------------------------------------------
# 5. Recuperar el tamaño del test externo desde la matriz de confusión
# ---------------------------------------------------------

confusion_columns = [
    "true_negative",
    "false_positive",
    "false_negative",
    "true_positive",
]

observed_external_test_rows = int(
    external_test_results.loc[
        external_test_results.index[0],
        confusion_columns
    ].sum()
)


# ---------------------------------------------------------
# 6. Construir las validaciones consolidadas
# ---------------------------------------------------------

validation_checks = [
    {
        "check": "Tipo de modelo definitivo",
        "expected": "LogisticRegression",
        "observed": type(final_model).__name__,
        "valid": (
            type(final_model).__name__
            == "LogisticRegression"
        ),
    },
    {
        "check": "Hiperparámetros definitivos",
        "expected": str(expected_model_parameters),
        "observed": str(
            {
                parameter: model_parameters.get(parameter)
                for parameter in expected_model_parameters
            }
        ),
        "valid": all(
            model_parameters.get(parameter) == expected_value
            for parameter, expected_value
            in expected_model_parameters.items()
        ),
    },
    {
        "check": "Threshold definitivo",
        "expected": expected_final_threshold,
        "observed": final_decision_threshold,
        "valid": np.isclose(
            final_decision_threshold,
            expected_final_threshold,
            atol=1e-6,
        ),
    },
    {
        "check": "Dimensión transformada final",
        "expected": expected_transformed_features,
        "observed": total_transformed_features,
        "valid": (
            total_transformed_features
            == expected_transformed_features
        ),
    },
    {
        "check": "Dimensión del modelo",
        "expected": expected_transformed_features,
        "observed": int(final_model.n_features_in_),
        "valid": (
            int(final_model.n_features_in_)
            == expected_transformed_features
        ),
    },
    {
        "check": "Scores de validación 2025",
        "expected": expected_validation_rows,
        "observed": int(validation_probability.shape[0]),
        "valid": (
            int(validation_probability.shape[0])
            == expected_validation_rows
        ),
    },
    {
        "check": "Archivos Parquet externos 2026",
        "expected": 34,
        "observed": len(test_parquet_files),
        "valid": (
            len(test_parquet_files) == 34
        ),
    },
    {
        "check": "Observaciones externas persistidas",
        "expected": expected_external_test_rows,
        "observed": observed_external_test_rows,
        "valid": (
            observed_external_test_rows
            == expected_external_test_rows
        ),
    },
]


# ---------------------------------------------------------
# 7. Consolidar y mostrar el resultado de las validaciones
# ---------------------------------------------------------

input_validation_results = pd.DataFrame(
    validation_checks
)

display(input_validation_results)

all_input_checks_valid = bool(
    input_validation_results["valid"].all()
)

print(
    "\nValidación global de productos de entrada:",
    all_input_checks_valid
)

,check,expected,observed,valid
0,Tipo de modelo definitivo,LogisticRegression,LogisticRegression,True
1,Hiperparámetros definitivos,"{'C': 0.1, 'tol': 0.01, 'solver': 'saga', 'pen...","{'C': 0.1, 'tol': 0.01, 'solver': 'saga', 'pen...",True
2,Threshold definitivo,0.406588,0.406588,True
3,Dimensión transformada final,853,853,True
4,Dimensión del modelo,853,853,True
5,Scores de validación 2025,7597494,7597494,True
6,Archivos Parquet externos 2026,34,34,True
7,Observaciones externas persistidas,3102447,3102447,True



Validación global de productos de entrada: True


#### Interpretación

La validación consolidada de los productos de entrada se ha completado satisfactoriamente. El modelo recuperado corresponde a la configuración definitiva de regresión logística y conserva los hiperparámetros seleccionados durante el proceso de modelado. Asimismo, el threshold de decisión recuperado coincide con el valor fijado previamente sobre la validación temporal de 2025.

La dimensión del espacio transformado, compuesta por 853 características, coincide tanto con el preprocesamiento persistido como con la dimensión de entrada esperada por el modelo definitivo. Esta correspondencia confirma que el encoder categórico, el escalador numérico y el modelo pueden utilizarse conjuntamente sin incompatibilidades estructurales.

Los 7,597,494 scores recuperados para 2025 coinciden con el tamaño de la población utilizada durante la validación temporal, mientras que los 34 archivos Parquet localizados para el período externo de 2026 representan un total de 3,102,447 observaciones, coherente con la evaluación externa previamente persistida.

En conjunto, estas comprobaciones permiten cerrar la fase de configuración y recuperación con una base reproducible y consistente. Los productos necesarios para el análisis operativo se encuentran disponibles y mantienen la trazabilidad con las decisiones metodológicas adoptadas en el Notebook 03, por lo que puede iniciarse la construcción del dataset analítico de alertas sin introducir nuevas decisiones de modelado.

## 2. Construcción del dataset analítico de alertas

Una vez recuperados y validados los productos definitivos del modelado, el análisis operativo requiere construir una representación de la población externa que permita relacionar cada observación con la decisión emitida por el sistema. Esta etapa constituye el vínculo entre la evaluación predictiva realizada en el Notebook 03 y la caracterización posterior del funcionamiento de las alertas.

La población de referencia estará formada por los vuelos correspondientes a enero–mayo de 2026. Para cada observación se conservarán únicamente las variables necesarias para los análisis posteriores y se aplicará el preprocesamiento definitivo recuperado en el bloque anterior. El modelo se utilizará exclusivamente en modo de inferencia, sin realizar ningún nuevo ajuste sobre el encoder, el escalador o la regresión logística.

La salida continua del modelo se tratará como un score de riesgo predictivo y no como una probabilidad calibrada de retraso. Esta distinción resulta especialmente relevante debido al uso de `class_weight="balanced"` durante el entrenamiento y a la ausencia de un procedimiento específico de calibración probabilística. La alerta binaria se obtendrá aplicando al score el threshold definitivo de 0,406588, fijado previamente mediante la validación temporal de 2025.

La combinación entre la alerta generada y el valor observado de `ARR_DEL15` permitirá clasificar posteriormente cada vuelo en una de las cuatro situaciones fundamentales: verdadero positivo, falso positivo, verdadero negativo o falso negativo. Esta clasificación trasladará la matriz de confusión desde una representación agregada hacia el nivel de observación y proporcionará la base necesaria para estudiar dónde se concentran las detecciones correctas, las falsas alertas y los retrasos no identificados.

Debido al tamaño de la población externa, formada por más de tres millones de registros distribuidos en 34 archivos Parquet, el procedimiento deberá preservar los criterios de eficiencia computacional establecidos en el proyecto. Se priorizará la lectura selectiva de columnas y el procesamiento incremental, evitando mantener simultáneamente en memoria estructuras masivas que puedan generarse y persistirse de forma progresiva.

El producto resultante tendrá carácter analítico y post-test. Su finalidad será describir el comportamiento del sistema definitivo sobre 2026 y preparar las agregaciones posteriores, sin utilizar la información obtenida para modificar retrospectivamente el modelo, el preprocesamiento o la regla de decisión.

En este bloque se abordarán progresivamente:

1. la recuperación de la población externa de análisis correspondiente a enero–mayo de 2026;
2. la generación del score de riesgo, la alerta binaria y la incorporación del resultado realmente observado;
3. la clasificación de las observaciones en verdaderos positivos, falsos positivos, verdaderos negativos y falsos negativos;
4. la persistencia y validación consolidada del dataset analítico de alertas.

### 2.1 Recuperación de la población externa de análisis

La construcción del dataset analítico requiere recuperar la población correspondiente al período externo enero–mayo de 2026 conservando únicamente la información necesaria para las etapas posteriores. Aunque esta población ya fue utilizada para evaluar el modelo definitivo, el Notebook 03 persistió los resultados agregados de dicha evaluación y no un dataset a nivel de vuelo que contenga conjuntamente las dimensiones de análisis y las decisiones generadas por el sistema.

En este subbloque se recuperarán, por tanto, las variables predictoras utilizadas por el modelo, la variable objetivo `ARR_DEL15` y la fecha del vuelo `FL_DATE`. La incorporación de esta última permitirá conservar la dimensión temporal necesaria para los análisis mensuales posteriores sin modificar el conjunto de predictores empleado por el modelo.

La lectura se realizará exclusivamente sobre los 34 archivos Parquet correspondientes al test externo y seleccionando únicamente las columnas necesarias. En esta etapa todavía no se aplicará el modelo ni se generarán scores o alertas. El objetivo consiste en establecer una población analítica de entrada cuya cobertura, período temporal y distribución de la variable objetivo puedan comprobarse antes de iniciar la inferencia.

Dado que el conjunto externo contiene 3.102.447 observaciones, la recuperación se realizará archivo por archivo con seguimiento del progreso. Esta estrategia mantiene la lectura restringida a la población relevante y evita acceder al conjunto histórico completo del proyecto. La materialización conjunta de 2026 se justifica en este caso porque constituye la población específica sobre la que se desarrollarán los análisis operativos posteriores y contiene un número de columnas deliberadamente limitado.

In [8]:
# ---------------------------------------------------------
# 1. Definir las columnas necesarias para el análisis
# ---------------------------------------------------------

categorical_features = [
    "MONTH",
    "DAY_OF_WEEK",
    "MKT_UNIQUE_CARRIER",
    "OP_UNIQUE_CARRIER",
    "ORIGIN",
    "DEST",
    "DEP_TIME_BLK",
    "ARR_TIME_BLK",
]

numerical_features = [
    "CRS_ELAPSED_TIME",
    "DISTANCE",
]

target_column = "ARR_DEL15"
date_column = "FL_DATE"

external_analysis_columns = (
    [date_column]
    + categorical_features
    + numerical_features
    + [target_column]
)


# ---------------------------------------------------------
# 2. Importar la herramienta de seguimiento del progreso
# ---------------------------------------------------------

from tqdm.auto import tqdm


# ---------------------------------------------------------
# 3. Recuperar incrementalmente los archivos del período externo
# ---------------------------------------------------------

external_analysis_parts = []

for parquet_file in tqdm(
    test_parquet_files,
    desc="Recuperando población externa 2026",
):
    file_data = pd.read_parquet(
        parquet_file,
        columns=external_analysis_columns,
    )

    external_analysis_parts.append(file_data)


# ---------------------------------------------------------
# 4. Consolidar la población externa de análisis
# ---------------------------------------------------------

external_analysis_data = pd.concat(
    external_analysis_parts,
    ignore_index=True,
)

del external_analysis_parts


# ---------------------------------------------------------
# 5. Normalizar la variable temporal
# ---------------------------------------------------------

external_analysis_data[date_column] = pd.to_datetime(
    external_analysis_data[date_column],
    errors="coerce",
)


# ---------------------------------------------------------
# 6. Construir un resumen consolidado de la población recuperada
# ---------------------------------------------------------

external_population_summary = pd.DataFrame(
    {
        "indicator": [
            "Archivos procesados",
            "Observaciones",
            "Columnas recuperadas",
            "Fecha mínima",
            "Fecha máxima",
            "Clase 0",
            "Clase 1",
            "Prevalencia positiva",
            "Fechas ausentes",
            "Target ausente",
        ],
        "value": [
            len(test_parquet_files),
            len(external_analysis_data),
            external_analysis_data.shape[1],
            external_analysis_data[date_column].min(),
            external_analysis_data[date_column].max(),
            int(
                (external_analysis_data[target_column] == 0).sum()
            ),
            int(
                (external_analysis_data[target_column] == 1).sum()
            ),
            float(
                external_analysis_data[target_column].mean()
            ),
            int(
                external_analysis_data[date_column].isna().sum()
            ),
            int(
                external_analysis_data[target_column].isna().sum()
            ),
        ],
    }
)

display(external_population_summary)

Recuperando población externa 2026:   0%|          | 0/34 [00:00<?, ?it/s]

,indicator,value
0,Archivos procesados,34
1,Observaciones,3102447
2,Columnas recuperadas,12
3,Fecha mínima,2026-01-01 00:00:00
4,Fecha máxima,2026-05-31 00:00:00
5,Clase 0,2445603
6,Clase 1,656844
7,Prevalencia positiva,0.211718
8,Fechas ausentes,0
9,Target ausente,0


#### Interpretación

La recuperación de la población externa de análisis se ha completado correctamente sobre los 34 archivos Parquet correspondientes al período enero–mayo de 2026. El conjunto consolidado contiene 3,102,447 observaciones y 12 variables, formadas por la fecha del vuelo, los diez predictores utilizados por el modelo definitivo y la variable objetivo `ARR_DEL15`.

La cobertura temporal se extiende desde el 1 de enero hasta el 31 de mayo de 2026, coincidiendo con el período definido previamente como test externo. La distribución de la variable objetivo reproduce igualmente la composición conocida de esta población: 2,445,603 observaciones pertenecen a la clase 0 y 656,844 a la clase 1, con una prevalencia positiva aproximada del 21.17 %.

No se identifican valores ausentes en la fecha del vuelo ni en la variable objetivo. Esta condición resulta especialmente relevante para las etapas posteriores, ya que permitirá construir las dimensiones temporales y clasificar cada observación según el resultado real sin necesidad de aplicar tratamientos adicionales sobre estas dos variables.

En conjunto, la población recuperada mantiene coherencia con la utilizada durante la evaluación externa del Notebook 03 y proporciona una base adecuada para aplicar el preprocesamiento definitivo y generar el score de riesgo y la alerta binaria en el siguiente subbloque.

### 2.2 Generación del score de riesgo, alerta y resultado observado

Una vez validada la población externa de análisis, el siguiente paso consiste en aplicar sobre cada observación el mismo procedimiento de preprocesamiento y el mismo modelo definitivo utilizados durante la evaluación externa del Notebook 03. El objetivo de este subbloque es construir, a nivel de vuelo, las dos salidas fundamentales del sistema: el score de riesgo continuo y la alerta binaria derivada del threshold previamente fijado.

Las variables categóricas se transformarán mediante el `OneHotEncoder` recuperado, mientras que las variables numéricas se estandarizarán con el `StandardScaler` definitivo. Ambos objetos fueron ajustados exclusivamente sobre el período de entrenamiento final 2022–2025, por lo que en esta etapa se utilizarán únicamente mediante `transform`, evitando cualquier reajuste sobre la población externa de 2026.

La salida asociada a la clase positiva se conservará como `risk_score`. Esta denominación evita interpretar el valor como una probabilidad necesariamente calibrada y lo presenta como una puntuación continua de riesgo producida por el modelo. A continuación, la regla de decisión definitiva se aplicará mediante el threshold 0,406588: los vuelos cuyo score sea igual o superior a dicho valor recibirán una alerta, mientras que el resto permanecerá sin alerta.

El procesamiento se realizará por bloques para limitar el tamaño de las matrices transformadas en memoria. Esta estrategia es especialmente importante porque la codificación categórica genera un espacio de 853 características y la población externa supera los tres millones de observaciones. El resultado final conservará el mismo número de filas que la población original e incorporará únicamente las nuevas columnas necesarias para el análisis operativo.

In [9]:
# ---------------------------------------------------------
# 1. Definir el tamaño de los bloques de inferencia
# ---------------------------------------------------------

inference_batch_size = 250_000

risk_scores = np.empty(
    len(external_analysis_data),
    dtype=np.float32,
)


# ---------------------------------------------------------
# 2. Aplicar el preprocesamiento y el modelo por bloques
# ---------------------------------------------------------

for start_idx in tqdm(
    range(0, len(external_analysis_data), inference_batch_size),
    desc="Generando scores de riesgo 2026",
):
    end_idx = min(
        start_idx + inference_batch_size,
        len(external_analysis_data),
    )

    batch_data = external_analysis_data.iloc[
        start_idx:end_idx
    ]

    categorical_batch = final_encoder.transform(
        batch_data[categorical_features]
    )

    numerical_batch = final_scaler.transform(
        batch_data[numerical_features]
    ).astype(np.float32)

    from scipy.sparse import csr_matrix, hstack

    transformed_batch = hstack(
        [
            categorical_batch,
            csr_matrix(numerical_batch),
        ],
        format="csr",
        dtype=np.float32,
    )

    risk_scores[start_idx:end_idx] = (
        final_model.predict_proba(
            transformed_batch
        )[:, 1].astype(np.float32)
    )


# ---------------------------------------------------------
# 3. Incorporar el score y generar la alerta binaria
# ---------------------------------------------------------

external_analysis_data["risk_score"] = risk_scores

external_analysis_data["alert"] = (
    external_analysis_data["risk_score"]
    >= final_decision_threshold
).astype(np.int8)


# ---------------------------------------------------------
# 4. Construir un resumen consolidado de la inferencia
# ---------------------------------------------------------

alert_generation_summary = pd.DataFrame(
    {
        "indicator": [
            "Observaciones procesadas",
            "Score mínimo",
            "Score máximo",
            "Score medio",
            "Threshold aplicado",
            "Alertas generadas",
            "Vuelos sin alerta",
            "Tasa de alerta",
            "Retrasos observados",
            "Tasa observada de retraso",
        ],
        "value": [
            len(external_analysis_data),
            float(
                external_analysis_data["risk_score"].min()
            ),
            float(
                external_analysis_data["risk_score"].max()
            ),
            float(
                external_analysis_data["risk_score"].mean()
            ),
            final_decision_threshold,
            int(
                external_analysis_data["alert"].sum()
            ),
            int(
                (
                    external_analysis_data["alert"] == 0
                ).sum()
            ),
            float(
                external_analysis_data["alert"].mean()
            ),
            int(
                external_analysis_data[target_column].sum()
            ),
            float(
                external_analysis_data[target_column].mean()
            ),
        ],
    }
)

display(alert_generation_summary)

Generando scores de riesgo 2026:   0%|          | 0/13 [00:00<?, ?it/s]

,indicator,value
0,Observaciones procesadas,3.102447e+06
1,Score mínimo,1.250792e-05
2,Score máximo,8.854018e-01
3,Score medio,4.708516e-01
4,Threshold aplicado,4.065878e-01
5,Alertas generadas,2.078168e+06
6,Vuelos sin alerta,1.024279e+06
7,Tasa de alerta,6.698480e-01
8,Retrasos observados,6.568440e+05
9,Tasa observada de retraso,2.117180e-01


#### Interpretación

La generación del score de riesgo y de la alerta binaria se ha completado correctamente para las 3,102,447 observaciones del período externo enero–mayo de 2026. El modelo produce scores comprendidos aproximadamente entre 0.00001 y 0.8854, con un valor medio cercano a 0.471.

Aplicando el threshold definitivo de 0.406588, el sistema genera 2,078,168 alertas y deja 1,024,279 observaciones sin alerta. Esto representa una tasa de alerta aproximada del 66.98 %, valor coherente con el resultado obtenido previamente durante la evaluación externa del modelo.

La proporción de alertas es considerablemente superior a la prevalencia observada de retrasos, situada en torno al 21.17 %. Esta diferencia es consistente con el criterio operativo adoptado para el sistema, orientado a mantener una cobertura elevada de los retrasos reales mediante un nivel de recall próximo al 80 %. Como consecuencia, la regla de decisión acepta una mayor cantidad de alertas positivas, incluyendo necesariamente una fracción relevante de falsas alertas.

La reproducción de la tasa de alerta previamente registrada confirma que el preprocesamiento, el modelo y el threshold se han aplicado de forma consistente sobre la población externa. El dataset dispone ya, a nivel individual, del resultado observado, del score de riesgo y de la decisión binaria necesaria para identificar en el siguiente subbloque los verdaderos positivos, falsos positivos, verdaderos negativos y falsos negativos.

### 2.3 Clasificación de las observaciones en TP/FP/TN/FN

Una vez disponibles el resultado observado y la alerta generada por el sistema, cada vuelo puede clasificarse según la correspondencia entre ambas variables. Esta clasificación traslada la matriz de confusión desde su representación agregada hacia el nivel individual y constituye la base de los análisis operativos posteriores.

Se considerará verdadero positivo (`TP`) un vuelo que presentó un retraso igual o superior a 15 minutos y para el que el sistema generó una alerta. Un falso positivo (`FP`) corresponderá a una alerta emitida sobre un vuelo que finalmente no presentó dicho retraso. Los verdaderos negativos (`TN`) representarán vuelos sin retraso correctamente mantenidos sin alerta, mientras que los falsos negativos (`FN`) identificarán retrasos reales que el sistema no anticipó mediante una alerta.

Desde la perspectiva del sistema de alerta temprana, esta distinción resulta especialmente relevante. Los falsos negativos representan retrasos reales que permanecen fuera de la cobertura del sistema, mientras que los falsos positivos representan alertas que no se materializan posteriormente en un retraso observado. El diseño adoptado prioriza la reducción relativa de los primeros mediante un nivel elevado de recall, aceptando como contrapartida una mayor presencia de falsas alertas.

La clasificación obtenida en este subbloque tendrá exclusivamente una finalidad descriptiva y post-test. No se utilizará para modificar el threshold ni ningún componente del modelo definitivo. Además, se comprobará que las frecuencias agregadas de las cuatro categorías reproduczcan la matriz de confusión persistida durante la evaluación externa de 2026, proporcionando una validación adicional de la construcción del dataset analítico.

In [10]:
# ---------------------------------------------------------
# 1. Recuperar las variables necesarias para la clasificación
# ---------------------------------------------------------

observed_target = external_analysis_data[
    target_column
].to_numpy()

generated_alert = external_analysis_data[
    "alert"
].to_numpy()


# ---------------------------------------------------------
# 2. Construir las condiciones de la matriz de confusión
# ---------------------------------------------------------

true_positive_mask = (
    (observed_target == 1)
    & (generated_alert == 1)
)

false_positive_mask = (
    (observed_target == 0)
    & (generated_alert == 1)
)

true_negative_mask = (
    (observed_target == 0)
    & (generated_alert == 0)
)

false_negative_mask = (
    (observed_target == 1)
    & (generated_alert == 0)
)


# ---------------------------------------------------------
# 3. Asignar la categoría correspondiente a cada observación
# ---------------------------------------------------------

confusion_category = np.empty(
    len(external_analysis_data),
    dtype="<U2",
)

confusion_category[true_positive_mask] = "TP"
confusion_category[false_positive_mask] = "FP"
confusion_category[true_negative_mask] = "TN"
confusion_category[false_negative_mask] = "FN"

external_analysis_data[
    "confusion_category"
] = pd.Categorical(
    confusion_category,
    categories=["TP", "FP", "TN", "FN"],
)


# ---------------------------------------------------------
# 4. Calcular las frecuencias obtenidas
# ---------------------------------------------------------

confusion_counts = (
    external_analysis_data["confusion_category"]
    .value_counts(sort=False)
)

generated_confusion_values = {
    "TP": int(confusion_counts["TP"]),
    "FP": int(confusion_counts["FP"]),
    "TN": int(confusion_counts["TN"]),
    "FN": int(confusion_counts["FN"]),
}


# ---------------------------------------------------------
# 5. Recuperar la matriz de confusión persistida
# ---------------------------------------------------------

expected_confusion_values = {
    "TP": int(
        external_test_results.iloc[0]["true_positive"]
    ),
    "FP": int(
        external_test_results.iloc[0]["false_positive"]
    ),
    "TN": int(
        external_test_results.iloc[0]["true_negative"]
    ),
    "FN": int(
        external_test_results.iloc[0]["false_negative"]
    ),
}


# ---------------------------------------------------------
# 6. Construir la validación consolidada de la clasificación
# ---------------------------------------------------------

confusion_validation = pd.DataFrame(
    {
        "category": ["TP", "FP", "TN", "FN"],
        "observed": [
            generated_confusion_values["TP"],
            generated_confusion_values["FP"],
            generated_confusion_values["TN"],
            generated_confusion_values["FN"],
        ],
        "expected": [
            expected_confusion_values["TP"],
            expected_confusion_values["FP"],
            expected_confusion_values["TN"],
            expected_confusion_values["FN"],
        ],
    }
)

confusion_validation["difference"] = (
    confusion_validation["observed"]
    - confusion_validation["expected"]
)

confusion_validation["valid"] = (
    confusion_validation["difference"] == 0
)

confusion_validation["share"] = (
    confusion_validation["observed"]
    / len(external_analysis_data)
)

display(confusion_validation)

print(
    "\nObservaciones clasificadas:",
    int(confusion_validation["observed"].sum()),
)

print(
    "Clasificación completamente válida:",
    bool(confusion_validation["valid"].all()),
)

,category,observed,expected,difference,valid,share
0,TP,525105,525105,0,True,0.169255
1,FP,1553063,1553063,0,True,0.500593
2,TN,892540,892540,0,True,0.287689
3,FN,131739,131739,0,True,0.042463



Observaciones clasificadas: 3102447
Clasificación completamente válida: True


#### Interpretación

La clasificación individual de las observaciones en verdaderos positivos, falsos positivos, verdaderos negativos y falsos negativos se ha completado correctamente para toda la población externa de 2026.

Los resultados obtenidos reproducen exactamente la matriz de confusión persistida durante la evaluación externa: 525,105 verdaderos positivos, 1,553,063 falsos positivos, 892,540 verdaderos negativos y 131,739 falsos negativos. La diferencia entre los valores reconstruidos a nivel de vuelo y los valores esperados es igual a cero en las cuatro categorías, lo que confirma la coherencia del dataset analítico generado.

En términos relativos, aproximadamente el 16.93 % de todas las observaciones corresponde a verdaderos positivos, el 50.06 % a falsos positivos, el 28.77 % a verdaderos negativos y el 4.25 % a falsos negativos. Esta distribución refleja el comportamiento operativo derivado del criterio de decisión adoptado: el sistema mantiene una cobertura elevada de los retrasos reales, pero a costa de generar una proporción considerable de alertas que no se materializan posteriormente en un retraso igual o superior a 15 minutos.

Desde la perspectiva del análisis operativo, estas cuatro categorías proporcionan ahora la variable fundamental para estudiar dónde se concentran tanto las detecciones correctas como los errores del sistema. Los análisis posteriores podrán comparar su distribución temporal, geográfica y operacional sin modificar el modelo ni la regla de decisión previamente fijada.

### 2.4 Persistencia y validación consolidada del dataset analítico de alertas

Una vez incorporados el score de riesgo, la alerta binaria y la categoría derivada de la matriz de confusión, el dataset analítico contiene la información necesaria para desarrollar los análisis operativos posteriores. Antes de avanzar, resulta conveniente persistir este producto para evitar repetir la transformación de las 3,102,447 observaciones y la inferencia del modelo cada vez que sea necesario recuperar los resultados.

La persistencia se realizará en formato Parquet, manteniendo las 15 variables actualmente disponibles: la fecha del vuelo, los diez predictores definitivos, la variable objetivo, el score de riesgo, la alerta generada y la categoría de clasificación. Este formato permite conservar los tipos de datos de forma eficiente y facilita posteriormente la lectura selectiva de columnas para los diferentes análisis temporales, operacionales y geográficos.

El producto persistido tendrá carácter exclusivamente analítico y post-test. Su contenido reproduce las decisiones del modelo definitivo sobre enero–mayo de 2026 y no constituye una nueva fuente de entrenamiento o selección. En particular, ni el score de riesgo, ni la clasificación de las alertas, ni los análisis derivados de este dataset podrán utilizarse para reajustar retrospectivamente el modelo o el threshold.

La validación final comprobará de forma consolidada la existencia del archivo, el número de observaciones, el número de variables, la cobertura temporal, la ausencia de valores críticos y la correspondencia de las cuatro categorías de la matriz de confusión. Con ello se cerrará la construcción del dataset analítico y se establecerá un producto persistido y reproducible para los siguientes bloques del notebook.

In [11]:
# ---------------------------------------------------------
# 1. Configurar la ruta de persistencia del análisis operativo
# ---------------------------------------------------------

operational_results_path = (
    project_root
    / "results"
    / "operational_analysis"
)

operational_results_path.mkdir(
    parents=True,
    exist_ok=True,
)

alert_dataset_path = (
    operational_results_path
    / "external_alert_analysis_2026.parquet"
)


# ---------------------------------------------------------
# 2. Definir el esquema definitivo del dataset analítico
# ---------------------------------------------------------

alert_analysis_columns = (
    [date_column]
    + categorical_features
    + numerical_features
    + [
        target_column,
        "risk_score",
        "alert",
        "confusion_category",
    ]
)

external_alert_analysis = external_analysis_data[
    alert_analysis_columns
].copy()


# ---------------------------------------------------------
# 3. Persistir el dataset analítico en formato Parquet
# ---------------------------------------------------------

external_alert_analysis.to_parquet(
    alert_dataset_path,
    index=False,
)


# ---------------------------------------------------------
# 4. Recuperar el producto persistido para su validación
# ---------------------------------------------------------

persisted_alert_analysis = pd.read_parquet(
    alert_dataset_path
)


# ---------------------------------------------------------
# 5. Recuperar las frecuencias de la matriz de confusión
# ---------------------------------------------------------

persisted_confusion_counts = (
    persisted_alert_analysis["confusion_category"]
    .value_counts()
)

expected_confusion_counts = {
    "TP": 525_105,
    "FP": 1_553_063,
    "TN": 892_540,
    "FN": 131_739,
}

confusion_counts_valid = all(
    int(persisted_confusion_counts.get(category, 0))
    == expected_count
    for category, expected_count
    in expected_confusion_counts.items()
)


# ---------------------------------------------------------
# 6. Construir las validaciones consolidadas del producto
# ---------------------------------------------------------

alert_dataset_validation = pd.DataFrame(
    [
        {
            "check": "Archivo persistido",
            "expected": True,
            "observed": alert_dataset_path.exists(),
            "valid": alert_dataset_path.exists(),
        },
        {
            "check": "Observaciones",
            "expected": 3_102_447,
            "observed": len(persisted_alert_analysis),
            "valid": (
                len(persisted_alert_analysis)
                == 3_102_447
            ),
        },
        {
            "check": "Variables",
            "expected": 15,
            "observed": persisted_alert_analysis.shape[1],
            "valid": (
                persisted_alert_analysis.shape[1]
                == 15
            ),
        },
        {
            "check": "Fecha mínima",
            "expected": pd.Timestamp("2026-01-01"),
            "observed": persisted_alert_analysis[
                date_column
            ].min(),
            "valid": (
                persisted_alert_analysis[
                    date_column
                ].min()
                == pd.Timestamp("2026-01-01")
            ),
        },
        {
            "check": "Fecha máxima",
            "expected": pd.Timestamp("2026-05-31"),
            "observed": persisted_alert_analysis[
                date_column
            ].max(),
            "valid": (
                persisted_alert_analysis[
                    date_column
                ].max()
                == pd.Timestamp("2026-05-31")
            ),
        },
        {
            "check": "Target ausente",
            "expected": 0,
            "observed": int(
                persisted_alert_analysis[
                    target_column
                ].isna().sum()
            ),
            "valid": (
                persisted_alert_analysis[
                    target_column
                ].isna().sum()
                == 0
            ),
        },
        {
            "check": "Score ausente",
            "expected": 0,
            "observed": int(
                persisted_alert_analysis[
                    "risk_score"
                ].isna().sum()
            ),
            "valid": (
                persisted_alert_analysis[
                    "risk_score"
                ].isna().sum()
                == 0
            ),
        },
        {
            "check": "Alerta ausente",
            "expected": 0,
            "observed": int(
                persisted_alert_analysis[
                    "alert"
                ].isna().sum()
            ),
            "valid": (
                persisted_alert_analysis[
                    "alert"
                ].isna().sum()
                == 0
            ),
        },
        {
            "check": "Categoría de confusión ausente",
            "expected": 0,
            "observed": int(
                persisted_alert_analysis[
                    "confusion_category"
                ].isna().sum()
            ),
            "valid": (
                persisted_alert_analysis[
                    "confusion_category"
                ].isna().sum()
                == 0
            ),
        },
        {
            "check": "Matriz de confusión",
            "expected": str(expected_confusion_counts),
            "observed": str(
                {
                    category: int(
                        persisted_confusion_counts.get(
                            category,
                            0,
                        )
                    )
                    for category
                    in ["TP", "FP", "TN", "FN"]
                }
            ),
            "valid": confusion_counts_valid,
        },
    ]
)


# ---------------------------------------------------------
# 7. Mostrar el resultado consolidado de la validación
# ---------------------------------------------------------

display(alert_dataset_validation)

all_alert_dataset_checks_valid = bool(
    alert_dataset_validation["valid"].all()
)

print(
    "\nValidación global del dataset analítico:",
    all_alert_dataset_checks_valid,
)

print(
    "Ruta del producto persistido:",
    alert_dataset_path,
)

print(
    "Tamaño del archivo:",
    f"{alert_dataset_path.stat().st_size / (1024 ** 2):,.2f} MB",
)

,check,expected,observed,valid
0,Archivo persistido,True,True,True
1,Observaciones,3102447,3102447,True
2,Variables,15,15,True
3,Fecha mínima,2026-01-01 00:00:00,2026-01-01 00:00:00,True
4,Fecha máxima,2026-05-31 00:00:00,2026-05-31 00:00:00,True
5,Target ausente,0,0,True
6,Score ausente,0,0,True
7,Alerta ausente,0,0,True
8,Categoría de confusión ausente,0,0,True
9,Matriz de confusión,"{'TP': 525105, 'FP': 1553063, 'TN': 892540, 'F...","{'TP': 525105, 'FP': 1553063, 'TN': 892540, 'F...",True



Validación global del dataset analítico: True
Ruta del producto persistido: G:\My Drive\MASTER Big Data\TFM\results\operational_analysis\external_alert_analysis_2026.parquet
Tamaño del archivo: 32.16 MB


#### Interpretación

La persistencia y validación del dataset analítico de alertas se ha completado satisfactoriamente. El producto almacenado contiene 3,102,447 observaciones y 15 variables, manteniendo íntegramente la población externa correspondiente al período comprendido entre el 1 de enero y el 31 de mayo de 2026.

La recuperación posterior del archivo confirma que no existen valores ausentes en la variable objetivo, el score de riesgo, la alerta binaria ni la categoría derivada de la matriz de confusión. Asimismo, las frecuencias reconstruidas mantienen exactamente los 525,105 verdaderos positivos, 1,553,063 falsos positivos, 892,540 verdaderos negativos y 131,739 falsos negativos obtenidos durante la evaluación externa.

El archivo persistido ocupa 32.16 MB, un tamaño reducido en relación con las más de 3.1 millones de observaciones que contiene. El uso de formato Parquet permite, por tanto, conservar el producto analítico a nivel de vuelo con un coste de almacenamiento limitado y facilita la lectura selectiva de las variables necesarias en los análisis posteriores.

Con esta persistencia queda consolidada una representación reproducible del comportamiento del sistema de alerta temprana sobre el período externo de 2026. A partir de este punto no será necesario repetir el preprocesamiento ni la inferencia del modelo para desarrollar los análisis descriptivos posteriores, reduciendo el coste computacional y manteniendo la trazabilidad respecto a los resultados definitivos obtenidos en el Notebook 03.

## 3. Caracterización global del sistema de alerta temprana

Una vez construido y validado el dataset analítico a nivel de vuelo, el análisis puede centrarse en caracterizar el comportamiento global del sistema de alerta temprana sobre el período externo enero–mayo de 2026. El objetivo de este bloque es transformar las métricas predictivas agregadas en indicadores directamente relacionados con el funcionamiento operativo de las alertas.

El análisis parte de una distinción fundamental entre la frecuencia con la que el sistema genera alertas y la frecuencia con la que realmente se producen retrasos. La comparación entre ambas magnitudes permite dimensionar el nivel de intervención asociado al threshold definitivo y contextualizar el compromiso establecido entre cobertura de retrasos y generación de falsas alertas.

La matriz de confusión se utilizará posteriormente para descomponer este comportamiento en detecciones correctas, falsas alertas, vuelos correctamente descartados y retrasos no detectados. Sin embargo, además de expresar estas categorías respecto al conjunto completo de vuelos, resulta necesario analizarlas desde la perspectiva específica de las alertas emitidas y de los retrasos realmente observados. De esta forma pueden diferenciarse conceptos operativamente distintos como la cobertura de los retrasos y la proporción de alertas que finalmente se corresponden con un retraso real.

El score de riesgo también será caracterizado de forma descriptiva. Su distribución permitirá estudiar cómo se posicionan las observaciones respecto al threshold de decisión y cómo difieren los scores entre las cuatro categorías de la matriz de confusión. Este análisis se interpretará como una caracterización de la puntuación producida por el modelo y no como un estudio de calibración probabilística.

Todos los resultados obtenidos en este bloque mantienen carácter descriptivo y post-test. Su finalidad es comprender el comportamiento del sistema definitivo sobre 2026, sin utilizar las diferencias observadas para reajustar el threshold, modificar el modelo o introducir nuevas decisiones de selección.

En este bloque se abordarán progresivamente:

1. la caracterización del volumen global de alertas y de retrasos observados;
2. la descomposición global de las alertas mediante la matriz de confusión y sus principales indicadores operativos;
3. la caracterización de la distribución del score de riesgo y su relación con el threshold de decisión;
4. la síntesis consolidada del comportamiento global del sistema de alerta temprana.

### 3.1 Volumen global de alertas y retrasos observados

La primera aproximación al comportamiento global del sistema consiste en comparar el volumen de alertas generadas con la frecuencia real de retrasos observados durante el período externo de 2026. Esta comparación permite dimensionar el alcance operativo de la regla de decisión antes de analizar la composición interna de las alertas.

El número de alertas representa la cantidad de vuelos clasificados por el sistema como situaciones de riesgo elevado, mientras que los retrasos observados corresponden a vuelos cuyo valor real de `ARR_DEL15` es igual a 1. Ambas magnitudes responden a conceptos diferentes y, por tanto, no se espera que sus frecuencias coincidan.

Dado que el threshold definitivo fue seleccionado para mantener una cobertura mínima aproximada del 80 % de los retrasos reales, resulta especialmente relevante cuantificar qué proporción de la población recibe una alerta y compararla con la prevalencia efectiva del evento. Esta relación permite contextualizar el coste operativo asociado a una estrategia orientada prioritariamente a reducir los retrasos no detectados.

Para evitar interpretaciones basadas exclusivamente en porcentajes, los resultados se expresarán simultáneamente mediante frecuencias absolutas y tasas sobre la población completa. Además, se calculará la relación entre alertas generadas y retrasos observados como indicador descriptivo de la intensidad con la que opera el sistema. Esta relación no representa por sí misma una medida de calidad predictiva, sino una caracterización del volumen de alertas necesario bajo la regla de decisión establecida.

In [12]:
# ---------------------------------------------------------
# 1. Recuperar los volúmenes globales de la población
# ---------------------------------------------------------

total_flights = len(external_alert_analysis)

total_alerts = int(
    external_alert_analysis["alert"].sum()
)

total_observed_delays = int(
    external_alert_analysis[target_column].sum()
)

total_no_alerts = (
    total_flights - total_alerts
)

total_no_delays = (
    total_flights - total_observed_delays
)


# ---------------------------------------------------------
# 2. Calcular las tasas globales
# ---------------------------------------------------------

alert_rate = (
    total_alerts / total_flights
)

observed_delay_rate = (
    total_observed_delays / total_flights
)

no_alert_rate = (
    total_no_alerts / total_flights
)

no_delay_rate = (
    total_no_delays / total_flights
)

alerts_per_observed_delay = (
    total_alerts / total_observed_delays
)

alert_rate_gap = (
    alert_rate - observed_delay_rate
)


# ---------------------------------------------------------
# 3. Construir la tabla consolidada de indicadores
# ---------------------------------------------------------

global_volume_summary = pd.DataFrame(
    {
        "indicator": [
            "Vuelos analizados",
            "Alertas generadas",
            "Vuelos sin alerta",
            "Retrasos observados",
            "Vuelos sin retraso observado",
            "Tasa de alerta",
            "Tasa sin alerta",
            "Tasa observada de retraso",
            "Tasa sin retraso observado",
            "Diferencia alerta-retraso",
            "Alertas por retraso observado",
        ],
        "value": [
            total_flights,
            total_alerts,
            total_no_alerts,
            total_observed_delays,
            total_no_delays,
            alert_rate,
            no_alert_rate,
            observed_delay_rate,
            no_delay_rate,
            alert_rate_gap,
            alerts_per_observed_delay,
        ],
    }
)

display(global_volume_summary)


# ---------------------------------------------------------
# 4. Mostrar una comparación directa de los eventos principales
# ---------------------------------------------------------

global_event_comparison = pd.DataFrame(
    {
        "event": [
            "Alerta generada",
            "Retraso observado",
        ],
        "count": [
            total_alerts,
            total_observed_delays,
        ],
        "rate": [
            alert_rate,
            observed_delay_rate,
        ],
    }
)

display(global_event_comparison)

,indicator,value
0,Vuelos analizados,3.102447e+06
1,Alertas generadas,2.078168e+06
2,Vuelos sin alerta,1.024279e+06
3,Retrasos observados,6.568440e+05
4,Vuelos sin retraso observado,2.445603e+06
5,Tasa de alerta,6.698480e-01
6,Tasa sin alerta,3.301520e-01
7,Tasa observada de retraso,2.117180e-01
8,Tasa sin retraso observado,7.882820e-01
9,Diferencia alerta-retraso,4.581300e-01


,event,count,rate
0,Alerta generada,2078168,0.669848
1,Retraso observado,656844,0.211718


#### Interpretación

Durante el período externo enero–mayo de 2026 se analizaron 3,102,447 vuelos, de los cuales 2,078,168 recibieron una alerta del sistema y 1,024,279 permanecieron sin alerta. Esto supone una tasa global de alerta del 66.98 % de la población analizada.

En contraste, se observaron 656,844 vuelos con un retraso igual o superior a 15 minutos, equivalentes al 21.17 % de la población. La tasa de alerta supera, por tanto, a la prevalencia observada de retrasos en 45.81 puntos porcentuales.

La relación entre ambas magnitudes indica que se generan aproximadamente 3.16 alertas por cada retraso observado en la población. Este indicador no debe interpretarse como el número de alertas necesario para detectar individualmente un retraso, sino como una medida descriptiva de la intensidad con la que opera el sistema bajo el threshold establecido.

La diferencia entre la frecuencia de alertas y la frecuencia real del evento es coherente con una estrategia de decisión orientada prioritariamente a mantener una cobertura elevada de los retrasos. No obstante, también anticipa que una parte considerable de las alertas no estará asociada posteriormente a un retraso real. La descomposición entre verdaderos y falsos positivos permitirá cuantificar esta contrapartida de forma específica en el siguiente análisis.

### 3.2 Descomposición global de las alertas e indicadores operativos

La elevada tasa global de alerta observada en el apartado anterior debe analizarse conjuntamente con el resultado real de los vuelos para determinar qué proporción de las alertas emitidas se corresponde efectivamente con retrasos y qué proporción constituye una falsa alerta. Del mismo modo, resulta necesario estudiar qué ocurre entre los vuelos que permanecen sin alerta, diferenciando aquellos correctamente identificados de los retrasos que el sistema no consiguió anticipar.

La matriz de confusión permite realizar esta descomposición desde dos perspectivas complementarias. Desde la perspectiva de las alertas emitidas, los verdaderos positivos representan alertas asociadas posteriormente a un retraso real, mientras que los falsos positivos representan alertas que no se materializaron en un retraso igual o superior a 15 minutos. Esta relación permite cuantificar tanto la precisión de las alertas como la proporción de falsas alertas.

Desde la perspectiva de los retrasos realmente observados, los verdaderos positivos representan los eventos cubiertos por el sistema y los falsos negativos aquellos que permanecieron sin alerta. Esta segunda relación permite expresar la cobertura operativa del sistema mediante el recall y, complementariamente, la proporción de retrasos no detectados.

El análisis incorporará tanto frecuencias absolutas como indicadores relativos. Esta combinación es necesaria porque el sistema se aplica sobre una población de más de 3.1 millones de vuelos y una misma tasa puede tener implicaciones operativas diferentes dependiendo del volumen absoluto de observaciones que representa.

Los resultados obtenidos mantienen un carácter descriptivo y post-test. En consecuencia, las tasas observadas sobre 2026 se utilizarán para caracterizar el comportamiento del sistema definitivo, pero no para modificar retrospectivamente el threshold seleccionado sobre 2025.

In [13]:
# ---------------------------------------------------------
# 1. Recuperar las frecuencias de la matriz de confusión
# ---------------------------------------------------------

tp = int(
    (
        external_alert_analysis["confusion_category"]
        == "TP"
    ).sum()
)

fp = int(
    (
        external_alert_analysis["confusion_category"]
        == "FP"
    ).sum()
)

tn = int(
    (
        external_alert_analysis["confusion_category"]
        == "TN"
    ).sum()
)

fn = int(
    (
        external_alert_analysis["confusion_category"]
        == "FN"
    ).sum()
)


# ---------------------------------------------------------
# 2. Calcular los indicadores desde la perspectiva de las alertas
# ---------------------------------------------------------

alert_precision = (
    tp / (tp + fp)
)

false_alert_share = (
    fp / (tp + fp)
)


# ---------------------------------------------------------
# 3. Calcular los indicadores desde la perspectiva de los retrasos
# ---------------------------------------------------------

delay_coverage = (
    tp / (tp + fn)
)

missed_delay_share = (
    fn / (tp + fn)
)


# ---------------------------------------------------------
# 4. Calcular los indicadores desde la perspectiva de los vuelos sin alerta
# ---------------------------------------------------------

no_alert_correct_share = (
    tn / (tn + fn)
)

delay_share_without_alert = (
    fn / (tn + fn)
)


# ---------------------------------------------------------
# 5. Construir la descomposición global de la matriz de confusión
# ---------------------------------------------------------

confusion_operational_summary = pd.DataFrame(
    {
        "category": [
            "TP",
            "FP",
            "TN",
            "FN",
        ],
        "count": [
            tp,
            fp,
            tn,
            fn,
        ],
        "population_share": [
            tp / total_flights,
            fp / total_flights,
            tn / total_flights,
            fn / total_flights,
        ],
    }
)

display(confusion_operational_summary)


# ---------------------------------------------------------
# 6. Construir los principales indicadores operativos
# ---------------------------------------------------------

operational_indicator_summary = pd.DataFrame(
    {
        "indicator": [
            "Precisión de las alertas",
            "Proporción de falsas alertas",
            "Cobertura de retrasos",
            "Proporción de retrasos no detectados",
            "Proporción correcta entre vuelos sin alerta",
            "Proporción de retrasos entre vuelos sin alerta",
        ],
        "value": [
            alert_precision,
            false_alert_share,
            delay_coverage,
            missed_delay_share,
            no_alert_correct_share,
            delay_share_without_alert,
        ],
    }
)

display(operational_indicator_summary)

,category,count,population_share
0,TP,525105,0.169255
1,FP,1553063,0.500593
2,TN,892540,0.287689
3,FN,131739,0.042463


,indicator,value
0,Precisión de las alertas,0.252677
1,Proporción de falsas alertas,0.747323
2,Cobertura de retrasos,0.799436
3,Proporción de retrasos no detectados,0.200564
4,Proporción correcta entre vuelos sin alerta,0.871384
5,Proporción de retrasos entre vuelos sin alerta,0.128616


#### Interpretación

La descomposición global de las decisiones del sistema muestra que, sobre los 3,102,447 vuelos analizados, 525,105 corresponden a verdaderos positivos, 1,553,063 a falsos positivos, 892,540 a verdaderos negativos y 131,739 a falsos negativos. En términos de la población completa, estas categorías representan respectivamente el 16.93 %, 50.06 %, 28.77 % y 4.25 % de las observaciones.

Desde la perspectiva de las alertas emitidas, el 25.27 % se corresponde con vuelos que posteriormente presentan un retraso igual o superior a 15 minutos, mientras que el 74.73 % constituye falsas alertas. Por tanto, aunque el sistema consigue identificar una parte considerable de los retrasos reales, la mayor parte de las alertas generadas no se materializa posteriormente en el evento objetivo.

Desde la perspectiva de los retrasos realmente observados, el sistema alcanza una cobertura del 79.94 %, dejando sin alerta el 20.06 % de los retrasos. Este resultado reproduce sobre el período externo de 2026 el comportamiento esperado a partir del criterio operativo fijado previamente, orientado a mantener un recall próximo al 80 %.

El análisis de los vuelos que permanecen sin alerta aporta una perspectiva complementaria. El 87.14 % de estas observaciones no presenta posteriormente un retraso igual o superior a 15 minutos, mientras que el 12.86 % sí lo presenta. Esta última proporción representa el riesgo residual observado dentro del grupo no alertado y permite expresar de forma operativa las consecuencias asociadas a los falsos negativos.

En conjunto, los resultados evidencian el compromiso central del sistema definitivo: una elevada cobertura de los retrasos se consigue mediante una estrategia de alerta intensiva, cuya principal contrapartida es una elevada proporción de falsas alertas. Esta característica resulta fundamental para interpretar el sistema como un mecanismo de alerta temprana y priorización del riesgo, y no como un predictor individual de alta certeza.

### 3.3 Distribución del score de riesgo y relación con el threshold

Los resultados anteriores muestran que el sistema alcanza una cobertura elevada de los retrasos, pero presenta simultáneamente una proporción considerable de falsas alertas. Para comprender mejor este comportamiento resulta necesario analizar la distribución del score de riesgo producido por el modelo y su relación con el threshold de decisión establecido en 0.406588.

El score constituye una medida continua de la intensidad con la que el modelo posiciona cada observación respecto a la clase positiva. Dado que el modelo fue entrenado utilizando ponderación de clases y no se aplicó posteriormente un procedimiento específico de calibración, estos valores se interpretarán como puntuaciones relativas de riesgo y no como probabilidades calibradas de retraso.

El análisis se realizará tanto sobre la población completa como sobre las cuatro categorías de la matriz de confusión. Para cada grupo se calcularán el número de observaciones, la media, la mediana y diferentes percentiles del score. Esta caracterización permitirá determinar si los verdaderos positivos reciben puntuaciones sustancialmente superiores a las falsas alertas y si los falsos negativos se encuentran principalmente próximos al threshold o presentan scores claramente inferiores.

La posición de las observaciones respecto al threshold también permitirá estudiar la intensidad de las decisiones. Una elevada concentración de observaciones cerca del punto de corte indicaría que una parte importante de las clasificaciones se produce en una región de separación reducida, mientras que una mayor distancia entre las distribuciones sugeriría una diferenciación más clara entre los distintos resultados.

Este análisis mantiene un carácter estrictamente descriptivo y post-test. El comportamiento observado en 2026 no se utilizará para modificar el threshold, sino exclusivamente para comprender la capacidad de separación del modelo definitivo y las características de sus errores.

In [14]:
# ---------------------------------------------------------
# 1. Definir los percentiles necesarios para caracterizar el score
# ---------------------------------------------------------

score_percentiles = [
    0.05,
    0.25,
    0.50,
    0.75,
    0.95,
]


# ---------------------------------------------------------
# 2. Caracterizar la distribución global del score
# ---------------------------------------------------------

global_score_summary = (
    external_alert_analysis["risk_score"]
    .describe(
        percentiles=score_percentiles
    )
    .to_frame(name="value")
)

display(global_score_summary)


# ---------------------------------------------------------
# 3. Caracterizar el score por categoría de la matriz de confusión
# ---------------------------------------------------------

score_by_confusion = (
    external_alert_analysis
    .groupby(
        "confusion_category",
        observed=True,
    )["risk_score"]
    .agg(
        observations="size",
        mean_score="mean",
        median_score="median",
        std_score="std",
        min_score="min",
        max_score="max",
    )
)

confusion_quantiles = (
    external_alert_analysis
    .groupby(
        "confusion_category",
        observed=True,
    )["risk_score"]
    .quantile(score_percentiles)
    .unstack()
    .rename(
        columns={
            0.05: "p05",
            0.25: "p25",
            0.50: "p50",
            0.75: "p75",
            0.95: "p95",
        }
    )
)

score_by_confusion = (
    score_by_confusion
    .join(confusion_quantiles)
    .reset_index()
)

display(score_by_confusion)


# ---------------------------------------------------------
# 4. Calcular la distancia del score respecto al threshold
# ---------------------------------------------------------

external_alert_analysis[
    "threshold_distance"
] = (
    external_alert_analysis["risk_score"]
    - final_decision_threshold
).astype(np.float32)


# ---------------------------------------------------------
# 5. Caracterizar la proximidad de las observaciones al threshold
# ---------------------------------------------------------

threshold_margins = [
    0.01,
    0.025,
    0.05,
]

threshold_proximity_rows = []

for margin in threshold_margins:
    within_margin = (
        external_alert_analysis[
            "threshold_distance"
        ].abs()
        <= margin
    )

    observations_within_margin = int(
        within_margin.sum()
    )

    threshold_proximity_rows.append(
        {
            "margin": margin,
            "lower_score": (
                final_decision_threshold - margin
            ),
            "upper_score": (
                final_decision_threshold + margin
            ),
            "observations": observations_within_margin,
            "population_share": (
                observations_within_margin
                / total_flights
            ),
        }
    )

threshold_proximity_summary = pd.DataFrame(
    threshold_proximity_rows
)

display(threshold_proximity_summary)

,value
count,3.102447e+06
mean,4.708516e-01
std,1.291528e-01
min,1.250792e-05
5%,2.560207e-01
25%,3.707010e-01
50%,4.774902e-01
75%,5.711643e-01
95%,6.726011e-01
max,8.854018e-01


,confusion_category,observations,mean_score,median_score,std_score,min_score,max_score,p05,p25,p50,p75,p95
0,TP,525105,0.565422,0.566234,0.084190,0.406588,0.878012,0.428531,0.499678,0.566234,0.627873,0.704336
1,FP,1553063,0.537322,0.531155,0.081109,0.406588,0.885402,0.419600,0.470550,0.531155,0.594969,0.681274
2,TN,892540,0.320112,0.328217,0.058224,0.004030,0.406588,0.213104,0.279772,0.328217,0.368426,0.398949
3,FN,131739,0.331546,0.341600,0.054401,0.000013,0.406587,0.227426,0.296198,0.341600,0.376346,0.400723


,margin,lower_score,upper_score,observations,population_share
0,0.010,0.396588,0.416588,141212,0.045516
1,0.025,0.381588,0.431588,352952,0.113766
2,0.050,0.356588,0.456588,708753,0.228450


#### Interpretación

La distribución global del score de riesgo presenta una media de 0.4709 y una mediana de 0.4775, con un rango comprendido aproximadamente entre 0.00001 y 0.8854. El threshold definitivo de 0.406588 se encuentra por debajo de la mediana de la distribución, lo que resulta coherente con la elevada tasa de alerta observada previamente, situada en el 66.98 % de la población.

La comparación entre las categorías de la matriz de confusión evidencia diferencias en la puntuación asignada por el modelo, aunque estas diferencias son relativamente reducidas. Los verdaderos positivos presentan un score medio de 0.5654 y una mediana de 0.5662, frente a una media de 0.5373 y una mediana de 0.5312 para los falsos positivos. La diferencia entre los scores medios de ambos grupos es, por tanto, de aproximadamente 0.0281.

La superposición también resulta visible a través de los percentiles. El 25 % de los verdaderos positivos presenta scores iguales o inferiores a 0.4997, mientras que el 75 % de los falsos positivos alcanza valores de hasta 0.5950. Por tanto, amplias regiones de la distribución contienen simultáneamente alertas que posteriormente se corresponden con retrasos reales y alertas que no se materializan en el evento objetivo. Esta superposición ayuda a explicar la limitada precisión observada en el análisis anterior.

Por debajo del threshold se reproduce un comportamiento similar. Los falsos negativos presentan un score medio de 0.3315, ligeramente superior al 0.3201 observado entre los verdaderos negativos. Sus medianas son respectivamente 0.3416 y 0.3282. La proximidad entre ambas distribuciones muestra que el modelo tampoco establece una separación pronunciada entre los vuelos correctamente descartados y los retrasos que permanecen sin detectar.

El análisis de proximidad al threshold muestra que 141,212 observaciones, equivalentes al 4.55 % de la población, se encuentran dentro de una banda de ±0.01 alrededor del punto de corte. La proporción aumenta al 11.38 % para una banda de ±0.025 y al 22.85 % para ±0.05. En consecuencia, existe una fracción relevante de observaciones relativamente próxima a la frontera de decisión, aunque la limitada separación entre verdaderos y falsos positivos no puede atribuirse exclusivamente a los casos inmediatamente cercanos al threshold.

En conjunto, los resultados confirman que el modelo contiene señal para ordenar parcialmente las observaciones según su riesgo, pero la separación entre los diferentes resultados reales es limitada. Esta evidencia es consistente con la elevada cobertura acompañada de una baja precisión observada previamente y refuerza la interpretación del sistema como un mecanismo de screening o priorización de riesgo, más que como un predictor individual de alta certeza.

### 3.4 Síntesis global del comportamiento del sistema

Los análisis anteriores han permitido caracterizar el sistema desde tres perspectivas complementarias: la intensidad con la que genera alertas, la correspondencia de estas decisiones con los retrasos realmente observados y la distribución del score de riesgo que sustenta la clasificación. La integración de estas dimensiones permite establecer una lectura global del comportamiento operativo del modelo definitivo sobre el período externo de 2026.

La síntesis debe considerar conjuntamente cobertura y selectividad. El sistema consigue mantener una cobertura cercana al 80 % de los retrasos observados, pero para alcanzar este nivel genera alertas sobre aproximadamente dos tercios de la población. Como consecuencia, la mayor parte de las alertas emitidas corresponde a falsos positivos y la precisión individual de la alerta permanece limitada.

El análisis del score aporta una explicación complementaria a este comportamiento. Aunque los vuelos retrasados tienden a recibir puntuaciones superiores, las distribuciones de verdaderos y falsos positivos presentan una superposición considerable. Por tanto, la elevada proporción de falsas alertas no puede atribuirse únicamente a observaciones situadas inmediatamente alrededor del threshold, sino que refleja una capacidad limitada del modelo para separar con claridad los resultados individuales.

En este subbloque se consolidarán exclusivamente los indicadores necesarios para resumir estas propiedades. La tabla resultante constituirá un producto de síntesis del análisis global y servirá como referencia para los bloques posteriores, evitando repetir cálculos ya realizados.

La interpretación continuará siendo estrictamente descriptiva y post-test. Los resultados obtenidos sobre 2026 permiten caracterizar las fortalezas y limitaciones del sistema definitivo, pero no se utilizarán para modificar el modelo, el preprocesamiento o el threshold previamente fijado.

In [15]:
# ---------------------------------------------------------
# 1. Calcular los indicadores relativos de concentración del riesgo
# ---------------------------------------------------------

alert_delay_lift = (
    alert_precision / observed_delay_rate
)

no_alert_delay_relative_risk = (
    delay_share_without_alert / observed_delay_rate
)

no_alert_relative_risk_reduction = (
    1 - no_alert_delay_relative_risk
)


# ---------------------------------------------------------
# 2. Recuperar indicadores de separación del score
# ---------------------------------------------------------

tp_mean_score = float(
    score_by_confusion.loc[
        score_by_confusion["confusion_category"] == "TP",
        "mean_score",
    ].iloc[0]
)

fp_mean_score = float(
    score_by_confusion.loc[
        score_by_confusion["confusion_category"] == "FP",
        "mean_score",
    ].iloc[0]
)

tn_mean_score = float(
    score_by_confusion.loc[
        score_by_confusion["confusion_category"] == "TN",
        "mean_score",
    ].iloc[0]
)

fn_mean_score = float(
    score_by_confusion.loc[
        score_by_confusion["confusion_category"] == "FN",
        "mean_score",
    ].iloc[0]
)

tp_fp_score_gap = (
    tp_mean_score - fp_mean_score
)

fn_tn_score_gap = (
    fn_mean_score - tn_mean_score
)


# ---------------------------------------------------------
# 3. Recuperar la proporción próxima al threshold
# ---------------------------------------------------------

threshold_share_005 = float(
    threshold_proximity_summary.loc[
        np.isclose(
            threshold_proximity_summary["margin"],
            0.05,
        ),
        "population_share",
    ].iloc[0]
)


# ---------------------------------------------------------
# 4. Construir la síntesis global del sistema
# ---------------------------------------------------------

global_system_summary = pd.DataFrame(
    {
        "dimension": [
            "Población",
            "Prevalencia",
            "Intensidad de alerta",
            "Cobertura",
            "Selectividad",
            "Selectividad",
            "Riesgo residual",
            "Concentración del riesgo",
            "Concentración del riesgo",
            "Separación del score",
            "Separación del score",
            "Proximidad al threshold",
        ],
        "indicator": [
            "Vuelos analizados",
            "Tasa observada de retraso",
            "Tasa de alerta",
            "Cobertura de retrasos",
            "Precisión de las alertas",
            "Proporción de falsas alertas",
            "Retrasos entre vuelos sin alerta",
            "Lift de retraso entre alertados",
            "Riesgo relativo entre no alertados",
            "Diferencia media TP-FP",
            "Diferencia media FN-TN",
            "Observaciones dentro de ±0.05",
        ],
        "value": [
            total_flights,
            observed_delay_rate,
            alert_rate,
            delay_coverage,
            alert_precision,
            false_alert_share,
            delay_share_without_alert,
            alert_delay_lift,
            no_alert_delay_relative_risk,
            tp_fp_score_gap,
            fn_tn_score_gap,
            threshold_share_005,
        ],
    }
)

display(global_system_summary)


# ---------------------------------------------------------
# 5. Persistir la síntesis para su reutilización posterior
# ---------------------------------------------------------

global_system_summary_path = (
    operational_results_path
    / "global_system_summary_2026.parquet"
)

global_system_summary.to_parquet(
    global_system_summary_path,
    index=False,
)

print(
    "\nSíntesis global persistida:",
    global_system_summary_path.exists(),
)

print(
    "Ruta:",
    global_system_summary_path,
)

,dimension,indicator,value
0,Población,Vuelos analizados,3.102447e+06
1,Prevalencia,Tasa observada de retraso,2.117180e-01
2,Intensidad de alerta,Tasa de alerta,6.698480e-01
3,Cobertura,Cobertura de retrasos,7.994364e-01
4,Selectividad,Precisión de las alertas,2.526769e-01
5,Selectividad,Proporción de falsas alertas,7.473231e-01
6,Riesgo residual,Retrasos entre vuelos sin alerta,1.286163e-01
7,Concentración del riesgo,Lift de retraso entre alertados,1.193459e+00
8,Concentración del riesgo,Riesgo relativo entre no alertados,6.074887e-01
9,Separación del score,Diferencia media TP-FP,2.809972e-02



Síntesis global persistida: True
Ruta: G:\My Drive\MASTER Big Data\TFM\results\operational_analysis\global_system_summary_2026.parquet


## 4. Análisis temporal del comportamiento de las alertas

La caracterización global ha mostrado que el sistema mantiene una cobertura elevada de los retrasos, aunque acompañada de una alta tasa de alertas y una proporción considerable de falsos positivos. Sin embargo, estos indicadores resumen conjuntamente cinco meses de operación y no permiten determinar si el comportamiento observado permanece estable a lo largo del período externo.

El análisis temporal permite estudiar esta cuestión sin introducir nuevas decisiones de modelado. Dado que enero–mayo de 2026 constituye el período utilizado para la evaluación externa definitiva, cualquier desagregación realizada en este bloque tendrá carácter exclusivamente descriptivo y post-test. Las diferencias identificadas entre períodos no se utilizarán para modificar el threshold, reajustar el modelo ni seleccionar configuraciones alternativas.

La primera dimensión de análisis será mensual. Para cada mes se considerarán conjuntamente el volumen de vuelos, la prevalencia real de retrasos, la tasa de alerta, la cobertura de los retrasos y la precisión de las alertas. La utilización simultánea de volúmenes y tasas permitirá evitar interpretaciones basadas en porcentajes correspondientes a poblaciones de distinto tamaño.

Posteriormente se analizará la evolución temporal de los errores mediante las categorías de la matriz de confusión. Esta perspectiva permitirá determinar si las falsas alertas y los retrasos no detectados mantienen una distribución relativamente estable o si presentan concentraciones temporales específicas dentro del período estudiado.

El análisis incorporará también el comportamiento del score de riesgo, con el objetivo de comprobar si su distribución experimenta desplazamientos entre meses. Estas variaciones se interpretarán como evidencia descriptiva de posibles cambios en las características de la población o en la respuesta del modelo, sin atribuirles causalidad ni considerarlas automáticamente evidencia de drift estadístico.

Finalmente, los indicadores temporales se consolidarán en un producto reutilizable que permita sintetizar la estabilidad operacional observada durante los cinco meses y preparar los resultados necesarios para la posterior fase de visualización.

En este bloque se abordarán progresivamente:

1. la evolución mensual del volumen de vuelos, la prevalencia de retrasos y la intensidad de las alertas;
2. la evolución mensual de la cobertura, la precisión y la composición de los errores del sistema;
3. la evolución temporal de la distribución del score de riesgo;
4. la síntesis y persistencia de los principales indicadores temporales.

### 4.1 Evolución mensual del volumen, los retrasos y las alertas

La caracterización temporal comienza comparando mensualmente el volumen de vuelos, la frecuencia real de retrasos y la intensidad con la que el sistema genera alertas. Esta primera aproximación permite determinar si los valores globales obtenidos para enero–mayo de 2026 representan un comportamiento relativamente homogéneo o si existen diferencias relevantes entre los meses que componen el período externo.

Para cada mes se calcularán el número total de vuelos, el número de retrasos observados y el número de alertas generadas. Estas frecuencias se acompañarán de la prevalencia mensual de retrasos y de la tasa mensual de alerta, de manera que las diferencias porcentuales puedan interpretarse conjuntamente con el tamaño de la población sobre la que se producen.

También se calculará la diferencia, expresada en puntos porcentuales, entre la tasa de alerta y la prevalencia observada de retrasos. Este indicador permitirá cuantificar mensualmente la distancia entre la frecuencia con la que el sistema clasifica vuelos como situaciones de riesgo elevado y la frecuencia con la que realmente se observa el evento objetivo.

El objetivo de este análisis no es identificar un mes óptimo ni modificar el funcionamiento del sistema, sino caracterizar su comportamiento temporal sobre el período externo. Las diferencias observadas se interpretarán de forma descriptiva y no se atribuirán automáticamente a cambios en el modelo, estacionalidad o drift sin evidencia adicional.

In [16]:
# ---------------------------------------------------------
# 1. Construir la dimensión mensual de análisis
# ---------------------------------------------------------

external_alert_analysis["analysis_month"] = (
    external_alert_analysis[date_column]
    .dt.to_period("M")
    .astype(str)
)


# ---------------------------------------------------------
# 2. Agregar los principales indicadores por mes
# ---------------------------------------------------------

monthly_volume_summary = (
    external_alert_analysis
    .groupby(
        "analysis_month",
        observed=True,
    )
    .agg(
        flights=(target_column, "size"),
        observed_delays=(target_column, "sum"),
        alerts=("alert", "sum"),
    )
    .reset_index()
)


# ---------------------------------------------------------
# 3. Calcular las tasas mensuales
# ---------------------------------------------------------

monthly_volume_summary[
    "observed_delay_rate"
] = (
    monthly_volume_summary["observed_delays"]
    / monthly_volume_summary["flights"]
)

monthly_volume_summary[
    "alert_rate"
] = (
    monthly_volume_summary["alerts"]
    / monthly_volume_summary["flights"]
)

monthly_volume_summary[
    "alert_delay_gap_pp"
] = (
    monthly_volume_summary["alert_rate"]
    - monthly_volume_summary["observed_delay_rate"]
) * 100


# ---------------------------------------------------------
# 4. Calcular la relación entre alertas y retrasos observados
# ---------------------------------------------------------

monthly_volume_summary[
    "alerts_per_observed_delay"
] = (
    monthly_volume_summary["alerts"]
    / monthly_volume_summary["observed_delays"]
)


# ---------------------------------------------------------
# 5. Validar la cobertura de la agregación temporal
# ---------------------------------------------------------

monthly_totals_valid = (
    int(monthly_volume_summary["flights"].sum())
    == total_flights
    and int(
        monthly_volume_summary[
            "observed_delays"
        ].sum()
    )
    == total_observed_delays
    and int(
        monthly_volume_summary[
            "alerts"
        ].sum()
    )
    == total_alerts
)


# ---------------------------------------------------------
# 6. Mostrar los resultados mensuales
# ---------------------------------------------------------

display(monthly_volume_summary)

print(
    "\nCobertura temporal completamente válida:",
    monthly_totals_valid,
)

,analysis_month,flights,observed_delays,alerts,observed_delay_rate,alert_rate,alert_delay_gap_pp,alerts_per_observed_delay
0,2026-01,572599,118672,399200,0.207251,0.697172,48.992052,3.363894
1,2026-02,554297,107998,331460,0.194838,0.597983,40.314488,3.069131
2,2026-03,653297,157756,439607,0.241477,0.672905,43.142858,2.786626
3,2026-04,653335,130203,432425,0.19929,0.661873,46.258351,3.32116
4,2026-05,668919,142215,475476,0.212604,0.710813,49.82083,3.34336



Cobertura temporal completamente válida: True


#### Interpretación

La agregación mensual reproduce correctamente la totalidad de los 3,102,447 vuelos, los 656,844 retrasos observados y las 2,078,168 alertas generadas durante el período externo, por lo que la cobertura temporal del análisis queda validada.

La prevalencia mensual de retrasos presenta una variación moderada entre los cinco meses analizados. El valor mínimo se registra en febrero, con un 19.48 %, mientras que el máximo corresponde a marzo, con un 24.15 %. Enero, abril y mayo presentan prevalencias del 20.73 %, 19.93 % y 21.26 %, respectivamente.

La intensidad de las alertas muestra una variación más pronunciada. La tasa mensual oscila entre el 59.80 % registrado en febrero y el 71.08 % observado en mayo, lo que supone una amplitud de aproximadamente 11.28 puntos porcentuales. Enero presenta igualmente una tasa elevada, del 69.72 %, mientras que marzo y abril alcanzan el 67.29 % y el 66.19 %, respectivamente.

La evolución de las alertas no reproduce de forma paralela la evolución de la prevalencia real de retrasos. Marzo constituye el ejemplo más evidente: registra la mayor prevalencia del período, con un 24.15 %, pero su tasa de alerta, del 67.29 %, permanece por debajo de las observadas en enero y mayo. Por tanto, las diferencias mensuales en la intensidad de alerta no pueden explicarse únicamente por cambios en la frecuencia observada del evento objetivo.

La distancia entre la tasa de alerta y la prevalencia de retrasos permanece elevada durante todo el período. El menor diferencial se observa en febrero, con 40.31 puntos porcentuales, mientras que el máximo alcanza 49.82 puntos porcentuales en mayo. En ningún mes, por tanto, desaparece la característica global identificada previamente: el sistema clasifica como situaciones de riesgo elevado una proporción de vuelos sustancialmente superior a la frecuencia con la que posteriormente se observa un retraso.

La relación entre alertas y retrasos observados oscila entre 2.79 en marzo y 3.36 en enero. Aunque este indicador no mide directamente la precisión de las alertas, confirma que la intensidad operativa del sistema presenta variaciones temporales que justifican analizar a continuación si la cobertura, la precisión y la composición de los errores también permanecen estables entre meses.

### 4.2 Evolución mensual de la cobertura, la precisión y los errores

La variación mensual observada en la tasa de alerta no permite determinar por sí sola si el comportamiento del sistema mejora o empeora entre períodos. Para completar esta caracterización es necesario incorporar el resultado real de cada decisión y analizar cómo evolucionan temporalmente los verdaderos positivos, falsos positivos, verdaderos negativos y falsos negativos.

La cobertura mensual de retrasos se medirá mediante el recall, expresando qué proporción de los retrasos realmente observados había recibido previamente una alerta. Complementariamente, la precisión permitirá determinar qué proporción de las alertas emitidas en cada mes se corresponde posteriormente con un retraso igual o superior a 15 minutos.

Se incorporarán también la proporción de falsas alertas y la proporción de retrasos no detectados. Estos indicadores constituyen los complementos directos de la precisión y el recall, respectivamente, y permiten trasladar las métricas predictivas a una interpretación más próxima al funcionamiento operativo del sistema.

La comparación se realizará conjuntamente con las frecuencias absolutas de las cuatro categorías de la matriz de confusión. De esta manera, las diferencias porcentuales podrán contextualizarse según el número real de vuelos afectados y no únicamente mediante tasas relativas.

El objetivo será determinar si el compromiso global identificado entre cobertura y falsas alertas permanece relativamente estable entre enero y mayo de 2026 o si existen meses en los que la relación entre ambos componentes cambia de forma relevante. Cualquier diferencia observada tendrá carácter descriptivo y post-test y no será utilizada para modificar retrospectivamente la regla de decisión.

In [17]:
# ---------------------------------------------------------
# 1. Construir las frecuencias mensuales de la matriz de confusión
# ---------------------------------------------------------

monthly_confusion_counts = (
    external_alert_analysis
    .groupby(
        [
            "analysis_month",
            "confusion_category",
        ],
        observed=True,
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)

for category in ["TP", "FP", "TN", "FN"]:
    if category not in monthly_confusion_counts.columns:
        monthly_confusion_counts[category] = 0


# ---------------------------------------------------------
# 2. Calcular los principales indicadores mensuales
# ---------------------------------------------------------

monthly_operational_summary = (
    monthly_volume_summary[
        [
            "analysis_month",
            "flights",
            "observed_delays",
            "alerts",
            "observed_delay_rate",
            "alert_rate",
        ]
    ]
    .merge(
        monthly_confusion_counts,
        on="analysis_month",
        how="left",
    )
)

monthly_operational_summary[
    "recall"
] = (
    monthly_operational_summary["TP"]
    / (
        monthly_operational_summary["TP"]
        + monthly_operational_summary["FN"]
    )
)

monthly_operational_summary[
    "precision"
] = (
    monthly_operational_summary["TP"]
    / (
        monthly_operational_summary["TP"]
        + monthly_operational_summary["FP"]
    )
)

monthly_operational_summary[
    "false_alert_share"
] = (
    monthly_operational_summary["FP"]
    / (
        monthly_operational_summary["TP"]
        + monthly_operational_summary["FP"]
    )
)

monthly_operational_summary[
    "missed_delay_share"
] = (
    monthly_operational_summary["FN"]
    / (
        monthly_operational_summary["TP"]
        + monthly_operational_summary["FN"]
    )
)


# ---------------------------------------------------------
# 3. Calcular el riesgo residual entre los vuelos sin alerta
# ---------------------------------------------------------

monthly_operational_summary[
    "delay_share_without_alert"
] = (
    monthly_operational_summary["FN"]
    / (
        monthly_operational_summary["TN"]
        + monthly_operational_summary["FN"]
    )
)


# ---------------------------------------------------------
# 4. Calcular el lift mensual de las alertas
# ---------------------------------------------------------

monthly_operational_summary[
    "alert_delay_lift"
] = (
    monthly_operational_summary["precision"]
    / monthly_operational_summary["observed_delay_rate"]
)


# ---------------------------------------------------------
# 5. Validar la reconstrucción mensual de la matriz de confusión
# ---------------------------------------------------------

monthly_confusion_valid = (
    int(monthly_operational_summary["TP"].sum()) == tp
    and int(monthly_operational_summary["FP"].sum()) == fp
    and int(monthly_operational_summary["TN"].sum()) == tn
    and int(monthly_operational_summary["FN"].sum()) == fn
)


# ---------------------------------------------------------
# 6. Seleccionar y mostrar los indicadores relevantes
# ---------------------------------------------------------

monthly_operational_display = (
    monthly_operational_summary[
        [
            "analysis_month",
            "flights",
            "TP",
            "FP",
            "TN",
            "FN",
            "recall",
            "precision",
            "false_alert_share",
            "missed_delay_share",
            "delay_share_without_alert",
            "alert_delay_lift",
        ]
    ]
    .copy()
)

display(monthly_operational_display)

print(
    "\nReconstrucción mensual de la matriz válida:",
    monthly_confusion_valid,
)


# ---------------------------------------------------------
# 7. Resumir la amplitud temporal de los indicadores principales
# ---------------------------------------------------------

temporal_metric_range = pd.DataFrame(
    {
        "metric": [
            "Recall",
            "Precision",
            "Proporción de falsas alertas",
            "Retrasos entre vuelos sin alerta",
            "Lift de retraso entre alertados",
        ],
        "minimum": [
            monthly_operational_summary["recall"].min(),
            monthly_operational_summary["precision"].min(),
            monthly_operational_summary["false_alert_share"].min(),
            monthly_operational_summary[
                "delay_share_without_alert"
            ].min(),
            monthly_operational_summary[
                "alert_delay_lift"
            ].min(),
        ],
        "maximum": [
            monthly_operational_summary["recall"].max(),
            monthly_operational_summary["precision"].max(),
            monthly_operational_summary["false_alert_share"].max(),
            monthly_operational_summary[
                "delay_share_without_alert"
            ].max(),
            monthly_operational_summary[
                "alert_delay_lift"
            ].max(),
        ],
    }
)

temporal_metric_range["range"] = (
    temporal_metric_range["maximum"]
    - temporal_metric_range["minimum"]
)

display(temporal_metric_range)

,analysis_month,flights,TP,FP,TN,FN,recall,precision,false_alert_share,missed_delay_share,delay_share_without_alert,alert_delay_lift
0,2026-01,572599,93077,306123,147804,25595,0.784321,0.233159,0.766841,0.215679,0.147608,1.125004
1,2026-02,554297,78556,252904,193395,29442,0.727384,0.237000,0.763000,0.272616,0.132123,1.216396
2,2026-03,653297,126752,312855,182686,31004,0.803469,0.288330,0.711670,0.196531,0.145089,1.194029
3,2026-04,653335,105425,327000,196132,24778,0.809697,0.243800,0.756200,0.190303,0.112163,1.223342
4,2026-05,668919,121295,354181,172523,20920,0.852899,0.255102,0.744898,0.147101,0.108146,1.199893



Reconstrucción mensual de la matriz válida: True


,metric,minimum,maximum,range
0,Recall,0.727384,0.852899,0.125515
1,Precision,0.233159,0.288330,0.055171
2,Proporción de falsas alertas,0.711670,0.766841,0.055171
3,Retrasos entre vuelos sin alerta,0.108146,0.147608,0.039462
4,Lift de retraso entre alertados,1.125004,1.223342,0.098337


#### Interpretación

La reconstrucción mensual de la matriz de confusión reproduce correctamente los resultados globales del período externo, confirmando la consistencia de la desagregación temporal. No obstante, los indicadores obtenidos muestran que el comportamiento operativo del sistema presenta variaciones relevantes entre enero y mayo de 2026.

La cobertura de los retrasos es el indicador que presenta la mayor variación. El recall oscila entre un mínimo del 72.74 % en febrero y un máximo del 85.29 % en mayo, lo que supone una amplitud de 12.55 puntos porcentuales. Enero alcanza un 78.43 %, mientras que marzo y abril registran valores del 80.35 % y 80.97 %, respectivamente. En consecuencia, el recall global del 79.94 % no representa un nivel de cobertura uniforme durante todo el período externo.

Febrero constituye el mes con menor intensidad de alerta y también con menor cobertura. La tasa de alerta desciende hasta el 59.80 % y el recall hasta el 72.74 %, dejando sin alerta el 27.26 % de los retrasos realmente observados durante ese mes. En mayo se produce la situación opuesta: la tasa de alerta alcanza el 71.08 % y el recall aumenta hasta el 85.29 %, reduciendo la proporción de retrasos no detectados al 14.71 %. Estos resultados muestran que las diferencias mensuales en la intensidad de alerta están asociadas, al menos descriptivamente, con diferencias relevantes en la cobertura alcanzada.

La precisión permanece reducida durante todo el período, aunque también presenta variación mensual. Su mínimo se registra en enero, con un 23.32 %, y su máximo en marzo, con un 28.83 %. Como consecuencia, la proporción de falsas alertas permanece elevada en los cinco meses, oscilando entre el 71.17 % y el 76.68 %. Por tanto, ningún mes elimina la principal limitación operativa identificada en el análisis global.

El riesgo residual entre los vuelos sin alerta también presenta diferencias temporales. La proporción de retrasos dentro de este grupo alcanza el 14.76 % en enero y disminuye progresivamente hasta el 10.81 % en mayo. Esta evolución indica que la capacidad del sistema para construir un grupo de menor riesgo tampoco permanece completamente constante durante el período analizado.

El lift mensual se mantiene entre 1.13 y 1.22. Esto confirma que en todos los meses los vuelos alertados presentan una prevalencia de retrasos superior a la correspondiente población mensual, pero la concentración adicional del riesgo continúa siendo moderada. Incluso en el mejor resultado, registrado en abril con un lift de 1.22, la separación entre el grupo alertado y la prevalencia base sigue siendo limitada.

En conjunto, el análisis temporal muestra que las métricas globales ocultan una variabilidad operacional relevante. El sistema mantiene en todos los meses una elevada proporción de falsas alertas y una capacidad moderada de concentración del riesgo, mientras que la cobertura presenta oscilaciones más pronunciadas. Estos resultados aconsejan interpretar el recall global del 79.94 % como una medida agregada del período externo y no como una garantía de cobertura constante a lo largo del tiempo.

### 4.3 Evolución temporal del score de riesgo

Las diferencias mensuales identificadas en la tasa de alerta y en la cobertura pueden estar acompañadas por cambios en la distribución del score de riesgo generado por el modelo. Dado que la regla de decisión utiliza un threshold fijo, cualquier desplazamiento de la distribución del score puede modificar la proporción de observaciones que queda situada por encima o por debajo de dicha frontera.

Para caracterizar este comportamiento se analizarán mensualmente la media, la mediana, la desviación estándar y varios percentiles representativos del score. Esta información permitirá evaluar si los meses con mayor intensidad de alerta presentan también una distribución desplazada hacia valores superiores.

Se incorporará además la distancia media respecto al threshold y la proporción de observaciones situadas dentro de una banda de ±0.05 alrededor de la frontera de decisión. Este último indicador permitirá determinar si determinados meses concentran una mayor proporción de observaciones próximas al punto de clasificación.

El análisis tendrá carácter estrictamente descriptivo. Un desplazamiento mensual del score no será interpretado automáticamente como drift del modelo ni como evidencia de deterioro, ya que las diferencias podrían estar asociadas a cambios en la composición temporal de la población analizada. El objetivo será únicamente comprobar si la variabilidad operacional observada en los subbloques anteriores aparece acompañada por cambios sistemáticos en la distribución del score.

In [18]:
# ---------------------------------------------------------
# 1. Calcular los estadísticos mensuales del score de riesgo
# ---------------------------------------------------------

monthly_score_summary = (
    external_alert_analysis
    .groupby(
        "analysis_month",
        observed=True,
    )["risk_score"]
    .agg(
        observations="size",
        mean_score="mean",
        median_score="median",
        std_score="std",
        min_score="min",
        max_score="max",
    )
    .reset_index()
)


# ---------------------------------------------------------
# 2. Calcular percentiles mensuales representativos
# ---------------------------------------------------------

monthly_score_quantiles = (
    external_alert_analysis
    .groupby(
        "analysis_month",
        observed=True,
    )["risk_score"]
    .quantile(
        [0.05, 0.25, 0.75, 0.95]
    )
    .unstack()
    .reset_index()
    .rename(
        columns={
            0.05: "p05_score",
            0.25: "p25_score",
            0.75: "p75_score",
            0.95: "p95_score",
        }
    )
)

monthly_score_summary = (
    monthly_score_summary
    .merge(
        monthly_score_quantiles,
        on="analysis_month",
        how="left",
    )
)


# ---------------------------------------------------------
# 3. Calcular la distancia media respecto al threshold
# ---------------------------------------------------------

monthly_threshold_distance = (
    external_alert_analysis
    .groupby(
        "analysis_month",
        observed=True,
    )["threshold_distance"]
    .agg(
        mean_threshold_distance="mean",
        median_threshold_distance="median",
    )
    .reset_index()
)

monthly_score_summary = (
    monthly_score_summary
    .merge(
        monthly_threshold_distance,
        on="analysis_month",
        how="left",
    )
)


# ---------------------------------------------------------
# 4. Calcular la proporción mensual próxima al threshold
# ---------------------------------------------------------

threshold_margin = 0.05

monthly_threshold_proximity = (
    external_alert_analysis
    .assign(
        within_threshold_margin=(
            external_alert_analysis[
                "threshold_distance"
            ].abs()
            <= threshold_margin
        )
    )
    .groupby(
        "analysis_month",
        observed=True,
    )["within_threshold_margin"]
    .mean()
    .reset_index(
        name="share_within_005"
    )
)

monthly_score_summary = (
    monthly_score_summary
    .merge(
        monthly_threshold_proximity,
        on="analysis_month",
        how="left",
    )
)


# ---------------------------------------------------------
# 5. Incorporar la tasa mensual de alerta para facilitar la comparación
# ---------------------------------------------------------

monthly_score_summary = (
    monthly_score_summary
    .merge(
        monthly_volume_summary[
            [
                "analysis_month",
                "alert_rate",
            ]
        ],
        on="analysis_month",
        how="left",
    )
)


# ---------------------------------------------------------
# 6. Validar la cobertura mensual del análisis del score
# ---------------------------------------------------------

monthly_score_valid = (
    int(
        monthly_score_summary[
            "observations"
        ].sum()
    )
    == total_flights
)


# ---------------------------------------------------------
# 7. Mostrar la evolución mensual del score
# ---------------------------------------------------------

display(monthly_score_summary)

print(
    "\nCobertura mensual del score válida:",
    monthly_score_valid,
)


# ---------------------------------------------------------
# 8. Resumir la amplitud temporal de los principales indicadores
# ---------------------------------------------------------

score_temporal_range = pd.DataFrame(
    {
        "metric": [
            "Media del score",
            "Mediana del score",
            "Desviación estándar",
            "Distancia media al threshold",
            "Observaciones dentro de ±0.05",
        ],
        "minimum": [
            monthly_score_summary[
                "mean_score"
            ].min(),
            monthly_score_summary[
                "median_score"
            ].min(),
            monthly_score_summary[
                "std_score"
            ].min(),
            monthly_score_summary[
                "mean_threshold_distance"
            ].min(),
            monthly_score_summary[
                "share_within_005"
            ].min(),
        ],
        "maximum": [
            monthly_score_summary[
                "mean_score"
            ].max(),
            monthly_score_summary[
                "median_score"
            ].max(),
            monthly_score_summary[
                "std_score"
            ].max(),
            monthly_score_summary[
                "mean_threshold_distance"
            ].max(),
            monthly_score_summary[
                "share_within_005"
            ].max(),
        ],
    }
)

score_temporal_range[
    "range"
] = (
    score_temporal_range["maximum"]
    - score_temporal_range["minimum"]
)

display(score_temporal_range)

,analysis_month,observations,mean_score,median_score,std_score,min_score,max_score,p05_score,p25_score,p75_score,p95_score,mean_threshold_distance,median_threshold_distance,share_within_005,alert_rate
0,2026-01,572599,0.481893,0.489112,0.129472,0.000013,0.830105,0.264530,0.382294,0.581791,0.683746,0.075305,0.082524,0.222272,0.697172
1,2026-02,554297,0.441056,0.446066,0.127645,0.087871,0.795795,0.231412,0.341024,0.539452,0.643201,0.034468,0.039478,0.242397,0.597983
2,2026-03,653297,0.472874,0.480681,0.129456,0.051418,0.805564,0.257851,0.371219,0.573852,0.673798,0.066286,0.074093,0.222844,0.672905
3,2026-04,653335,0.467614,0.473932,0.127722,0.024587,0.812770,0.256217,0.367591,0.567214,0.667414,0.061026,0.067344,0.231122,0.661873
4,2026-05,668919,0.487276,0.494814,0.126843,0.112178,0.885402,0.274900,0.388887,0.586713,0.683334,0.080688,0.088226,0.225045,0.710813



Cobertura mensual del score válida: True


,metric,minimum,maximum,range
0,Media del score,0.441056,0.487276,0.046220
1,Mediana del score,0.446066,0.494814,0.048747
2,Desviación estándar,0.126843,0.129472,0.002628
3,Distancia media al threshold,0.034468,0.080688,0.046220
4,Observaciones dentro de ±0.05,0.222272,0.242397,0.020125


#### Interpretación

El análisis mensual del score de riesgo reproduce correctamente las 3,102,447 observaciones del período externo y muestra diferencias temporales en la posición de su distribución. El score medio oscila entre 0.4411 en febrero y 0.4873 en mayo, mientras que la mediana varía entre 0.4461 y 0.4948. La amplitud temporal alcanza, por tanto, aproximadamente 0.0462 para la media y 0.0487 para la mediana.

Estas diferencias aparecen acompañadas por cambios coherentes en la tasa de alerta. Febrero presenta simultáneamente el menor score medio, de 0.4411, y la menor tasa de alerta, del 59.80 %. En sentido contrario, mayo registra el mayor score medio, de 0.4873, y la mayor tasa de alerta, del 71.08 %. Enero también presenta una distribución relativamente elevada, con un score medio de 0.4819 y una tasa de alerta del 69.72 %. Dado que el threshold permanece fijo durante todo el período, estos resultados muestran que los meses en los que la distribución se sitúa en valores superiores generan una mayor proporción de observaciones clasificadas como alerta.

El desplazamiento no se limita exclusivamente a la media. Entre febrero y mayo, por ejemplo, el percentil 25 aumenta desde 0.3410 hasta 0.3889 y el percentil 75 desde 0.5395 hasta 0.5867. La mediana experimenta igualmente un desplazamiento desde 0.4461 hasta 0.4948. La evolución conjunta de distintos puntos de la distribución indica que las diferencias mensuales afectan a una parte amplia de los scores y no únicamente a observaciones extremas.

En contraste, la dispersión del score permanece notablemente estable. La desviación estándar oscila únicamente entre 0.1268 y 0.1295, con una amplitud de aproximadamente 0.0026. Por tanto, las diferencias temporales identificadas se manifiestan principalmente como desplazamientos en la posición de la distribución y no como cambios importantes en su dispersión.

La distancia media respecto al threshold refleja necesariamente este mismo patrón, al pasar de 0.0345 en febrero a 0.0807 en mayo. Por su parte, la proporción de observaciones situadas dentro de una banda de ±0.05 alrededor del threshold presenta una variación relativamente reducida, desde el 22.23 % hasta el 24.24 %. Febrero registra incluso la mayor concentración próxima a la frontera, con un 24.24 %, pero esta diferencia es insuficiente para explicar por sí sola la variación mensual de la tasa de alerta.

En conjunto, la variabilidad operacional identificada previamente aparece acompañada por desplazamientos temporales de la distribución del score. Los meses con scores globalmente más elevados tienden a presentar una mayor intensidad de alerta, lo que resulta coherente con la utilización de un threshold fijo. No obstante, este resultado debe interpretarse exclusivamente como una caracterización descriptiva del comportamiento temporal del modelo. El análisis realizado no permite determinar si los desplazamientos responden a estacionalidad, cambios en la composición de los vuelos, drift de las variables de entrada u otros factores.

### 4.4 Síntesis y persistencia de los indicadores temporales

Los análisis mensuales realizados han mostrado que el comportamiento agregado del sistema durante enero–mayo de 2026 presenta variaciones que afectan tanto a la intensidad de las alertas como a la cobertura alcanzada y a la distribución del score de riesgo. Para cerrar esta caracterización temporal resulta conveniente integrar los principales indicadores en un único producto analítico.

La síntesis combinará, para cada mes, el volumen de vuelos, la prevalencia observada de retrasos, la tasa de alerta, el recall, la precisión, la proporción de falsas alertas, el riesgo residual entre los vuelos sin alerta y los principales estadísticos del score. Esta integración permitirá conservar conjuntamente las dimensiones necesarias para interpretar el comportamiento temporal sin depender de múltiples tablas intermedias.

El producto resultante tendrá una finalidad descriptiva y de reutilización. Su persistencia permitirá emplearlo posteriormente en la preparación de visualizaciones y en la documentación de resultados sin volver a procesar los más de tres millones de registros individuales del período externo.

Como validación final se comprobará que existen exactamente cinco meses, que el volumen agregado reproduce las 3,102,447 observaciones externas y que no existen valores ausentes en los indicadores seleccionados. También se verificará que los meses se encuentran correctamente ordenados y que el archivo persistido puede recuperarse manteniendo su estructura.

Esta síntesis no introduce nuevas decisiones sobre el modelo. Las diferencias temporales observadas continúan teniendo carácter post-test y se utilizarán exclusivamente para caracterizar el comportamiento operativo del sistema definitivo.

In [19]:
# ---------------------------------------------------------
# 1. Seleccionar los indicadores operativos mensuales
# ---------------------------------------------------------

temporal_operational_metrics = (
    monthly_operational_summary[
        [
            "analysis_month",
            "flights",
            "observed_delays",
            "alerts",
            "observed_delay_rate",
            "alert_rate",
            "recall",
            "precision",
            "false_alert_share",
            "missed_delay_share",
            "delay_share_without_alert",
            "alert_delay_lift",
        ]
    ]
    .copy()
)


# ---------------------------------------------------------
# 2. Seleccionar los indicadores mensuales del score
# ---------------------------------------------------------

temporal_score_metrics = (
    monthly_score_summary[
        [
            "analysis_month",
            "mean_score",
            "median_score",
            "std_score",
            "p05_score",
            "p25_score",
            "p75_score",
            "p95_score",
            "mean_threshold_distance",
            "share_within_005",
        ]
    ]
    .copy()
)


# ---------------------------------------------------------
# 3. Construir el producto temporal consolidado
# ---------------------------------------------------------

temporal_system_summary = (
    temporal_operational_metrics
    .merge(
        temporal_score_metrics,
        on="analysis_month",
        how="inner",
        validate="one_to_one",
    )
    .sort_values(
        "analysis_month"
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 4. Validar la consistencia del producto temporal
# ---------------------------------------------------------

expected_months = [
    "2026-01",
    "2026-02",
    "2026-03",
    "2026-04",
    "2026-05",
]

temporal_summary_validation = pd.DataFrame(
    {
        "validation": [
            "Cinco meses disponibles",
            "Meses esperados y ordenados",
            "Volumen total correcto",
            "Retrasos totales correctos",
            "Alertas totales correctas",
            "Ausencia de valores nulos",
        ],
        "valid": [
            len(temporal_system_summary) == 5,
            temporal_system_summary[
                "analysis_month"
            ].tolist()
            == expected_months,
            int(
                temporal_system_summary[
                    "flights"
                ].sum()
            )
            == total_flights,
            int(
                temporal_system_summary[
                    "observed_delays"
                ].sum()
            )
            == total_observed_delays,
            int(
                temporal_system_summary[
                    "alerts"
                ].sum()
            )
            == total_alerts,
            not temporal_system_summary.isna().any().any(),
        ],
    }
)


# ---------------------------------------------------------
# 5. Persistir el producto temporal consolidado
# ---------------------------------------------------------

temporal_system_summary_path = (
    operational_results_path
    / "temporal_system_summary_2026.parquet"
)

temporal_system_summary.to_parquet(
    temporal_system_summary_path,
    index=False,
)


# ---------------------------------------------------------
# 6. Recuperar el archivo persistido y validar su estructura
# ---------------------------------------------------------

persisted_temporal_system_summary = pd.read_parquet(
    temporal_system_summary_path
)

persisted_temporal_valid = (
    persisted_temporal_system_summary.shape
    == temporal_system_summary.shape
    and persisted_temporal_system_summary.columns.tolist()
    == temporal_system_summary.columns.tolist()
    and persisted_temporal_system_summary[
        "analysis_month"
    ].tolist()
    == expected_months
)


# ---------------------------------------------------------
# 7. Mostrar el producto y las validaciones finales
# ---------------------------------------------------------

display(temporal_system_summary)

display(temporal_summary_validation)

print(
    "\nArchivo temporal persistido:",
    temporal_system_summary_path.exists(),
)

print(
    "Recuperación del producto válida:",
    persisted_temporal_valid,
)

print(
    "Ruta:",
    temporal_system_summary_path,
)

,analysis_month,flights,observed_delays,alerts,observed_delay_rate,alert_rate,recall,precision,false_alert_share,missed_delay_share,...,alert_delay_lift,mean_score,median_score,std_score,p05_score,p25_score,p75_score,p95_score,mean_threshold_distance,share_within_005
0,2026-01,572599,118672,399200,0.207251,0.697172,0.784321,0.233159,0.766841,0.215679,...,1.125004,0.481893,0.489112,0.129472,0.264530,0.382294,0.581791,0.683746,0.075305,0.222272
1,2026-02,554297,107998,331460,0.194838,0.597983,0.727384,0.237000,0.763000,0.272616,...,1.216396,0.441056,0.446066,0.127645,0.231412,0.341024,0.539452,0.643201,0.034468,0.242397
2,2026-03,653297,157756,439607,0.241477,0.672905,0.803469,0.288330,0.711670,0.196531,...,1.194029,0.472874,0.480681,0.129456,0.257851,0.371219,0.573852,0.673798,0.066286,0.222844
3,2026-04,653335,130203,432425,0.19929,0.661873,0.809697,0.243800,0.756200,0.190303,...,1.223342,0.467614,0.473932,0.127722,0.256217,0.367591,0.567214,0.667414,0.061026,0.231122
4,2026-05,668919,142215,475476,0.212604,0.710813,0.852899,0.255102,0.744898,0.147101,...,1.199893,0.487276,0.494814,0.126843,0.274900,0.388887,0.586713,0.683334,0.080688,0.225045


,validation,valid
0,Cinco meses disponibles,True
1,Meses esperados y ordenados,True
2,Volumen total correcto,True
3,Retrasos totales correctos,True
4,Alertas totales correctas,True
5,Ausencia de valores nulos,True



Archivo temporal persistido: True
Recuperación del producto válida: True
Ruta: G:\My Drive\MASTER Big Data\TFM\results\operational_analysis\temporal_system_summary_2026.parquet


#### Interpretación

La síntesis temporal reproduce correctamente los 3,102,447 vuelos del período externo y confirma que el comportamiento del sistema presenta variaciones mensuales relevantes. La tasa de alerta oscila entre el 59.80 % y el 71.08 %, mientras que el recall varía entre el 72.74 % y el 85.29 %, por lo que el valor global del 79.94 % no representa una cobertura uniforme en el tiempo.

La precisión permanece limitada durante todos los meses, entre el 23.32 % y el 28.83 %, manteniéndose la proporción de falsas alertas por encima del 71 %. Por tanto, esta limitación no corresponde a un mes aislado, sino que persiste durante todo el período externo.

La distribución del score también presenta desplazamientos temporales: su media varía entre 0.4411 y 0.4873, mientras que su dispersión permanece prácticamente estable. Con un threshold fijo, estos desplazamientos son coherentes con las diferencias observadas en la intensidad de alerta y el recall.

En conjunto, los resultados evidencian variabilidad temporal en el comportamiento operacional del sistema, aunque no permiten atribuirla directamente a drift, estacionalidad o deterioro del modelo. Estas posibles causas requerirían un análisis específico adicional.

## 5. Análisis operacional y geográfico

Una vez caracterizado el comportamiento global y temporal del sistema, el análisis se amplía hacia las principales dimensiones operativas y geográficas disponibles antes de la salida programada del vuelo. El objetivo es determinar si la frecuencia de los retrasos y el comportamiento de las alertas presentan diferencias relevantes entre grupos de vuelos.

Este análisis mantiene un carácter estrictamente descriptivo y post-test. Las diferencias identificadas no se utilizarán para modificar el modelo, seleccionar nuevas variables ni reajustar el threshold definitivo.

Las comparaciones combinarán volumen de observaciones, prevalencia de retrasos, tasa de alerta, recall y precisión. Esta perspectiva resulta necesaria porque una categoría con una tasa elevada pero pocos vuelos puede tener menor relevancia operacional que otra con resultados menos extremos pero un volumen considerablemente superior.

Se estudiarán progresivamente las dimensiones asociadas al horario programado, los operadores aéreos y la estructura geográfica definida por los aeropuertos de origen y destino. Finalmente, los resultados se consolidarán en productos agregados reutilizables para la posterior preparación de visualizaciones.

En este bloque se abordarán progresivamente:

1. el comportamiento de las alertas según las franjas horarias programadas;
2. el comportamiento operacional según las principales aerolíneas;
3. la distribución geográfica de retrasos y alertas según los aeropuertos de origen y destino;
4. la identificación de concentraciones relevantes de errores operativos;
5. la síntesis y persistencia de los principales indicadores operacionales y geográficos.

### 5.1 Comportamiento según las franjas horarias programadas

El horario programado constituye una dimensión relevante para caracterizar el comportamiento operativo del sistema, ya que la prevalencia de retrasos puede variar a lo largo del día y modificar tanto la frecuencia como la efectividad de las alertas.

El análisis se centrará inicialmente en la franja horaria programada de salida (`DEP_TIME_BLK`). Para cada intervalo se calcularán el volumen de vuelos, la prevalencia observada de retrasos, la tasa de alerta, el recall, la precisión y la proporción de falsas alertas. De este modo, las diferencias entre franjas podrán evaluarse considerando conjuntamente frecuencia y volumen.

También se incorporará el lift de las alertas respecto a la prevalencia propia de cada franja. Este indicador permitirá distinguir entre una precisión elevada asociada simplemente a una mayor frecuencia de retrasos y una concentración adicional del evento dentro del grupo alertado.

El análisis será descriptivo y post-test. Las diferencias identificadas permitirán caracterizar cuándo funciona mejor o peor el sistema, pero no se utilizarán para modificar el modelo ni su regla de decisión.

In [20]:
# ---------------------------------------------------------
# 1. Construir la agregación por franja programada de salida
# ---------------------------------------------------------

departure_time_summary = (
    external_alert_analysis
    .groupby(
        "DEP_TIME_BLK",
        observed=True,
    )
    .agg(
        flights=(target_column, "size"),
        observed_delays=(target_column, "sum"),
        alerts=("alert", "sum"),
    )
    .reset_index()
)


# ---------------------------------------------------------
# 2. Construir la matriz de confusión por franja horaria
# ---------------------------------------------------------

departure_time_confusion = (
    external_alert_analysis
    .groupby(
        [
            "DEP_TIME_BLK",
            "confusion_category",
        ],
        observed=True,
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)

for category in ["TP", "FP", "TN", "FN"]:
    if category not in departure_time_confusion.columns:
        departure_time_confusion[category] = 0


# ---------------------------------------------------------
# 3. Integrar frecuencias y resultados de clasificación
# ---------------------------------------------------------

departure_time_summary = (
    departure_time_summary
    .merge(
        departure_time_confusion,
        on="DEP_TIME_BLK",
        how="left",
        validate="one_to_one",
    )
)


# ---------------------------------------------------------
# 4. Calcular los indicadores operativos
# ---------------------------------------------------------

departure_time_summary[
    "observed_delay_rate"
] = (
    departure_time_summary["observed_delays"]
    / departure_time_summary["flights"]
)

departure_time_summary[
    "alert_rate"
] = (
    departure_time_summary["alerts"]
    / departure_time_summary["flights"]
)

departure_time_summary[
    "recall"
] = (
    departure_time_summary["TP"]
    / (
        departure_time_summary["TP"]
        + departure_time_summary["FN"]
    )
)

departure_time_summary[
    "precision"
] = (
    departure_time_summary["TP"]
    / (
        departure_time_summary["TP"]
        + departure_time_summary["FP"]
    )
)

departure_time_summary[
    "false_alert_share"
] = (
    1
    - departure_time_summary["precision"]
)

departure_time_summary[
    "alert_delay_lift"
] = (
    departure_time_summary["precision"]
    / departure_time_summary["observed_delay_rate"]
)


# ---------------------------------------------------------
# 5. Ordenar cronológicamente las franjas de salida
# ---------------------------------------------------------

departure_time_summary[
    "departure_start_hour"
] = (
    departure_time_summary[
        "DEP_TIME_BLK"
    ]
    .astype(str)
    .str[:2]
    .astype(int)
)

departure_time_summary = (
    departure_time_summary
    .sort_values(
        "departure_start_hour"
    )
    .drop(
        columns="departure_start_hour"
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 6. Validar la cobertura de la agregación
# ---------------------------------------------------------

departure_time_valid = (
    int(
        departure_time_summary[
            "flights"
        ].sum()
    )
    == total_flights
    and int(
        departure_time_summary[
            "observed_delays"
        ].sum()
    )
    == total_observed_delays
    and int(
        departure_time_summary[
            "alerts"
        ].sum()
    )
    == total_alerts
)


# ---------------------------------------------------------
# 7. Mostrar los resultados
# ---------------------------------------------------------

departure_time_display = (
    departure_time_summary[
        [
            "DEP_TIME_BLK",
            "flights",
            "observed_delay_rate",
            "alert_rate",
            "recall",
            "precision",
            "false_alert_share",
            "alert_delay_lift",
        ]
    ]
    .copy()
)

display(departure_time_display)

print(
    "\nCobertura por franja de salida válida:",
    departure_time_valid,
)

,DEP_TIME_BLK,flights,observed_delay_rate,alert_rate,recall,precision,false_alert_share,alert_delay_lift
0,0001-0559,87493,0.092385,0.042975,0.074230,0.159574,0.840426,1.727286
1,0600-0659,209879,0.10446,0.051792,0.065636,0.132383,0.867617,1.267303
2,0700-0759,226450,0.127388,0.150316,0.183381,0.155410,0.844590,1.219974
3,0800-0859,211494,0.147815,0.285455,0.340477,0.176307,0.823693,1.192753
4,0900-0959,178254,0.161786,0.392249,0.471584,0.194508,0.805492,1.202255
5,1000-1059,198488,0.177875,0.499567,0.577749,0.205712,0.794288,1.1565
6,1100-1159,190075,0.192498,0.639905,0.714750,0.215013,0.784987,1.116963
7,1200-1259,195766,0.209893,0.742284,0.800049,0.226227,0.773773,1.07782
8,1300-1359,182109,0.223729,0.838218,0.886263,0.236552,0.763448,1.057318
9,1400-1459,184049,0.239431,0.930676,0.952504,0.245046,0.754954,1.023454



Cobertura por franja de salida válida: True


#### Interpretación

El comportamiento del sistema presenta una fuerte dependencia de la franja horaria de salida. La prevalencia de retrasos aumenta desde el 9.24 % en la primera franja hasta valores cercanos al 30 % durante la tarde y la noche, mientras que la tasa de alerta crece todavía con mayor intensidad.

A partir de las 14:00, la tasa de alerta supera el 93 % y llega a situarse por encima del 99 % entre las 16:00 y las 19:59. Como consecuencia, el recall se aproxima al 100 %, pero esta elevada cobertura se consigue alertando prácticamente a toda la población de esas franjas.

La precisión aumenta durante el día, alcanzando aproximadamente el 29 % en las últimas horas de la tarde. Sin embargo, el lift converge progresivamente hacia 1.00. Esto indica que, en esas franjas, el grupo alertado apenas concentra una prevalencia de retrasos superior a la población completa de la propia franja.

En conjunto, el análisis revela una limitación operacional importante: durante buena parte de la tarde el sistema obtiene una cobertura casi total mediante una clasificación muy poco selectiva. Por tanto, el elevado recall de estas franjas no debe interpretarse aisladamente como una mejora de la capacidad discriminativa.

### 5.2 Comportamiento según las principales aerolíneas

El análisis por aerolínea permite determinar si el comportamiento del sistema presenta diferencias relevantes entre operadores. Esta comparación resulta especialmente útil para identificar si la prevalencia de retrasos, la intensidad de las alertas y su capacidad de discriminación se distribuyen de forma homogénea entre compañías.

Para evitar conclusiones basadas en categorías con escaso soporte, las métricas se interpretarán conjuntamente con el volumen de vuelos. Para cada aerolínea se calcularán la prevalencia observada de retrasos, la tasa de alerta, el recall, la precisión, la proporción de falsas alertas y el lift respecto a su propia prevalencia.

Se utilizará `OP_UNIQUE_CARRIER` como dimensión principal, ya que identifica al operador efectivo del vuelo. El análisis mantiene un carácter descriptivo y post-test y no se utilizará para modificar el modelo ni su regla de decisión.

In [21]:
# ---------------------------------------------------------
# 1. Construir la agregación por aerolínea operadora
# ---------------------------------------------------------

carrier_summary = (
    external_alert_analysis
    .groupby(
        "OP_UNIQUE_CARRIER",
        observed=True,
    )
    .agg(
        flights=(target_column, "size"),
        observed_delays=(target_column, "sum"),
        alerts=("alert", "sum"),
    )
    .reset_index()
)


# ---------------------------------------------------------
# 2. Construir la matriz de confusión por aerolínea
# ---------------------------------------------------------

carrier_confusion = (
    external_alert_analysis
    .groupby(
        [
            "OP_UNIQUE_CARRIER",
            "confusion_category",
        ],
        observed=True,
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)

for category in ["TP", "FP", "TN", "FN"]:
    if category not in carrier_confusion.columns:
        carrier_confusion[category] = 0


# ---------------------------------------------------------
# 3. Integrar frecuencias y resultados de clasificación
# ---------------------------------------------------------

carrier_summary = (
    carrier_summary
    .merge(
        carrier_confusion,
        on="OP_UNIQUE_CARRIER",
        how="left",
        validate="one_to_one",
    )
)


# ---------------------------------------------------------
# 4. Calcular los indicadores operativos
# ---------------------------------------------------------

carrier_summary[
    "observed_delay_rate"
] = (
    carrier_summary["observed_delays"]
    / carrier_summary["flights"]
)

carrier_summary[
    "alert_rate"
] = (
    carrier_summary["alerts"]
    / carrier_summary["flights"]
)

carrier_summary[
    "recall"
] = (
    carrier_summary["TP"]
    / (
        carrier_summary["TP"]
        + carrier_summary["FN"]
    )
)

carrier_summary[
    "precision"
] = (
    carrier_summary["TP"]
    / (
        carrier_summary["TP"]
        + carrier_summary["FP"]
    )
)

carrier_summary[
    "false_alert_share"
] = (
    1
    - carrier_summary["precision"]
)

carrier_summary[
    "alert_delay_lift"
] = (
    carrier_summary["precision"]
    / carrier_summary["observed_delay_rate"]
)


# ---------------------------------------------------------
# 5. Calcular el peso de cada aerolínea en la población
# ---------------------------------------------------------

carrier_summary[
    "population_share"
] = (
    carrier_summary["flights"]
    / total_flights
)


# ---------------------------------------------------------
# 6. Ordenar las aerolíneas por volumen de vuelos
# ---------------------------------------------------------

carrier_summary = (
    carrier_summary
    .sort_values(
        "flights",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 7. Validar la cobertura de la agregación
# ---------------------------------------------------------

carrier_summary_valid = (
    int(
        carrier_summary["flights"].sum()
    )
    == total_flights
    and int(
        carrier_summary[
            "observed_delays"
        ].sum()
    )
    == total_observed_delays
    and int(
        carrier_summary["alerts"].sum()
    )
    == total_alerts
)


# ---------------------------------------------------------
# 8. Mostrar los resultados consolidados
# ---------------------------------------------------------

carrier_display = (
    carrier_summary[
        [
            "OP_UNIQUE_CARRIER",
            "flights",
            "population_share",
            "observed_delay_rate",
            "alert_rate",
            "recall",
            "precision",
            "false_alert_share",
            "alert_delay_lift",
        ]
    ]
    .copy()
)

display(carrier_display)

print(
    "\nNúmero de aerolíneas:",
    len(carrier_display),
)

print(
    "Cobertura por aerolínea válida:",
    carrier_summary_valid,
)

,OP_UNIQUE_CARRIER,flights,population_share,observed_delay_rate,alert_rate,recall,precision,false_alert_share,alert_delay_lift
0,WN,557982,0.179852,0.221824,0.697766,0.887004,0.281984,0.718016,1.271205
1,DL,408780,0.131761,0.178499,0.567836,0.698686,0.219632,0.780368,1.230436
2,AA,393936,0.126976,0.232665,0.777796,0.885953,0.265018,0.734982,1.139055
3,OO,342154,0.110285,0.213611,0.594481,0.696298,0.250197,0.749803,1.17127
4,UA,332937,0.107314,0.185257,0.65892,0.776877,0.218421,0.781579,1.179015
5,YX,144862,0.046693,0.202517,0.569494,0.695504,0.247327,0.752673,1.221267
6,AS,132614,0.042745,0.185305,0.60845,0.661471,0.201452,0.798548,1.087142
7,MQ,125950,0.040597,0.184422,0.640254,0.750603,0.216208,0.783792,1.172351
8,9E,99998,0.032232,0.226915,0.567731,0.675246,0.269887,0.730113,1.189375
9,B6,94444,0.030442,0.270393,0.91001,0.952187,0.282925,0.717075,1.046348



Número de aerolíneas: 19
Cobertura por aerolínea válida: True


#### Interpretación

El análisis por aerolínea muestra una variabilidad operacional relevante entre los 19 operadores analizados. La tasa de alerta oscila aproximadamente entre el 48.09 % y el 91.23 %, mientras que el recall varía entre el 58.27 % y el 95.22 %.

Los operadores con mayor recall no presentan necesariamente una mejor capacidad discriminativa. B6 y F9 superan el 93 % de cobertura, pero generan alertas en más del 91 % de sus vuelos y presentan lifts próximos a 1.00, lo que indica una selección poco diferenciada.

Por el contrario, algunos operadores como WN presentan un mejor equilibrio relativo entre cobertura y concentración del riesgo, con un recall del 88.70 % y un lift de 1.27. Aun así, la proporción de falsas alertas continúa siendo elevada en prácticamente todos los operadores.

En conjunto, los resultados confirman que el comportamiento del sistema depende también de la aerolínea operadora y que un recall elevado debe interpretarse siempre junto con la tasa de alerta y el lift.

### 5.3 Distribución geográfica según aeropuertos de origen y destino

La dimensión geográfica permite analizar si el comportamiento del sistema presenta diferencias relevantes según los aeropuertos implicados en la operación. Para ello se estudiarán separadamente los aeropuertos de origen y destino, evitando mezclar dos dimensiones que representan posiciones distintas dentro del trayecto.

Para cada aeropuerto se calcularán el volumen de vuelos, la prevalencia observada de retrasos, la tasa de alerta, el recall, la precisión, la proporción de falsas alertas y el lift de las alertas. Dado el elevado número potencial de aeropuertos, las comparaciones se acompañarán siempre del volumen de observaciones.

La caracterización principal se centrará en los aeropuertos con mayor actividad durante el período externo. Este criterio permite identificar patrones geográficos con relevancia operacional y reduce el riesgo de interpretar tasas extremas procedentes de categorías con escaso número de vuelos.

El análisis mantiene un carácter descriptivo y post-test. Las diferencias geográficas observadas permitirán caracterizar el comportamiento del sistema, pero no se utilizarán para modificar su configuración.

In [22]:
# ---------------------------------------------------------
# 1. Definir una función auxiliar para la agregación geográfica
# ---------------------------------------------------------

def build_airport_summary(data, airport_column):
    # ---------------------------------------------------------
    # 1.1. Calcular volumen, retrasos y alertas
    # ---------------------------------------------------------

    airport_summary = (
        data
        .groupby(
            airport_column,
            observed=True,
        )
        .agg(
            flights=(target_column, "size"),
            observed_delays=(target_column, "sum"),
            alerts=("alert", "sum"),
        )
        .reset_index()
    )

    # ---------------------------------------------------------
    # 1.2. Construir la matriz de confusión
    # ---------------------------------------------------------

    airport_confusion = (
        data
        .groupby(
            [
                airport_column,
                "confusion_category",
            ],
            observed=True,
        )
        .size()
        .unstack(
            fill_value=0
        )
        .reset_index()
    )

    for category in ["TP", "FP", "TN", "FN"]:
        if category not in airport_confusion.columns:
            airport_confusion[category] = 0

    # ---------------------------------------------------------
    # 1.3. Integrar los resultados
    # ---------------------------------------------------------

    airport_summary = (
        airport_summary
        .merge(
            airport_confusion,
            on=airport_column,
            how="left",
            validate="one_to_one",
        )
    )

    # ---------------------------------------------------------
    # 1.4. Calcular los indicadores operativos
    # ---------------------------------------------------------

    airport_summary["observed_delay_rate"] = (
        airport_summary["observed_delays"]
        / airport_summary["flights"]
    )

    airport_summary["alert_rate"] = (
        airport_summary["alerts"]
        / airport_summary["flights"]
    )

    airport_summary["recall"] = (
        airport_summary["TP"]
        / (
            airport_summary["TP"]
            + airport_summary["FN"]
        )
    )

    airport_summary["precision"] = (
        airport_summary["TP"]
        / (
            airport_summary["TP"]
            + airport_summary["FP"]
        )
    )

    airport_summary["false_alert_share"] = (
        1 - airport_summary["precision"]
    )

    airport_summary["alert_delay_lift"] = (
        airport_summary["precision"]
        / airport_summary["observed_delay_rate"]
    )

    airport_summary["population_share"] = (
        airport_summary["flights"]
        / total_flights
    )

    return (
        airport_summary
        .sort_values(
            "flights",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )


# ---------------------------------------------------------
# 2. Construir los resultados por origen y destino
# ---------------------------------------------------------

origin_summary = build_airport_summary(
    external_alert_analysis,
    "ORIGIN",
)

destination_summary = build_airport_summary(
    external_alert_analysis,
    "DEST",
)


# ---------------------------------------------------------
# 3. Validar la cobertura de ambas agregaciones
# ---------------------------------------------------------

origin_summary_valid = (
    int(origin_summary["flights"].sum()) == total_flights
    and int(origin_summary["observed_delays"].sum())
    == total_observed_delays
    and int(origin_summary["alerts"].sum()) == total_alerts
)

destination_summary_valid = (
    int(destination_summary["flights"].sum()) == total_flights
    and int(destination_summary["observed_delays"].sum())
    == total_observed_delays
    and int(destination_summary["alerts"].sum()) == total_alerts
)


# ---------------------------------------------------------
# 4. Seleccionar los aeropuertos con mayor volumen
# ---------------------------------------------------------

top_airports = 20

airport_display_columns = [
    "flights",
    "population_share",
    "observed_delay_rate",
    "alert_rate",
    "recall",
    "precision",
    "false_alert_share",
    "alert_delay_lift",
]

top_origin_summary = (
    origin_summary
    .head(top_airports)[
        ["ORIGIN"] + airport_display_columns
    ]
    .copy()
)

top_destination_summary = (
    destination_summary
    .head(top_airports)[
        ["DEST"] + airport_display_columns
    ]
    .copy()
)


# ---------------------------------------------------------
# 5. Mostrar los resultados principales
# ---------------------------------------------------------

print("Aeropuertos de origen con mayor volumen:")
display(top_origin_summary)

print("\nAeropuertos de destino con mayor volumen:")
display(top_destination_summary)

print(
    "\nNúmero de aeropuertos de origen:",
    len(origin_summary),
)

print(
    "Número de aeropuertos de destino:",
    len(destination_summary),
)

print(
    "Cobertura por origen válida:",
    origin_summary_valid,
)

print(
    "Cobertura por destino válida:",
    destination_summary_valid,
)

Aeropuertos de origen con mayor volumen:


,ORIGIN,flights,population_share,observed_delay_rate,alert_rate,recall,precision,false_alert_share,alert_delay_lift
0,ORD,155123,0.05,0.296468,0.772619,0.850421,0.326322,0.673678,1.100699
1,ATL,139855,0.045079,0.216152,0.683966,0.794046,0.250941,0.749059,1.160944
2,DEN,125889,0.040577,0.200701,0.827817,0.904298,0.219243,0.780757,1.09239
3,DFW,120536,0.038852,0.252821,0.85562,0.925510,0.273472,0.726528,1.081684
4,CLT,92686,0.029875,0.202771,0.740824,0.850750,0.232859,0.767141,1.148384
5,PHX,86074,0.027744,0.202721,0.645828,0.776893,0.243861,0.756139,1.20294
6,LAX,76674,0.024714,0.197433,0.606151,0.718259,0.233949,0.766051,1.184951
7,LAS,70672,0.022779,0.208145,0.79944,0.917539,0.238893,0.761107,1.147728
8,SEA,69643,0.022448,0.166578,0.67958,0.761572,0.186676,0.813324,1.120651
9,IAH,68865,0.022197,0.177768,0.714223,0.790394,0.196727,0.803273,1.106648



Aeropuertos de destino con mayor volumen:


,DEST,flights,population_share,observed_delay_rate,alert_rate,recall,precision,false_alert_share,alert_delay_lift
0,ORD,154482,0.049794,0.263215,0.671483,0.785770,0.308015,0.691985,1.170202
1,ATL,139815,0.045066,0.175353,0.526574,0.675287,0.224875,0.775125,1.282414
2,DEN,125866,0.04057,0.181217,0.64111,0.786488,0.222309,0.777691,1.226759
3,DFW,120114,0.038716,0.203365,0.707178,0.827691,0.238021,0.761979,1.170413
4,CLT,92740,0.029893,0.162691,0.586877,0.729388,0.202197,0.797803,1.242828
5,PHX,86126,0.027761,0.182279,0.597148,0.745270,0.227494,0.772506,1.248049
6,LAX,76744,0.024737,0.187037,0.58643,0.716664,0.228575,0.771425,1.22208
7,LAS,70720,0.022795,0.204468,0.735633,0.880152,0.244637,0.755363,1.196455
8,SEA,69633,0.022445,0.173869,0.555843,0.667961,0.208939,0.791061,1.201708
9,IAH,68841,0.022189,0.167313,0.582226,0.710193,0.204087,0.795913,1.219789



Número de aeropuertos de origen: 365
Número de aeropuertos de destino: 365
Cobertura por origen válida: True
Cobertura por destino válida: True


#### Interpretación

El análisis geográfico confirma una variabilidad relevante entre aeropuertos de origen y destino. Entre los principales aeropuertos por volumen, el recall y la tasa de alerta presentan diferencias considerables, por lo que el comportamiento del sistema no es homogéneo territorialmente.

En origen, algunos aeropuertos como DEN, DFW, LAS o MCO alcanzan recalls superiores al 90 %, pero acompañados de tasas de alerta también muy elevadas. En estos casos, los lifts se mantienen relativamente próximos a 1.00, lo que indica que parte de la elevada cobertura se obtiene mediante una clasificación poco selectiva.

En destino se observa un patrón similar, aunque con algunos contrastes más marcados. SLC, por ejemplo, presenta un lift de 1.38 con una tasa de alerta del 42.49 %, mientras que otros aeropuertos con recalls superiores al 90 % presentan lifts claramente menores.

En conjunto, la dimensión geográfica aporta evidencia de que la utilidad relativa de las alertas varía entre aeropuertos. Por ello, resulta necesario complementar las métricas globales con análisis desagregados por localización y volumen operativo.

### 5.4 Concentración operacional de falsas alertas y retrasos no detectados

Las métricas relativas permiten comparar el comportamiento del sistema entre grupos, pero no identifican necesariamente dónde se concentra el mayor volumen de errores. Desde una perspectiva operativa, esta distinción es relevante porque una categoría con una tasa de error moderada y un elevado número de vuelos puede acumular más errores que otra con una tasa superior pero escaso volumen.

El análisis se centrará en los falsos positivos y falsos negativos. Los primeros representan alertas que no se materializan posteriormente en un retraso, mientras que los segundos corresponden a retrasos reales que permanecen sin alerta.

Se analizará su concentración según franja horaria de salida, aerolínea operadora y aeropuerto de origen. Para cada dimensión se calculará tanto el número absoluto de errores como su participación sobre el total global de falsos positivos o falsos negativos.

El objetivo es identificar concentraciones operativamente relevantes y no establecer causalidad. Las diferencias observadas describen dónde se acumulan los errores del sistema definitivo, pero no permiten atribuirlos directamente a una aerolínea, aeropuerto o franja horaria concreta.

In [23]:
# ---------------------------------------------------------
# 1. Definir una función para analizar la concentración de errores
# ---------------------------------------------------------

def build_error_concentration(data, group_column):
    # ---------------------------------------------------------
    # 1.1. Contabilizar falsos positivos y falsos negativos
    # ---------------------------------------------------------

    error_summary = (
        data[
            data["confusion_category"].isin(
                ["FP", "FN"]
            )
        ]
        .groupby(
            [
                group_column,
                "confusion_category",
            ],
            observed=True,
        )
        .size()
        .unstack(
            fill_value=0
        )
        .reset_index()
    )

    for category in ["FP", "FN"]:
        if category not in error_summary.columns:
            error_summary[category] = 0

    # ---------------------------------------------------------
    # 1.2. Calcular la participación sobre cada tipo de error
    # ---------------------------------------------------------

    error_summary["fp_share"] = (
        error_summary["FP"] / fp
    )

    error_summary["fn_share"] = (
        error_summary["FN"] / fn
    )

    return error_summary


# ---------------------------------------------------------
# 2. Construir las concentraciones por dimensión operacional
# ---------------------------------------------------------

departure_error_concentration = build_error_concentration(
    external_alert_analysis,
    "DEP_TIME_BLK",
)

carrier_error_concentration = build_error_concentration(
    external_alert_analysis,
    "OP_UNIQUE_CARRIER",
)

origin_error_concentration = build_error_concentration(
    external_alert_analysis,
    "ORIGIN",
)


# ---------------------------------------------------------
# 3. Validar la reconstrucción de los errores globales
# ---------------------------------------------------------

error_concentration_validation = pd.DataFrame(
    {
        "dimension": [
            "Franja de salida",
            "Aerolínea operadora",
            "Aeropuerto de origen",
        ],
        "fp_valid": [
            int(departure_error_concentration["FP"].sum()) == fp,
            int(carrier_error_concentration["FP"].sum()) == fp,
            int(origin_error_concentration["FP"].sum()) == fp,
        ],
        "fn_valid": [
            int(departure_error_concentration["FN"].sum()) == fn,
            int(carrier_error_concentration["FN"].sum()) == fn,
            int(origin_error_concentration["FN"].sum()) == fn,
        ],
    }
)


# ---------------------------------------------------------
# 4. Seleccionar las mayores concentraciones de falsos positivos
# ---------------------------------------------------------

top_n_errors = 10

top_fp_departure = (
    departure_error_concentration
    .nlargest(
        top_n_errors,
        "FP",
    )[
        [
            "DEP_TIME_BLK",
            "FP",
            "fp_share",
        ]
    ]
    .reset_index(drop=True)
)

top_fp_carrier = (
    carrier_error_concentration
    .nlargest(
        top_n_errors,
        "FP",
    )[
        [
            "OP_UNIQUE_CARRIER",
            "FP",
            "fp_share",
        ]
    ]
    .reset_index(drop=True)
)

top_fp_origin = (
    origin_error_concentration
    .nlargest(
        top_n_errors,
        "FP",
    )[
        [
            "ORIGIN",
            "FP",
            "fp_share",
        ]
    ]
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 5. Seleccionar las mayores concentraciones de falsos negativos
# ---------------------------------------------------------

top_fn_departure = (
    departure_error_concentration
    .nlargest(
        top_n_errors,
        "FN",
    )[
        [
            "DEP_TIME_BLK",
            "FN",
            "fn_share",
        ]
    ]
    .reset_index(drop=True)
)

top_fn_carrier = (
    carrier_error_concentration
    .nlargest(
        top_n_errors,
        "FN",
    )[
        [
            "OP_UNIQUE_CARRIER",
            "FN",
            "fn_share",
        ]
    ]
    .reset_index(drop=True)
)

top_fn_origin = (
    origin_error_concentration
    .nlargest(
        top_n_errors,
        "FN",
    )[
        [
            "ORIGIN",
            "FN",
            "fn_share",
        ]
    ]
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 6. Mostrar las principales concentraciones de falsos positivos
# ---------------------------------------------------------

print("Mayores concentraciones de falsos positivos por franja:")
display(top_fp_departure)

print("\nMayores concentraciones de falsos positivos por aerolínea:")
display(top_fp_carrier)

print("\nMayores concentraciones de falsos positivos por origen:")
display(top_fp_origin)


# ---------------------------------------------------------
# 7. Mostrar las principales concentraciones de falsos negativos
# ---------------------------------------------------------

print("\nMayores concentraciones de falsos negativos por franja:")
display(top_fn_departure)

print("\nMayores concentraciones de falsos negativos por aerolínea:")
display(top_fn_carrier)

print("\nMayores concentraciones de falsos negativos por origen:")
display(top_fn_origin)


# ---------------------------------------------------------
# 8. Mostrar la validación consolidada
# ---------------------------------------------------------

display(error_concentration_validation)

Mayores concentraciones de falsos positivos por franja:


confusion_category,DEP_TIME_BLK,FP,fp_share
0,1700-1759,142973,0.092059
1,1600-1659,129557,0.083420
2,1400-1459,129316,0.083265
3,1800-1859,127919,0.082366
4,1500-1559,126194,0.081255
5,1900-1959,120830,0.077801
6,1300-1359,116538,0.075038
7,1200-1259,112440,0.072399
8,1100-1159,95478,0.061477
9,2000-2059,93158,0.059983



Mayores concentraciones de falsos positivos por aerolínea:


confusion_category,OP_UNIQUE_CARRIER,FP,fp_share
0,WN,279553,0.180001
1,AA,225200,0.145004
2,DL,181139,0.116633
3,UA,171462,0.110402
4,OO,152513,0.098201
5,AS,64434,0.041488
6,MQ,63205,0.040697
7,YX,62094,0.039982
8,B6,61629,0.039682
9,F9,57424,0.036975



Mayores concentraciones de falsos positivos por origen:


...


Mayores concentraciones de falsos negativos por aerolínea:


confusion_category,OP_UNIQUE_CARRIER,FN,fn_share
0,OO,22197,0.168492
1,DL,21986,0.166891
2,WN,13986,0.106164
3,UA,13762,0.104464
4,AA,10453,0.079346
5,YX,8933,0.067808
6,AS,8319,0.063148
7,9E,7369,0.055936
8,MQ,5793,0.043973
9,OH,3805,0.028883



Mayores concentraciones de falsos negativos por origen:


confusion_category,ORIGIN,FN,fn_share
0,ORD,6879,0.052217
1,ATL,6226,0.047260
2,LAX,4265,0.032375
3,DTW,4132,0.031365
4,PHX,3893,0.029551
5,MSP,3273,0.024845
6,SFO,3062,0.023243
7,LGA,2942,0.022332
8,CLT,2805,0.021292
9,SEA,2766,0.020996


,dimension,fp_valid,fn_valid
0,Franja de salida,True,True
1,Aerolínea operadora,True,True
2,Aeropuerto de origen,True,True


#### Interpretación

La distribución de los errores muestra una clara asimetría operacional. Los falsos positivos se concentran principalmente durante la tarde: la franja 17:00–17:59 acumula por sí sola el 9.21 % de todos los FP, coherentemente con las elevadas tasas de alerta observadas previamente en estas horas.

Los falsos negativos presentan el patrón contrario. Las franjas 06:00–10:59 concentran una parte sustancial de los retrasos no detectados, destacando 07:00–07:59 con el 17.88 % del total. Por tanto, durante las primeras horas el principal problema es la pérdida de cobertura, mientras que durante la tarde predomina el exceso de alertas.

La concentración también varía entre operadores y aeropuertos, aunque parte de estos resultados está condicionada por su volumen de operaciones. Por ello, estas cifras deben interpretarse como localización del volumen absoluto de errores y no como evidencia de peor comportamiento causal de una aerolínea o aeropuerto concreto.

### 5.5 Síntesis y persistencia de los indicadores operacionales y geográficos

Los análisis anteriores han caracterizado el comportamiento del sistema desde tres dimensiones complementarias: las franjas horarias programadas, las aerolíneas operadoras y la distribución geográfica por aeropuertos. Asimismo, se ha identificado dónde se concentran los falsos positivos y los retrasos que permanecen sin alerta.

Para garantizar la trazabilidad y evitar la repetición de cálculos sobre los 3,102,447 vuelos del período externo, en este subbloque se consolidarán y persistirán los productos analíticos generados durante el bloque.

Se conservarán las tablas completas, incluyendo todas las categorías disponibles y no únicamente aquellas mostradas en las tablas resumidas. Estos productos constituirán la base de entrada del posterior proceso de preparación para visualización.

Finalmente, se realizará una validación consolidada de los archivos persistidos, verificando su existencia, número de registros y consistencia con los objetos analíticos originales.

In [25]:
# ---------------------------------------------------------
# 1. Recuperar las tablas consolidadas disponibles
# ---------------------------------------------------------

# Se reutilizan los resultados ya generados durante el bloque 5.
# En caso de diferencias de nomenclatura, se localiza el objeto
# correspondiente sin repetir los análisis realizados.

candidate_time_objects = [
    "time_block_summary",
    "departure_time_summary",
    "departure_time_block_summary",
]

candidate_carrier_objects = [
    "carrier_summary",
    "operator_summary",
    "carrier_operational_summary",
]

time_summary_name = next(
    (
        name
        for name in candidate_time_objects
        if name in globals()
    ),
    None,
)

carrier_summary_name = next(
    (
        name
        for name in candidate_carrier_objects
        if name in globals()
    ),
    None,
)

if time_summary_name is None:
    raise NameError(
        "No se encontró en memoria la tabla consolidada "
        "generada en el subbloque 5.1."
    )

if carrier_summary_name is None:
    raise NameError(
        "No se encontró en memoria la tabla consolidada "
        "generada en el subbloque 5.2."
    )

time_block_summary = globals()[time_summary_name]
carrier_summary = globals()[carrier_summary_name]


# ---------------------------------------------------------
# 2. Validar la disponibilidad de los productos del bloque 5
# ---------------------------------------------------------

required_operational_objects = {
    "time_block_summary": time_block_summary,
    "carrier_summary": carrier_summary,
    "origin_summary": origin_summary,
    "destination_summary": destination_summary,
    "departure_error_concentration": departure_error_concentration,
    "carrier_error_concentration": carrier_error_concentration,
    "origin_error_concentration": origin_error_concentration,
}

missing_operational_objects = [
    object_name
    for object_name, object_data in required_operational_objects.items()
    if object_data is None
]

if missing_operational_objects:
    raise RuntimeError(
        "Faltan productos necesarios del bloque 5: "
        + ", ".join(missing_operational_objects)
    )


# ---------------------------------------------------------
# 3. Definir los productos operacionales que se persistirán
# ---------------------------------------------------------

operational_products = {
    "time_block_summary_2026.parquet": time_block_summary,
    "carrier_summary_2026.parquet": carrier_summary,
    "origin_airport_summary_2026.parquet": origin_summary,
    "destination_airport_summary_2026.parquet": destination_summary,
    "departure_error_concentration_2026.parquet": departure_error_concentration,
    "carrier_error_concentration_2026.parquet": carrier_error_concentration,
    "origin_error_concentration_2026.parquet": origin_error_concentration,
}


# ---------------------------------------------------------
# 4. Persistir los productos analíticos
# ---------------------------------------------------------

persisted_operational_products = {}

for file_name, product_data in operational_products.items():
    product_path = operational_results_path / file_name

    product_data.to_parquet(
        product_path,
        index=False,
    )

    persisted_operational_products[file_name] = product_path


# ---------------------------------------------------------
# 5. Recuperar y validar los productos persistidos
# ---------------------------------------------------------

validation_records = []

for file_name, original_data in operational_products.items():
    product_path = persisted_operational_products[file_name]

    recovered_data = pd.read_parquet(
        product_path
    )

    validation_records.append(
        {
            "file": file_name,
            "file_exists": product_path.exists(),
            "expected_rows": len(original_data),
            "persisted_rows": len(recovered_data),
            "expected_columns": original_data.shape[1],
            "persisted_columns": recovered_data.shape[1],
            "rows_valid": (
                len(recovered_data)
                == len(original_data)
            ),
            "columns_valid": (
                recovered_data.shape[1]
                == original_data.shape[1]
            ),
        }
    )

operational_products_validation = pd.DataFrame(
    validation_records
)


# ---------------------------------------------------------
# 6. Construir y persistir la síntesis metodológica del bloque
# ---------------------------------------------------------

operational_block_summary = pd.DataFrame(
    {
        "dimension": [
            "Franja horaria",
            "Aerolínea operadora",
            "Aeropuerto de origen",
            "Aeropuerto de destino",
        ],
        "categories": [
            len(time_block_summary),
            len(carrier_summary),
            len(origin_summary),
            len(destination_summary),
        ],
        "main_indicator": [
            "Recall",
            "Recall",
            "Recall",
            "Recall",
        ],
        "complementary_indicator": [
            "Alert rate / lift",
            "Alert rate / lift",
            "Alert rate / lift",
            "Alert rate / lift",
        ],
    }
)

operational_block_summary_path = (
    operational_results_path
    / "operational_block_summary_2026.parquet"
)

operational_block_summary.to_parquet(
    operational_block_summary_path,
    index=False,
)


# ---------------------------------------------------------
# 7. Realizar la validación consolidada
# ---------------------------------------------------------

all_files_exist = bool(
    operational_products_validation[
        "file_exists"
    ].all()
)

all_rows_valid = bool(
    operational_products_validation[
        "rows_valid"
    ].all()
)

all_columns_valid = bool(
    operational_products_validation[
        "columns_valid"
    ].all()
)

block_summary_valid = (
    operational_block_summary_path.exists()
    and len(operational_block_summary) == 4
)


# ---------------------------------------------------------
# 8. Mostrar la síntesis y las validaciones
# ---------------------------------------------------------

print(
    "Objeto recuperado para franjas horarias:",
    time_summary_name,
)

print(
    "Objeto recuperado para aerolíneas:",
    carrier_summary_name,
)

print("\nSíntesis de las dimensiones analizadas:")
display(operational_block_summary)

print("\nValidación de los productos persistidos:")
display(operational_products_validation)

print(
    "\nTodos los archivos existen:",
    all_files_exist,
)

print(
    "Número de registros válido:",
    all_rows_valid,
)

print(
    "Número de columnas válido:",
    all_columns_valid,
)

print(
    "Síntesis del bloque persistida correctamente:",
    block_summary_valid,
)

print(
    "\nDirectorio de persistencia:",
    operational_results_path,
)

Objeto recuperado para franjas horarias: departure_time_summary
Objeto recuperado para aerolíneas: carrier_summary

Síntesis de las dimensiones analizadas:


,dimension,categories,main_indicator,complementary_indicator
0,Franja horaria,19,Recall,Alert rate / lift
1,Aerolínea operadora,19,Recall,Alert rate / lift
2,Aeropuerto de origen,365,Recall,Alert rate / lift
3,Aeropuerto de destino,365,Recall,Alert rate / lift



Validación de los productos persistidos:


,file,file_exists,expected_rows,persisted_rows,expected_columns,persisted_columns,rows_valid,columns_valid
0,time_block_summary_2026.parquet,True,19,19,14,14,True,True
1,carrier_summary_2026.parquet,True,19,19,15,15,True,True
2,origin_airport_summary_2026.parquet,True,365,365,15,15,True,True
3,destination_airport_summary_2026.parquet,True,365,365,15,15,True,True
4,departure_error_concentration_2026.parquet,True,19,19,5,5,True,True
5,carrier_error_concentration_2026.parquet,True,19,19,5,5,True,True
6,origin_error_concentration_2026.parquet,True,365,365,5,5,True,True



Todos los archivos existen: True
Número de registros válido: True
Número de columnas válido: True
Síntesis del bloque persistida correctamente: True

Directorio de persistencia: G:\My Drive\MASTER Big Data\TFM\results\operational_analysis


#### Interpretación

La persistencia de los productos operacionales y geográficos se completó correctamente. Se conservaron las agregaciones correspondientes a 19 franjas horarias, 19 aerolíneas operadoras y 365 aeropuertos tanto de origen como de destino, junto con las tablas completas de concentración de falsos positivos y falsos negativos.

Las validaciones confirman que todos los archivos fueron generados, mantienen el mismo número de registros y columnas que los objetos originales y pueden recuperarse de forma consistente desde el directorio de resultados operacionales.

Estos productos permiten cerrar el análisis operacional sin necesidad de repetir cálculos sobre los 3,102,447 vuelos del período externo y constituyen la base reproducible para la preparación de los resultados destinados a visualización.

## 6. Preparación de productos para visualización

Los resultados obtenidos durante el análisis global, temporal, operacional y geográfico contienen diferentes niveles de granularidad y responden a preguntas complementarias sobre el comportamiento del sistema de alerta temprana. Para trasladar estos resultados a una herramienta de visualización resulta necesario construir productos analíticos específicamente preparados para su consumo, evitando que la capa de presentación tenga que reproducir transformaciones o cálculos metodológicos realizados en el notebook.

Esta etapa no pretende ampliar la evaluación del modelo ni modificar ninguna de las decisiones adoptadas previamente. Su función consiste en organizar los indicadores ya obtenidos en estructuras compactas, interpretables y reutilizables que permitan representar tanto el comportamiento general del sistema como sus principales diferencias temporales y operacionales.

La preparación mantendrá separados los distintos niveles de análisis cuando su granularidad sea diferente. De este modo, los indicadores globales, mensuales, horarios, por aerolínea y por aeropuerto conservarán una unidad de observación claramente definida. Asimismo, se incorporarán los indicadores necesarios para contextualizar conjuntamente cobertura, precisión, intensidad de alertas y concentración de errores, evitando visualizaciones basadas en una única métrica aislada.

Los productos resultantes estarán orientados principalmente a su utilización posterior en Tableau y a la selección de evidencias para la presentación del TFM. Por este motivo, se priorizarán estructuras simples, nombres comprensibles y variables directamente interpretables, manteniendo siempre la trazabilidad con los resultados persistidos durante los bloques anteriores.

En este bloque se abordarán progresivamente:

1. la definición de la estructura y granularidad de los productos destinados a visualización;
2. la preparación del producto de indicadores globales y temporales;
3. la preparación del producto de comportamiento operacional por franja horaria y aerolínea;
4. la preparación del producto geográfico por aeropuertos;
5. la preparación del producto de concentración de errores;
6. la persistencia y validación consolidada de los productos destinados a visualización.

### 6.1 Definición de la estructura y granularidad de los productos de visualización

La construcción de productos destinados a visualización requiere definir previamente la unidad de observación de cada tabla. Esta decisión evita combinar indicadores pertenecientes a niveles de agregación diferentes y reduce el riesgo de obtener cálculos incorrectos cuando los datos sean posteriormente utilizados en Tableau.

Se mantendrán cinco niveles analíticos diferenciados: global y temporal, franja horaria, aerolínea operadora, aeropuerto y concentración de errores. Cada producto conservará una granularidad única y estará vinculado directamente con los resultados persistidos en los bloques anteriores.

La estructura propuesta responde al principio de que cada fila debe representar una entidad analítica inequívoca. Por ejemplo, una fila del producto temporal representará un mes, mientras que una fila del producto geográfico representará un aeropuerto dentro de una función operacional concreta, como origen o destino.

En este subbloque se comprobará la disponibilidad de los productos necesarios y se construirá un inventario de las tablas que serán preparadas posteriormente. No se modificarán todavía sus indicadores ni se generarán archivos definitivos de visualización.

In [26]:
# ---------------------------------------------------------
# 1. Definir las rutas de los productos analíticos disponibles
# ---------------------------------------------------------

analysis_product_paths = {
    "global": (
        operational_results_path
        / "global_system_summary_2026.parquet"
    ),
    "temporal": (
        operational_results_path
        / "temporal_system_summary_2026.parquet"
    ),
    "time_block": (
        operational_results_path
        / "time_block_summary_2026.parquet"
    ),
    "carrier": (
        operational_results_path
        / "carrier_summary_2026.parquet"
    ),
    "origin_airport": (
        operational_results_path
        / "origin_airport_summary_2026.parquet"
    ),
    "destination_airport": (
        operational_results_path
        / "destination_airport_summary_2026.parquet"
    ),
    "departure_errors": (
        operational_results_path
        / "departure_error_concentration_2026.parquet"
    ),
    "carrier_errors": (
        operational_results_path
        / "carrier_error_concentration_2026.parquet"
    ),
    "origin_errors": (
        operational_results_path
        / "origin_error_concentration_2026.parquet"
    ),
}


# ---------------------------------------------------------
# 2. Comprobar la disponibilidad y dimensiones de los productos
# ---------------------------------------------------------

analysis_inventory_records = []

for product_name, product_path in analysis_product_paths.items():
    file_exists = product_path.exists()

    if file_exists:
        product_data = pd.read_parquet(
            product_path
        )

        rows = len(product_data)
        columns = product_data.shape[1]
    else:
        rows = None
        columns = None

    analysis_inventory_records.append(
        {
            "product": product_name,
            "file_exists": file_exists,
            "rows": rows,
            "columns": columns,
        }
    )

analysis_products_inventory = pd.DataFrame(
    analysis_inventory_records
)


# ---------------------------------------------------------
# 3. Definir los productos finales previstos para visualización
# ---------------------------------------------------------

visualization_products_structure = pd.DataFrame(
    {
        "visualization_product": [
            "global_temporal",
            "time_block",
            "carrier",
            "airport",
            "error_concentration",
        ],
        "unit_of_observation": [
            "Indicador global o mes",
            "Franja horaria de salida",
            "Aerolínea operadora",
            "Aeropuerto y función operacional",
            "Categoría dentro de una dimensión operacional",
        ],
        "source_products": [
            "global + temporal",
            "time_block",
            "carrier",
            "origin_airport + destination_airport",
            "departure_errors + carrier_errors + origin_errors",
        ],
        "preparation_subblock": [
            "6.2",
            "6.3",
            "6.3",
            "6.4",
            "6.5",
        ],
    }
)


# ---------------------------------------------------------
# 4. Validar la disponibilidad de todas las fuentes
# ---------------------------------------------------------

all_analysis_products_available = bool(
    analysis_products_inventory[
        "file_exists"
    ].all()
)

expected_visualization_products = 5

visualization_structure_valid = (
    len(visualization_products_structure)
    == expected_visualization_products
    and visualization_products_structure[
        "unit_of_observation"
    ].notna().all()
)


# ---------------------------------------------------------
# 5. Mostrar el inventario y la estructura prevista
# ---------------------------------------------------------

print("Inventario de productos analíticos disponibles:")
display(analysis_products_inventory)

print("\nEstructura prevista para visualización:")
display(visualization_products_structure)

print(
    "\nTodos los productos analíticos están disponibles:",
    all_analysis_products_available,
)

print(
    "Estructura de productos de visualización válida:",
    visualization_structure_valid,
)

Inventario de productos analíticos disponibles:


,product,file_exists,rows,columns
0,global,True,12,3
1,temporal,True,5,21
2,time_block,True,19,14
3,carrier,True,19,15
4,origin_airport,True,365,15
5,destination_airport,True,365,15
6,departure_errors,True,19,5
7,carrier_errors,True,19,5
8,origin_errors,True,365,5



Estructura prevista para visualización:


,visualization_product,unit_of_observation,source_products,preparation_subblock
0,global_temporal,Indicador global o mes,global + temporal,6.2
1,time_block,Franja horaria de salida,time_block,6.3
2,carrier,Aerolínea operadora,carrier,6.3
3,airport,Aeropuerto y función operacional,origin_airport + destination_airport,6.4
4,error_concentration,Categoría dentro de una dimensión operacional,departure_errors + carrier_errors + origin_errors,6.5



Todos los productos analíticos están disponibles: True
Estructura de productos de visualización válida: True


#### Interpretación

La estructura de productos para visualización quedó definida y validada correctamente. Las cinco salidas previstas mantienen granularidades diferenciadas: global-temporal, franja horaria, aerolínea, aeropuerto y concentración de errores.

La disponibilidad de todas las fuentes confirma que los siguientes subbloques podrán construirse exclusivamente a partir de resultados ya persistidos, sin repetir cálculos sobre la población externa.

Esta separación reduce el riesgo de agregaciones incorrectas en Tableau y preserva la trazabilidad entre cada visualización y su nivel analítico de origen.

### 6.2 Preparación de indicadores globales y temporales

La caracterización general del sistema combina dos perspectivas complementarias. La primera resume su comportamiento sobre el conjunto completo del período externo, mientras que la segunda permite observar cómo varían los principales indicadores entre enero y mayo de 2026. Aunque ambas perspectivas describen el mismo sistema, sus unidades de observación son diferentes y deben conservarse separadas.

El producto global proporcionará los indicadores necesarios para representar de forma sintética la magnitud del sistema, su cobertura y su capacidad de discriminación operacional. El producto temporal conservará una observación por mes y permitirá representar conjuntamente prevalencia de retrasos, intensidad de alertas, recall, precisión, falsas alertas y comportamiento del score.

La preparación se limitará a seleccionar, ordenar y normalizar los campos ya calculados. No se recalcularán métricas ni se utilizará la información del período externo para modificar el modelo o su regla de decisión.

Esta estructura permitirá utilizar el resumen global para indicadores de síntesis y el producto mensual para gráficos de evolución, evitando mezclar en una misma tabla observaciones correspondientes a niveles temporales diferentes.

In [30]:
# ---------------------------------------------------------
# 1. Recuperar los productos global y temporal persistidos
# ---------------------------------------------------------

global_source = pd.read_parquet(
    analysis_product_paths["global"]
)

temporal_source = pd.read_parquet(
    analysis_product_paths["temporal"]
)


# ---------------------------------------------------------
# 2. Preparar el producto global para visualización
# ---------------------------------------------------------

global_visualization = (
    global_source
    .copy()
)

global_visualization.insert(
    0,
    "analysis_period",
    "2026-01 a 2026-05",
)

global_visualization.insert(
    0,
    "analysis_level",
    "Global",
)


# ---------------------------------------------------------
# 3. Definir los indicadores temporales relevantes
# ---------------------------------------------------------

temporal_visualization_columns = [
    "analysis_month",
    "flights",
    "observed_delays",
    "alerts",
    "observed_delay_rate",
    "alert_rate",
    "recall",
    "precision",
    "false_alert_share",
    "missed_delay_share",
    "delay_share_without_alert",
    "alert_delay_lift",
    "mean_score",
    "median_score",
    "std_score",
    "p05_score",
    "p25_score",
    "p75_score",
    "p95_score",
    "mean_threshold_distance",
    "share_within_005",
]


# ---------------------------------------------------------
# 4. Seleccionar las columnas temporales disponibles
# ---------------------------------------------------------

temporal_visualization_columns = [
    column
    for column in temporal_visualization_columns
    if column in temporal_source.columns
]

temporal_visualization = (
    temporal_source[
        temporal_visualization_columns
    ]
    .copy()
)

temporal_visualization.insert(
    0,
    "analysis_level",
    "Mensual",
)


# ---------------------------------------------------------
# 5. Ordenar temporalmente las observaciones mensuales
# ---------------------------------------------------------

temporal_visualization = (
    temporal_visualization
    .sort_values(
        "analysis_month"
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 6. Validar el producto global
# ---------------------------------------------------------

expected_global_columns = {
    "analysis_level",
    "analysis_period",
    "dimension",
    "indicator",
    "value",
}

global_visualization_valid = (
    len(global_visualization) == len(global_source)
    and len(global_visualization) == 12
    and expected_global_columns.issubset(
        global_visualization.columns
    )
    and global_visualization[
        [
            "dimension",
            "indicator",
            "value",
        ]
    ].isna().sum().sum() == 0
)


# ---------------------------------------------------------
# 7. Validar el producto temporal
# ---------------------------------------------------------

temporal_visualization_valid = (
    len(temporal_visualization) == 5
    and "analysis_month" in temporal_visualization.columns
    and temporal_visualization.isna().sum().sum() == 0
)

temporal_volume_valid = (
    "flights" in temporal_visualization.columns
    and int(
        temporal_visualization["flights"].sum()
    )
    == total_flights
)

temporal_months_valid = (
    temporal_visualization[
        "analysis_month"
    ].nunique()
    == 5
)


# ---------------------------------------------------------
# 8. Mostrar los productos preparados
# ---------------------------------------------------------

print(
    "Producto global preparado para visualización:"
)

display(
    global_visualization
)

print(
    "\nProducto temporal preparado para visualización:"
)

display(
    temporal_visualization
)

print(
    "\nProducto global válido:",
    global_visualization_valid,
)

print(
    "Producto temporal válido:",
    temporal_visualization_valid,
)

print(
    "Volumen temporal consistente con la población externa:",
    temporal_volume_valid,
)

print(
    "Cobertura mensual válida:",
    temporal_months_valid,
)

print(
    "\nNúmero de indicadores globales:",
    len(global_visualization),
)

print(
    "Número de observaciones temporales:",
    len(temporal_visualization),
)

print(
    "\nColumnas globales:",
    list(
        global_visualization.columns
    ),
)

print(
    "\nColumnas temporales:",
    list(
        temporal_visualization.columns
    ),
)

Producto global preparado para visualización:


,analysis_level,analysis_period,dimension,indicator,value
0,Global,2026-01 a 2026-05,Población,Vuelos analizados,3.102447e+06
1,Global,2026-01 a 2026-05,Prevalencia,Tasa observada de retraso,2.117180e-01
2,Global,2026-01 a 2026-05,Intensidad de alerta,Tasa de alerta,6.698480e-01
3,Global,2026-01 a 2026-05,Cobertura,Cobertura de retrasos,7.994364e-01
4,Global,2026-01 a 2026-05,Selectividad,Precisión de las alertas,2.526769e-01
5,Global,2026-01 a 2026-05,Selectividad,Proporción de falsas alertas,7.473231e-01
6,Global,2026-01 a 2026-05,Riesgo residual,Retrasos entre vuelos sin alerta,1.286163e-01
7,Global,2026-01 a 2026-05,Concentración del riesgo,Lift de retraso entre alertados,1.193459e+00
8,Global,2026-01 a 2026-05,Concentración del riesgo,Riesgo relativo entre no alertados,6.074887e-01
9,Global,2026-01 a 2026-05,Separación del score,Diferencia media TP-FP,2.809972e-02



Producto temporal preparado para visualización:


,analysis_level,analysis_month,flights,observed_delays,alerts,observed_delay_rate,alert_rate,recall,precision,false_alert_share,...,alert_delay_lift,mean_score,median_score,std_score,p05_score,p25_score,p75_score,p95_score,mean_threshold_distance,share_within_005
0,Mensual,2026-01,572599,118672,399200,0.207251,0.697172,0.784321,0.233159,0.766841,...,1.125004,0.481893,0.489112,0.129472,0.264530,0.382294,0.581791,0.683746,0.075305,0.222272
1,Mensual,2026-02,554297,107998,331460,0.194838,0.597983,0.727384,0.237000,0.763000,...,1.216396,0.441056,0.446066,0.127645,0.231412,0.341024,0.539452,0.643201,0.034468,0.242397
2,Mensual,2026-03,653297,157756,439607,0.241477,0.672905,0.803469,0.288330,0.711670,...,1.194029,0.472874,0.480681,0.129456,0.257851,0.371219,0.573852,0.673798,0.066286,0.222844
3,Mensual,2026-04,653335,130203,432425,0.19929,0.661873,0.809697,0.243800,0.756200,...,1.223342,0.467614,0.473932,0.127722,0.256217,0.367591,0.567214,0.667414,0.061026,0.231122
4,Mensual,2026-05,668919,142215,475476,0.212604,0.710813,0.852899,0.255102,0.744898,...,1.199893,0.487276,0.494814,0.126843,0.274900,0.388887,0.586713,0.683334,0.080688,0.225045



Producto global válido: True
Producto temporal válido: True
Volumen temporal consistente con la población externa: True
Cobertura mensual válida: True

Número de indicadores globales: 12
Número de observaciones temporales: 5

Columnas globales: ['analysis_level', 'analysis_period', 'dimension', 'indicator', 'value']

Columnas temporales: ['analysis_level', 'analysis_month', 'flights', 'observed_delays', 'alerts', 'observed_delay_rate', 'alert_rate', 'recall', 'precision', 'false_alert_share', 'missed_delay_share', 'delay_share_without_alert', 'alert_delay_lift', 'mean_score', 'median_score', 'std_score', 'p05_score', 'p25_score', 'p75_score', 'p95_score', 'mean_threshold_distance', 'share_within_005']


#### Interpretación

Los productos global y temporal quedaron preparados y validados correctamente para su utilización posterior en visualización.

El producto global conserva 12 indicadores en formato largo, lo que facilita su uso como conjunto de KPI organizados por dimensión, indicador y valor. El producto temporal mantiene una observación por mes entre enero y mayo de 2026 e incorpora los principales indicadores de volumen, cobertura, selectividad y distribución del score.

Las validaciones confirman que las cinco observaciones mensuales reconstruyen correctamente los 3,102,447 vuelos del período externo y que ambas estructuras mantienen una granularidad inequívoca. De este modo, los resultados globales y temporales quedan listos para su posterior persistencia y consumo en Tableau.

### 6.3 Preparación de indicadores operacionales

El análisis operacional realizado previamente mostró que el comportamiento del sistema varía tanto según la franja horaria programada como según la aerolínea operadora. Para trasladar estos resultados a la capa de visualización se prepararán dos productos independientes, preservando la granularidad específica de cada dimensión.

El producto horario mantendrá una observación por franja de salida y permitirá representar conjuntamente volumen, prevalencia de retrasos, intensidad de alertas, recall, precisión, falsas alertas y lift. Además, se incorporará un orden numérico de las franjas para garantizar su representación cronológica en Tableau.

El producto por aerolínea conservará una observación por operador e incluirá los mismos indicadores fundamentales junto con su volumen de operaciones. Esta combinación permitirá evitar comparaciones basadas exclusivamente en tasas, ya que las diferencias entre operadores deberán interpretarse considerando también su peso dentro de la población externa.

La preparación reutilizará exclusivamente los productos persistidos en el bloque anterior. No se recalcularán métricas ni se establecerán nuevos criterios de selección del modelo.

In [31]:
# ---------------------------------------------------------
# 1. Recuperar los productos operacionales persistidos
# ---------------------------------------------------------

time_block_source = pd.read_parquet(
    analysis_product_paths["time_block"]
)

carrier_source = pd.read_parquet(
    analysis_product_paths["carrier"]
)


# ---------------------------------------------------------
# 2. Definir los indicadores operacionales relevantes
# ---------------------------------------------------------

operational_preferred_columns = [
    "flights",
    "population_share",
    "observed_delays",
    "alerts",
    "observed_delay_rate",
    "alert_rate",
    "recall",
    "precision",
    "false_alert_share",
    "missed_delay_share",
    "delay_share_without_alert",
    "alert_delay_lift",
    "TP",
    "FP",
    "TN",
    "FN",
]


# ---------------------------------------------------------
# 3. Preparar el producto por franja horaria
# ---------------------------------------------------------

time_block_metric_columns = [
    column
    for column in operational_preferred_columns
    if column in time_block_source.columns
]

time_block_visualization = (
    time_block_source[
        ["DEP_TIME_BLK"] + time_block_metric_columns
    ]
    .copy()
)

time_block_visualization.insert(
    0,
    "analysis_dimension",
    "Franja horaria",
)

time_block_visualization["time_block_order"] = (
    time_block_visualization[
        "DEP_TIME_BLK"
    ]
    .str.split("-")
    .str[0]
    .astype(int)
)

time_block_visualization = (
    time_block_visualization
    .sort_values(
        "time_block_order"
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 4. Preparar el producto por aerolínea operadora
# ---------------------------------------------------------

carrier_metric_columns = [
    column
    for column in operational_preferred_columns
    if column in carrier_source.columns
]

carrier_visualization = (
    carrier_source[
        ["OP_UNIQUE_CARRIER"] + carrier_metric_columns
    ]
    .copy()
)

carrier_visualization.insert(
    0,
    "analysis_dimension",
    "Aerolínea operadora",
)

carrier_visualization = (
    carrier_visualization
    .sort_values(
        "flights",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 5. Validar el producto por franja horaria
# ---------------------------------------------------------

time_block_visualization_valid = (
    len(time_block_visualization) == 19
    and time_block_visualization[
        "DEP_TIME_BLK"
    ].nunique() == 19
    and time_block_visualization.isna().sum().sum() == 0
)

time_block_order_valid = (
    time_block_visualization[
        "time_block_order"
    ].is_monotonic_increasing
)

time_block_volume_valid = (
    int(
        time_block_visualization[
            "flights"
        ].sum()
    )
    == total_flights
)


# ---------------------------------------------------------
# 6. Validar el producto por aerolínea
# ---------------------------------------------------------

carrier_visualization_valid = (
    len(carrier_visualization) == 19
    and carrier_visualization[
        "OP_UNIQUE_CARRIER"
    ].nunique() == 19
    and carrier_visualization.isna().sum().sum() == 0
)

carrier_volume_valid = (
    int(
        carrier_visualization[
            "flights"
        ].sum()
    )
    == total_flights
)


# ---------------------------------------------------------
# 7. Mostrar los productos preparados
# ---------------------------------------------------------

print(
    "Producto por franja horaria preparado para visualización:"
)

display(
    time_block_visualization
)

print(
    "\nProducto por aerolínea preparado para visualización:"
)

display(
    carrier_visualization
)

print(
    "\nProducto por franja horaria válido:",
    time_block_visualization_valid,
)

print(
    "Orden cronológico de franjas válido:",
    time_block_order_valid,
)

print(
    "Volumen por franjas consistente:",
    time_block_volume_valid,
)

print(
    "Producto por aerolínea válido:",
    carrier_visualization_valid,
)

print(
    "Volumen por aerolíneas consistente:",
    carrier_volume_valid,
)

print(
    "\nColumnas del producto horario:",
    list(
        time_block_visualization.columns
    ),
)

print(
    "\nColumnas del producto por aerolínea:",
    list(
        carrier_visualization.columns
    ),
)

Producto por franja horaria preparado para visualización:


,analysis_dimension,DEP_TIME_BLK,flights,observed_delays,alerts,observed_delay_rate,alert_rate,recall,precision,false_alert_share,alert_delay_lift,TP,FP,TN,FN,time_block_order
0,Franja horaria,0001-0559,87493,8083,3760,0.092385,0.042975,0.074230,0.159574,0.840426,1.727286,600,3160,76250,7483,1
1,Franja horaria,0600-0659,209879,21924,10870,0.10446,0.051792,0.065636,0.132383,0.867617,1.267303,1439,9431,178524,20485,600
2,Franja horaria,0700-0759,226450,28847,34039,0.127388,0.150316,0.183381,0.155410,0.844590,1.219974,5290,28749,168854,23557,700
3,Franja horaria,0800-0859,211494,31262,60372,0.147815,0.285455,0.340477,0.176307,0.823693,1.192753,10644,49728,130504,20618,800
4,Franja horaria,0900-0959,178254,28839,69920,0.161786,0.392249,0.471584,0.194508,0.805492,1.202255,13600,56320,93095,15239,900
5,Franja horaria,1000-1059,198488,35306,99158,0.177875,0.499567,0.577749,0.205712,0.794288,1.1565,20398,78760,84422,14908,1000
6,Franja horaria,1100-1159,190075,36589,121630,0.192498,0.639905,0.714750,0.215013,0.784987,1.116963,26152,95478,58008,10437,1100
7,Franja horaria,1200-1259,195766,41090,145314,0.209893,0.742284,0.800049,0.226227,0.773773,1.07782,32874,112440,42236,8216,1200
8,Franja horaria,1300-1359,182109,40743,152647,0.223729,0.838218,0.886263,0.236552,0.763448,1.057318,36109,116538,24828,4634,1300
9,Franja horaria,1400-1459,184049,44067,171290,0.239431,0.930676,0.952504,0.245046,0.754954,1.023454,41974,129316,10666,2093,1400



Producto por aerolínea preparado para visualización:


,analysis_dimension,OP_UNIQUE_CARRIER,flights,population_share,observed_delays,alerts,observed_delay_rate,alert_rate,recall,precision,false_alert_share,alert_delay_lift,TP,FP,TN,FN
0,Aerolínea operadora,WN,557982,0.179852,123774,389341,0.221824,0.697766,0.887004,0.281984,0.718016,1.271205,109788,279553,154655,13986
1,Aerolínea operadora,DL,408780,0.131761,72967,232120,0.178499,0.567836,0.698686,0.219632,0.780368,1.230436,50981,181139,154674,21986
2,Aerolínea operadora,AA,393936,0.126976,91655,306402,0.232665,0.777796,0.885953,0.265018,0.734982,1.139055,81202,225200,77081,10453
3,Aerolínea operadora,OO,342154,0.110285,73088,203404,0.213611,0.594481,0.696298,0.250197,0.749803,1.17127,50891,152513,116553,22197
4,Aerolínea operadora,UA,332937,0.107314,61679,219379,0.185257,0.65892,0.776877,0.218421,0.781579,1.179015,47917,171462,99796,13762
5,Aerolínea operadora,YX,144862,0.046693,29337,82498,0.202517,0.569494,0.695504,0.247327,0.752673,1.221267,20404,62094,53431,8933
6,Aerolínea operadora,AS,132614,0.042745,24574,80689,0.185305,0.60845,0.661471,0.201452,0.798548,1.087142,16255,64434,43606,8319
7,Aerolínea operadora,MQ,125950,0.040597,23228,80640,0.184422,0.640254,0.750603,0.216208,0.783792,1.172351,17435,63205,39517,5793
8,Aerolínea operadora,9E,99998,0.032232,22691,56772,0.226915,0.567731,0.675246,0.269887,0.730113,1.189375,15322,41450,35857,7369
9,Aerolínea operadora,B6,94444,0.030442,25537,85945,0.270393,0.91001,0.952187,0.282925,0.717075,1.046348,24316,61629,7278,1221



Producto por franja horaria válido: True
Orden cronológico de franjas válido: True
Volumen por franjas consistente: True
Producto por aerolínea válido: True
Volumen por aerolíneas consistente: True

Columnas del producto horario: ['analysis_dimension', 'DEP_TIME_BLK', 'flights', 'observed_delays', 'alerts', 'observed_delay_rate', 'alert_rate', 'recall', 'precision', 'false_alert_share', 'alert_delay_lift', 'TP', 'FP', 'TN', 'FN', 'time_block_order']

Columnas del producto por aerolínea: ['analysis_dimension', 'OP_UNIQUE_CARRIER', 'flights', 'population_share', 'observed_delays', 'alerts', 'observed_delay_rate', 'alert_rate', 'recall', 'precision', 'false_alert_share', 'alert_delay_lift', 'TP', 'FP', 'TN', 'FN']


#### Interpretación

Los productos operacionales por franja horaria y aerolínea quedaron preparados y validados correctamente. Ambos reproducen los 3,102,447 vuelos del período externo y conservan conjuntamente indicadores de volumen, prevalencia, intensidad de alertas, recall, precisión, lift y matriz de confusión.

El producto horario incorpora además un campo de orden que permitirá representar cronológicamente las 19 franjas en Tableau, mientras que el producto por aerolínea mantiene las 19 operadoras ordenadas por volumen de operaciones.

De este modo, ambas estructuras quedan preparadas para comparar el comportamiento operacional del sistema sin separar las métricas de rendimiento del volumen sobre el que fueron calculadas.

### 6.4 Preparación de indicadores geográficos

La representación geográfica del comportamiento del sistema requiere integrar las dos perspectivas analizadas previamente: el aeropuerto desde el que se inicia el vuelo y el aeropuerto al que se dirige. Ambas dimensiones comparten la misma unidad geográfica, pero representan funciones operacionales diferentes y no deben interpretarse como equivalentes.

Para facilitar su utilización en Tableau, los resultados de origen y destino se integrarán en un único producto mediante una estructura común. Cada fila representará la combinación entre un aeropuerto y su función operacional, identificada explícitamente mediante una variable que distinguirá entre origen y destino.

El producto conservará el volumen de vuelos y los principales indicadores de comportamiento del sistema, incluyendo prevalencia observada, tasa de alerta, recall, precisión, proporción de falsas alertas, lift y componentes de la matriz de confusión. Esta combinación permitirá analizar simultáneamente la relevancia operacional de cada aeropuerto y el comportamiento de las alertas asociado a él.

La integración tendrá un carácter exclusivamente estructural. No se agregarán conjuntamente las observaciones de origen y destino ni se recalcularán métricas, preservando así los resultados obtenidos previamente para cada función operacional.

In [33]:
# ---------------------------------------------------------
# 1. Recuperar los productos geográficos persistidos
# ---------------------------------------------------------

origin_airport_source = pd.read_parquet(
    analysis_product_paths["origin_airport"]
)

destination_airport_source = pd.read_parquet(
    analysis_product_paths["destination_airport"]
)


# ---------------------------------------------------------
# 2. Definir los indicadores geográficos relevantes
# ---------------------------------------------------------

geographic_preferred_columns = [
    "flights",
    "population_share",
    "observed_delays",
    "alerts",
    "observed_delay_rate",
    "alert_rate",
    "recall",
    "precision",
    "false_alert_share",
    "alert_delay_lift",
    "TP",
    "FP",
    "TN",
    "FN",
]

origin_metric_columns = [
    column
    for column in geographic_preferred_columns
    if column in origin_airport_source.columns
]

destination_metric_columns = [
    column
    for column in geographic_preferred_columns
    if column in destination_airport_source.columns
]


# ---------------------------------------------------------
# 3. Preparar la perspectiva de aeropuerto de origen
# ---------------------------------------------------------

origin_airport_visualization = (
    origin_airport_source[
        ["ORIGIN"] + origin_metric_columns
    ]
    .copy()
    .rename(
        columns={
            "ORIGIN": "airport",
        }
    )
)

origin_airport_visualization.insert(
    0,
    "airport_role",
    "Origen",
)


# ---------------------------------------------------------
# 4. Preparar la perspectiva de aeropuerto de destino
# ---------------------------------------------------------

destination_airport_visualization = (
    destination_airport_source[
        ["DEST"] + destination_metric_columns
    ]
    .copy()
    .rename(
        columns={
            "DEST": "airport",
        }
    )
)

destination_airport_visualization.insert(
    0,
    "airport_role",
    "Destino",
)


# ---------------------------------------------------------
# 5. Validar la compatibilidad estructural
# ---------------------------------------------------------

geographic_columns_compatible = (
    list(origin_airport_visualization.columns)
    == list(destination_airport_visualization.columns)
)

if not geographic_columns_compatible:
    raise ValueError(
        "Los productos de origen y destino no presentan "
        "una estructura compatible para su integración."
    )


# ---------------------------------------------------------
# 6. Integrar ambas funciones operacionales
# ---------------------------------------------------------

airport_visualization = pd.concat(
    [
        origin_airport_visualization,
        destination_airport_visualization,
    ],
    ignore_index=True,
)

airport_visualization = (
    airport_visualization
    .sort_values(
        [
            "airport",
            "airport_role",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 7. Analizar los valores ausentes del producto integrado
# ---------------------------------------------------------

airport_missing_summary = (
    airport_visualization
    .isna()
    .sum()
    .loc[
        lambda series: series > 0
    ]
    .rename("missing_values")
    .reset_index()
    .rename(
        columns={
            "index": "column",
        }
    )
)

airport_identifier_missing = (
    airport_visualization[
        [
            "airport",
            "airport_role",
        ]
    ]
    .isna()
    .sum()
    .sum()
)

airport_duplicate_pairs = int(
    airport_visualization[
        [
            "airport",
            "airport_role",
        ]
    ]
    .duplicated()
    .sum()
)


# ---------------------------------------------------------
# 8. Validar la granularidad geográfica
# ---------------------------------------------------------

origin_airports_valid = (
    len(origin_airport_visualization) == 365
    and origin_airport_visualization[
        "airport"
    ].nunique() == 365
)

destination_airports_valid = (
    len(destination_airport_visualization) == 365
    and destination_airport_visualization[
        "airport"
    ].nunique() == 365
)

airport_visualization_valid = (
    len(airport_visualization) == 730
    and airport_duplicate_pairs == 0
    and airport_identifier_missing == 0
)


# ---------------------------------------------------------
# 9. Validar el volumen de cada función operacional
# ---------------------------------------------------------

origin_volume_valid = (
    int(
        origin_airport_visualization[
            "flights"
        ].sum()
    )
    == total_flights
)

destination_volume_valid = (
    int(
        destination_airport_visualization[
            "flights"
        ].sum()
    )
    == total_flights
)


# ---------------------------------------------------------
# 10. Construir una vista de los aeropuertos de mayor volumen
# ---------------------------------------------------------

top_airport_visualization = (
    airport_visualization
    .sort_values(
        "flights",
        ascending=False,
    )
    .head(20)
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 11. Mostrar el producto y las validaciones
# ---------------------------------------------------------

print(
    "Vista de las combinaciones aeropuerto-función "
    "con mayor volumen:"
)

display(
    top_airport_visualization
)

print(
    "\nValores ausentes por columna:"
)

if airport_missing_summary.empty:
    print(
        "No se detectaron valores ausentes."
    )
else:
    display(
        airport_missing_summary
    )

print(
    "\nEstructuras de origen y destino compatibles:",
    geographic_columns_compatible,
)

print(
    "Aeropuertos de origen válidos:",
    origin_airports_valid,
)

print(
    "Aeropuertos de destino válidos:",
    destination_airports_valid,
)

print(
    "Pares aeropuerto-función duplicados:",
    airport_duplicate_pairs,
)

print(
    "Identificadores geográficos ausentes:",
    airport_identifier_missing,
)

print(
    "Producto geográfico integrado válido:",
    airport_visualization_valid,
)

print(
    "Volumen de origen consistente:",
    origin_volume_valid,
)

print(
    "Volumen de destino consistente:",
    destination_volume_valid,
)

print(
    "\nDimensiones del producto geográfico:",
    airport_visualization.shape,
)

print(
    "\nColumnas del producto geográfico:",
    list(
        airport_visualization.columns
    ),
)


Vista de las combinaciones aeropuerto-función con mayor volumen:


,airport_role,airport,flights,population_share,observed_delays,alerts,observed_delay_rate,alert_rate,recall,precision,false_alert_share,alert_delay_lift,TP,FP,TN,FN
0,Origen,ORD,155123,0.05,45989,119851,0.296468,0.772619,0.850421,0.326322,0.673678,1.100699,39110,80741,28393,6879
1,Destino,ORD,154482,0.049794,40662,103732,0.263215,0.671483,0.785770,0.308015,0.691985,1.170202,31951,71781,42039,8711
2,Origen,ATL,139855,0.045079,30230,95656,0.216152,0.683966,0.794046,0.250941,0.749059,1.160944,24004,71652,37973,6226
3,Destino,ATL,139815,0.045066,24517,73623,0.175353,0.526574,0.675287,0.224875,0.775125,1.282414,16556,57067,58231,7961
4,Origen,DEN,125889,0.040577,25266,104213,0.200701,0.827817,0.904298,0.219243,0.780757,1.09239,22848,81365,19258,2418
5,Destino,DEN,125866,0.04057,22809,80694,0.181217,0.64111,0.786488,0.222309,0.777691,1.226759,17939,62755,40302,4870
6,Origen,DFW,120536,0.038852,30474,103133,0.252821,0.85562,0.925510,0.273472,0.726528,1.081684,28204,74929,15133,2270
7,Destino,DFW,120114,0.038716,24427,84942,0.203365,0.707178,0.827691,0.238021,0.761979,1.170413,20218,64724,30963,4209
8,Destino,CLT,92740,0.029893,15088,54427,0.162691,0.586877,0.729388,0.202197,0.797803,1.242828,11005,43422,34230,4083
9,Origen,CLT,92686,0.029875,18794,68664,0.202771,0.740824,0.850750,0.232859,0.767141,1.148384,15989,52675,21217,2805



Valores ausentes por columna:


,column,missing_values
0,precision,14
1,false_alert_share,14
2,alert_delay_lift,14



Estructuras de origen y destino compatibles: True
Aeropuertos de origen válidos: True
Aeropuertos de destino válidos: True
Pares aeropuerto-función duplicados: 0
Identificadores geográficos ausentes: 0
Producto geográfico integrado válido: True
Volumen de origen consistente: True
Volumen de destino consistente: True

Dimensiones del producto geográfico: (730, 16)

Columnas del producto geográfico: ['airport_role', 'airport', 'flights', 'population_share', 'observed_delays', 'alerts', 'observed_delay_rate', 'alert_rate', 'recall', 'precision', 'false_alert_share', 'alert_delay_lift', 'TP', 'FP', 'TN', 'FN']


#### Interpretación

El producto geográfico integrado quedó correctamente preparado y validado, con 730 observaciones correspondientes a 365 aeropuertos considerados separadamente como origen y destino.

Las validaciones confirman la ausencia de pares aeropuerto-función duplicados, la integridad de los identificadores geográficos y la reconstrucción completa de los 3,102,447 vuelos tanto desde la perspectiva de origen como de destino.

Se detectaron 14 combinaciones con valores no definidos en precisión, proporción de falsas alertas y lift. Estos casos se mantienen como valores ausentes porque corresponden a situaciones en las que la métrica no puede calcularse válidamente, evitando sustituirlas por valores artificiales.

La estructura resultante permite analizar en Tableau el comportamiento del sistema por aeropuerto conservando explícitamente su función operacional como origen o destino.

### 6.5 Preparación de indicadores de concentración de errores

La caracterización operacional mostró que los falsos positivos y falsos negativos no se distribuyen uniformemente entre las distintas categorías analizadas. Para facilitar su representación conjunta, las tablas de concentración de errores por franja horaria, aerolínea y aeropuerto de origen se transformarán en una estructura común.

Cada fila representará una categoría dentro de una dimensión operacional concreta. La dimensión permitirá distinguir si la categoría corresponde a una franja horaria, una aerolínea o un aeropuerto, mientras que una variable común almacenará su identificador. De esta forma será posible utilizar un mismo producto para comparar la distribución de los errores sin confundir categorías pertenecientes a niveles diferentes.

Se conservarán tanto los recuentos absolutos de falsos positivos y falsos negativos como su participación sobre el total global de cada tipo de error. Esta combinación es necesaria porque permite diferenciar el volumen absoluto de errores de su concentración relativa dentro del sistema.

La transformación será exclusivamente estructural y reutilizará los resultados previamente calculados. No se modificarán las definiciones de falso positivo o falso negativo ni se introducirán nuevas métricas de evaluación.

In [34]:
# ---------------------------------------------------------
# 1. Recuperar los productos de concentración de errores
# ---------------------------------------------------------

departure_errors_source = pd.read_parquet(
    analysis_product_paths["departure_errors"]
)

carrier_errors_source = pd.read_parquet(
    analysis_product_paths["carrier_errors"]
)

origin_errors_source = pd.read_parquet(
    analysis_product_paths["origin_errors"]
)


# ---------------------------------------------------------
# 2. Preparar los errores por franja horaria
# ---------------------------------------------------------

departure_errors_visualization = (
    departure_errors_source[
        [
            "DEP_TIME_BLK",
            "FP",
            "FN",
            "fp_share",
            "fn_share",
        ]
    ]
    .copy()
    .rename(
        columns={
            "DEP_TIME_BLK": "category",
        }
    )
)

departure_errors_visualization.insert(
    0,
    "dimension",
    "Franja horaria",
)

departure_errors_visualization[
    "category_order"
] = (
    departure_errors_visualization[
        "category"
    ]
    .str.split("-")
    .str[0]
    .astype(int)
)


# ---------------------------------------------------------
# 3. Preparar los errores por aerolínea
# ---------------------------------------------------------

carrier_errors_visualization = (
    carrier_errors_source[
        [
            "OP_UNIQUE_CARRIER",
            "FP",
            "FN",
            "fp_share",
            "fn_share",
        ]
    ]
    .copy()
    .rename(
        columns={
            "OP_UNIQUE_CARRIER": "category",
        }
    )
)

carrier_errors_visualization.insert(
    0,
    "dimension",
    "Aerolínea operadora",
)

carrier_errors_visualization[
    "total_errors"
] = (
    carrier_errors_visualization["FP"]
    + carrier_errors_visualization["FN"]
)

carrier_errors_visualization = (
    carrier_errors_visualization
    .sort_values(
        "total_errors",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)

carrier_errors_visualization[
    "category_order"
] = (
    carrier_errors_visualization.index
    + 1
)

carrier_errors_visualization.drop(
    columns="total_errors",
    inplace=True,
)


# ---------------------------------------------------------
# 4. Preparar los errores por aeropuerto de origen
# ---------------------------------------------------------

origin_errors_visualization = (
    origin_errors_source[
        [
            "ORIGIN",
            "FP",
            "FN",
            "fp_share",
            "fn_share",
        ]
    ]
    .copy()
    .rename(
        columns={
            "ORIGIN": "category",
        }
    )
)

origin_errors_visualization.insert(
    0,
    "dimension",
    "Aeropuerto de origen",
)

origin_errors_visualization[
    "total_errors"
] = (
    origin_errors_visualization["FP"]
    + origin_errors_visualization["FN"]
)

origin_errors_visualization = (
    origin_errors_visualization
    .sort_values(
        "total_errors",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)

origin_errors_visualization[
    "category_order"
] = (
    origin_errors_visualization.index
    + 1
)

origin_errors_visualization.drop(
    columns="total_errors",
    inplace=True,
)


# ---------------------------------------------------------
# 5. Homogeneizar el orden de las columnas
# ---------------------------------------------------------

error_visualization_columns = [
    "dimension",
    "category",
    "category_order",
    "FP",
    "FN",
    "fp_share",
    "fn_share",
]

departure_errors_visualization = (
    departure_errors_visualization[
        error_visualization_columns
    ]
)

carrier_errors_visualization = (
    carrier_errors_visualization[
        error_visualization_columns
    ]
)

origin_errors_visualization = (
    origin_errors_visualization[
        error_visualization_columns
    ]
)


# ---------------------------------------------------------
# 6. Integrar las tres dimensiones operacionales
# ---------------------------------------------------------

error_concentration_visualization = pd.concat(
    [
        departure_errors_visualization,
        carrier_errors_visualization,
        origin_errors_visualization,
    ],
    ignore_index=True,
)


# ---------------------------------------------------------
# 7. Validar la granularidad del producto integrado
# ---------------------------------------------------------

expected_error_rows = (
    len(departure_errors_source)
    + len(carrier_errors_source)
    + len(origin_errors_source)
)

error_rows_valid = (
    len(error_concentration_visualization)
    == expected_error_rows
)

error_categories_valid = (
    error_concentration_visualization[
        [
            "dimension",
            "category",
        ]
    ]
    .duplicated()
    .sum()
    == 0
)

error_missing_valid = (
    error_concentration_visualization[
        [
            "dimension",
            "category",
            "category_order",
            "FP",
            "FN",
            "fp_share",
            "fn_share",
        ]
    ]
    .isna()
    .sum()
    .sum()
    == 0
)


# ---------------------------------------------------------
# 8. Validar la reconstrucción de FP y FN por dimensión
# ---------------------------------------------------------

error_dimension_validation = (
    error_concentration_visualization
    .groupby(
        "dimension",
        as_index=False,
    )
    .agg(
        categories=("category", "nunique"),
        total_fp=("FP", "sum"),
        total_fn=("FN", "sum"),
        fp_share_sum=("fp_share", "sum"),
        fn_share_sum=("fn_share", "sum"),
    )
)

error_dimension_validation[
    "fp_total_valid"
] = (
    error_dimension_validation[
        "total_fp"
    ]
    == fp
)

error_dimension_validation[
    "fn_total_valid"
] = (
    error_dimension_validation[
        "total_fn"
    ]
    == fn
)

error_dimension_validation[
    "fp_share_valid"
] = np.isclose(
    error_dimension_validation[
        "fp_share_sum"
    ],
    1.0,
)

error_dimension_validation[
    "fn_share_valid"
] = np.isclose(
    error_dimension_validation[
        "fn_share_sum"
    ],
    1.0,
)

error_totals_valid = bool(
    error_dimension_validation[
        [
            "fp_total_valid",
            "fn_total_valid",
            "fp_share_valid",
            "fn_share_valid",
        ]
    ]
    .all()
    .all()
)


# ---------------------------------------------------------
# 9. Construir una vista resumida para inspección
# ---------------------------------------------------------

top_error_visualization = (
    error_concentration_visualization
    .assign(
        total_errors=lambda data: (
            data["FP"]
            + data["FN"]
        )
    )
    .sort_values(
        [
            "dimension",
            "total_errors",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        "dimension",
        as_index=False,
        group_keys=False,
    )
    .head(5)
    .drop(
        columns="total_errors"
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# 10. Mostrar el producto y las validaciones
# ---------------------------------------------------------

print(
    "Principales concentraciones de errores "
    "por dimensión:"
)

display(
    top_error_visualization
)

print(
    "\nValidación por dimensión:"
)

display(
    error_dimension_validation
)

print(
    "\nNúmero esperado de filas:",
    expected_error_rows,
)

print(
    "Número obtenido de filas:",
    len(error_concentration_visualization),
)

print(
    "Número de filas válido:",
    error_rows_valid,
)

print(
    "Combinaciones dimensión-categoría válidas:",
    error_categories_valid,
)

print(
    "Ausencia de valores perdidos:",
    error_missing_valid,
)

print(
    "Reconstrucción global de errores válida:",
    error_totals_valid,
)

print(
    "\nDimensiones del producto de errores:",
    error_concentration_visualization.shape,
)

print(
    "\nColumnas del producto de errores:",
    list(
        error_concentration_visualization.columns
    ),
)

Principales concentraciones de errores por dimensión:


,dimension,category,category_order,FP,FN,fp_share,fn_share
0,Aerolínea operadora,WN,1,279553,13986,0.180001,0.106164
1,Aerolínea operadora,AA,2,225200,10453,0.145004,0.079346
2,Aerolínea operadora,DL,3,181139,21986,0.116633,0.166891
3,Aerolínea operadora,UA,4,171462,13762,0.110402,0.104464
4,Aerolínea operadora,OO,5,152513,22197,0.098201,0.168492
5,Aeropuerto de origen,ORD,1,80741,6879,0.051988,0.052217
6,Aeropuerto de origen,DEN,2,81365,2418,0.052390,0.018354
7,Aeropuerto de origen,ATL,3,71652,6226,0.046136,0.047260
8,Aeropuerto de origen,DFW,4,74929,2270,0.048246,0.017231
9,Aeropuerto de origen,CLT,5,52675,2805,0.033917,0.021292



Validación por dimensión:


,dimension,categories,total_fp,total_fn,fp_share_sum,fn_share_sum,fp_total_valid,fn_total_valid,fp_share_valid,fn_share_valid
0,Aerolínea operadora,19,1553063,131739,1.0,1.0,True,True,True,True
1,Aeropuerto de origen,365,1553063,131739,1.0,1.0,True,True,True,True
2,Franja horaria,19,1553063,131739,1.0,1.0,True,True,True,True



Número esperado de filas: 403
Número obtenido de filas: 403
Número de filas válido: True
Combinaciones dimensión-categoría válidas: True
Ausencia de valores perdidos: True
Reconstrucción global de errores válida: True

Dimensiones del producto de errores: (403, 7)

Columnas del producto de errores: ['dimension', 'category', 'category_order', 'FP', 'FN', 'fp_share', 'fn_share']


#### Interpretación

El producto de concentración de errores quedó correctamente preparado con 403 observaciones: 19 franjas horarias, 19 aerolíneas operadoras y 365 aeropuertos de origen.

Cada dimensión reconstruye de forma independiente los 1,553,063 falsos positivos y los 131,739 falsos negativos, mientras que las participaciones relativas suman 1.00 en todos los casos. Esto confirma que la integración conserva íntegramente la distribución de los errores.

La estructura común basada en dimensión, categoría, recuento y participación permitirá comparar en Tableau dónde se concentran los FP y FN, manteniendo separadas las distintas unidades operacionales.

### 6.6 Persistencia y validación consolidada de los productos de visualización

La preparación de los resultados para visualización ha generado productos diferenciados según su nivel de análisis: indicadores globales, evolución temporal, comportamiento por franja horaria y aerolínea, caracterización geográfica y concentración de errores. Una vez validadas individualmente estas estructuras, resulta necesario persistirlas como productos finales independientes del estado actual del notebook.

La persistencia permitirá que la posterior construcción de visualizaciones en Tableau utilice directamente resultados consolidados, sin necesidad de acceder nuevamente al dataset externo de 3,102,447 vuelos ni reproducir las transformaciones desarrolladas durante este notebook.

Cada archivo conservará una granularidad específica y una estructura orientada a su finalidad analítica. En particular, los indicadores globales y temporales permanecerán separados, al igual que los productos por franja horaria y aerolínea, mientras que las perspectivas compatibles de origen y destino permanecerán integradas mediante la variable que identifica la función operacional del aeropuerto.

Finalmente, se realizará una validación consolidada mediante la recuperación de todos los archivos persistidos y la comparación de sus dimensiones con los objetos originales. Esta comprobación permitirá cerrar el bloque garantizando que los productos destinados a visualización son reproducibles, íntegros y utilizables de forma independiente.

In [35]:
# ---------------------------------------------------------
# 1. Definir el directorio de productos de visualización
# ---------------------------------------------------------

visualization_results_path = (
    operational_results_path
    / "visualization"
)

visualization_results_path.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------
# 2. Definir los productos finales de visualización
# ---------------------------------------------------------

visualization_products = {
    "global_indicators_2026.parquet": global_visualization,
    "temporal_indicators_2026.parquet": temporal_visualization,
    "time_block_indicators_2026.parquet": time_block_visualization,
    "carrier_indicators_2026.parquet": carrier_visualization,
    "airport_indicators_2026.parquet": airport_visualization,
    "error_concentration_2026.parquet": error_concentration_visualization,
}


# ---------------------------------------------------------
# 3. Persistir los productos finales
# ---------------------------------------------------------

visualization_product_paths = {}

for file_name, product_data in visualization_products.items():
    product_path = (
        visualization_results_path
        / file_name
    )

    product_data.to_parquet(
        product_path,
        index=False,
    )

    visualization_product_paths[
        file_name
    ] = product_path


# ---------------------------------------------------------
# 4. Recuperar y validar los archivos persistidos
# ---------------------------------------------------------

visualization_validation_records = []

for file_name, original_data in visualization_products.items():
    product_path = visualization_product_paths[
        file_name
    ]

    recovered_data = pd.read_parquet(
        product_path
    )

    visualization_validation_records.append(
        {
            "file": file_name,
            "file_exists": product_path.exists(),
            "expected_rows": len(original_data),
            "persisted_rows": len(recovered_data),
            "expected_columns": original_data.shape[1],
            "persisted_columns": recovered_data.shape[1],
            "rows_valid": (
                len(recovered_data)
                == len(original_data)
            ),
            "columns_valid": (
                list(recovered_data.columns)
                == list(original_data.columns)
            ),
        }
    )

visualization_products_validation = pd.DataFrame(
    visualization_validation_records
)


# ---------------------------------------------------------
# 5. Construir el inventario final para Tableau
# ---------------------------------------------------------

tableau_product_inventory = pd.DataFrame(
    {
        "product": [
            "Indicadores globales",
            "Indicadores temporales",
            "Franjas horarias",
            "Aerolíneas",
            "Aeropuertos",
            "Concentración de errores",
        ],
        "file": [
            "global_indicators_2026.parquet",
            "temporal_indicators_2026.parquet",
            "time_block_indicators_2026.parquet",
            "carrier_indicators_2026.parquet",
            "airport_indicators_2026.parquet",
            "error_concentration_2026.parquet",
        ],
        "unit_of_observation": [
            "Indicador global",
            "Mes",
            "Franja horaria",
            "Aerolínea operadora",
            "Aeropuerto y función operacional",
            "Categoría dentro de una dimensión",
        ],
        "rows": [
            len(global_visualization),
            len(temporal_visualization),
            len(time_block_visualization),
            len(carrier_visualization),
            len(airport_visualization),
            len(error_concentration_visualization),
        ],
    }
)

tableau_inventory_path = (
    visualization_results_path
    / "tableau_product_inventory_2026.parquet"
)

tableau_product_inventory.to_parquet(
    tableau_inventory_path,
    index=False,
)


# ---------------------------------------------------------
# 6. Realizar la validación consolidada
# ---------------------------------------------------------

all_visualization_files_exist = bool(
    visualization_products_validation[
        "file_exists"
    ].all()
)

all_visualization_rows_valid = bool(
    visualization_products_validation[
        "rows_valid"
    ].all()
)

all_visualization_columns_valid = bool(
    visualization_products_validation[
        "columns_valid"
    ].all()
)

tableau_inventory_valid = (
    tableau_inventory_path.exists()
    and len(tableau_product_inventory) == 6
)

visualization_block_valid = (
    all_visualization_files_exist
    and all_visualization_rows_valid
    and all_visualization_columns_valid
    and tableau_inventory_valid
)


# ---------------------------------------------------------
# 7. Mostrar el inventario y la validación final
# ---------------------------------------------------------

print(
    "Inventario final de productos para visualización:"
)

display(
    tableau_product_inventory
)

print(
    "\nValidación de los productos persistidos:"
)

display(
    visualization_products_validation
)

print(
    "\nTodos los archivos existen:",
    all_visualization_files_exist,
)

print(
    "Número de registros válido:",
    all_visualization_rows_valid,
)

print(
    "Estructura de columnas válida:",
    all_visualization_columns_valid,
)

print(
    "Inventario de productos válido:",
    tableau_inventory_valid,
)

print(
    "Bloque de visualización validado:",
    visualization_block_valid,
)

print(
    "\nDirectorio final de productos:"
)

print(
    visualization_results_path
)

Inventario final de productos para visualización:


,product,file,unit_of_observation,rows
0,Indicadores globales,global_indicators_2026.parquet,Indicador global,12
1,Indicadores temporales,temporal_indicators_2026.parquet,Mes,5
2,Franjas horarias,time_block_indicators_2026.parquet,Franja horaria,19
3,Aerolíneas,carrier_indicators_2026.parquet,Aerolínea operadora,19
4,Aeropuertos,airport_indicators_2026.parquet,Aeropuerto y función operacional,730
5,Concentración de errores,error_concentration_2026.parquet,Categoría dentro de una dimensión,403



Validación de los productos persistidos:


,file,file_exists,expected_rows,persisted_rows,expected_columns,persisted_columns,rows_valid,columns_valid
0,global_indicators_2026.parquet,True,12,12,5,5,True,True
1,temporal_indicators_2026.parquet,True,5,5,22,22,True,True
2,time_block_indicators_2026.parquet,True,19,19,16,16,True,True
3,carrier_indicators_2026.parquet,True,19,19,16,16,True,True
4,airport_indicators_2026.parquet,True,730,730,16,16,True,True
5,error_concentration_2026.parquet,True,403,403,7,7,True,True



Todos los archivos existen: True
Número de registros válido: True
Estructura de columnas válida: True
Inventario de productos válido: True
Bloque de visualización validado: True

Directorio final de productos:
G:\My Drive\MASTER Big Data\TFM\results\operational_analysis\visualization


#### Interpretación

Los seis productos destinados a visualización fueron persistidos y recuperados correctamente, conservando tanto el número de observaciones como su estructura de columnas. El inventario final incluye 12 indicadores globales, 5 registros mensuales, 19 franjas horarias, 19 aerolíneas, 730 combinaciones aeropuerto-función y 403 categorías de concentración de errores.

La validación consolidada confirma que estos productos pueden utilizarse de forma independiente del dataset analítico original y sin necesidad de reproducir los cálculos desarrollados previamente. La separación por granularidad mantiene además una correspondencia clara entre cada archivo y su unidad de análisis.

Con ello queda completada la preparación de resultados para su posterior explotación en Tableau, manteniendo la trazabilidad con los indicadores obtenidos durante el análisis operativo.

## 7. Conclusiones del análisis operativo

El análisis realizado sobre el período externo comprendido entre enero y mayo de 2026 permite complementar la evaluación predictiva con una perspectiva centrada en el comportamiento operativo del sistema de alerta temprana. Una vez fijados el modelo y el threshold antes de acceder al período externo, los resultados de este notebook han permitido estudiar cómo se distribuyen las alertas, qué nivel de cobertura proporcionan y en qué contextos se concentran sus principales errores.

La interpretación conjunta resulta especialmente relevante porque una métrica agregada no describe por sí sola la utilidad práctica del sistema. La cobertura de los retrasos debe considerarse simultáneamente con la proporción de vuelos alertados, la precisión alcanzada, la frecuencia de falsas alertas y la variabilidad observada entre períodos y segmentos operacionales.

El objetivo de este bloque final es sintetizar estas evidencias sin introducir nuevas decisiones sobre el modelo. Los resultados obtenidos sobre 2026 mantienen, por tanto, un carácter de evaluación y diagnóstico posterior al test y no se utilizan para modificar el modelo, el preprocesamiento o el threshold previamente establecidos.

En este bloque se abordarán progresivamente:

1. Síntesis de los principales resultados operativos.
2. Limitaciones e implicaciones para el uso del sistema de alerta temprana.
3. Cierre del Notebook 04 y conexión con la fase de visualización.

### 7.1 Síntesis de los principales resultados operativos

La evaluación operacional sobre 3,102,447 vuelos muestra que el sistema mantiene una elevada capacidad para identificar retrasos, alcanzando una cobertura del 79.94 %. Sin embargo, esta cobertura requiere generar alertas sobre el 66.98 % de los vuelos analizados. Como consecuencia, la precisión de las alertas se sitúa en el 25.27 % y el 74.73 % de las alertas emitidas corresponden a vuelos que finalmente no presentan un retraso igual o superior a 15 minutos.

La comparación con la prevalencia observada, situada en el 21.17 %, muestra que los vuelos alertados presentan una mayor concentración de retrasos que la población general, con un lift aproximado de 1.19. Existe, por tanto, capacidad de ordenación del riesgo, aunque la separación individual es limitada. Esta conclusión es coherente con la proximidad de las distribuciones del score correspondientes a verdaderos y falsos positivos y con la elevada proporción de vuelos que deben ser alertados para alcanzar el nivel de cobertura establecido.

El comportamiento tampoco es completamente uniforme en el tiempo. El recall mensual varía entre el 72.74 % y el 85.29 %, mientras que la precisión permanece entre el 23.32 % y el 28.83 %. Estas diferencias muestran que el resultado agregado debe complementarse con una lectura temporal, aunque las evidencias disponibles no permiten atribuirlas por sí solas a deriva, deterioro del modelo o efectos estacionales.

El análisis operacional revela además una marcada dependencia de la franja horaria. Durante las horas de la tarde el sistema llega a alertar prácticamente todos los vuelos, obteniendo recalls cercanos al 100 % pero con valores de lift próximos a 1.00. En estas franjas, la elevada cobertura procede principalmente de la expansión del volumen de alertas y no de una elevada capacidad de discriminación. En sentido contrario, los falsos negativos se concentran especialmente durante las primeras horas del día, con aproximadamente el 71.96 % de ellos entre las 06:00 y las 10:59.

También se observa heterogeneidad entre aerolíneas y aeropuertos. No obstante, estas diferencias deben interpretarse conjuntamente con el volumen de operaciones de cada categoría y no constituyen evidencia causal sobre el efecto de una aerolínea o aeropuerto concreto en el comportamiento del modelo.

En conjunto, los resultados describen un sistema capaz de priorizar una parte del riesgo de retraso y alcanzar una cobertura elevada, pero con una capacidad de discriminación individual limitada y un coste considerable en términos de falsas alertas. Esta combinación constituye el principal elemento que debe considerarse al determinar su posible utilidad operacional.

### 7.2 Limitaciones e implicaciones para el uso del sistema de alerta temprana

La principal limitación operacional del sistema se encuentra en la relación entre cobertura y selectividad. El threshold seleccionado permite identificar aproximadamente el 79.94 % de los retrasos observados, pero para conseguir esta cobertura es necesario emitir alertas sobre el 66.98 % de los vuelos. Como resultado, el 74.73 % de las alertas son falsas y únicamente el 25.27 % de los vuelos alertados termina registrando un retraso igual o superior a 15 minutos.

Esta proporción de falsas alertas constituye una limitación relevante si el sistema se interpreta como un predictor individual de alta certeza. Una alerta no debe entenderse como la afirmación de que un vuelo sufrirá necesariamente un retraso, sino como la identificación de un nivel de riesgo suficientemente elevado para superar el criterio operativo establecido. La diferencia es fundamental para evitar atribuir al modelo una capacidad predictiva superior a la observada.

El sistema resulta, por tanto, más coherente con una función de detección temprana y priorización del riesgo que con una herramienta destinada a proporcionar predicciones individuales deterministas. Bajo esta perspectiva, el elevado recall permite reducir los retrasos que permanecen sin alerta, mientras que la elevada frecuencia de falsos positivos representa el principal coste operacional del criterio utilizado.

Esta limitación adquiere especial importancia en un sistema orientado al pasajero. Una frecuencia excesiva de avisos incorrectos puede reducir progresivamente la utilidad percibida y la confianza en las alertas. Por ello, el objetivo de cobertura del 80 % utilizado en este trabajo debe interpretarse como un criterio operacional definido para el experimento y no como un óptimo universal. Otros contextos de aplicación podrían justificar un equilibrio diferente entre recall y precisión en función del coste relativo asociado a una alerta innecesaria y a un retraso no detectado.

Los resultados segmentados refuerzan esta necesidad de interpretación contextual. El comportamiento varía entre meses, franjas horarias, aerolíneas y aeropuertos, y durante determinadas horas de la tarde la elevada cobertura se obtiene alertando prácticamente la totalidad de los vuelos. Estas diferencias permiten caracterizar dónde el sistema pierde cobertura o selectividad, pero no demuestran por sí mismas relaciones causales ni justifican modificaciones posteriores del modelo utilizando el período externo.

Finalmente, el análisis realizado sobre enero-mayo de 2026 debe mantenerse separado del proceso de selección. El modelo, el preprocesamiento y el threshold fueron fijados previamente, por lo que los análisis desarrollados posteriormente sobre este período tienen carácter diagnóstico. Modificar cualquiera de estos elementos a partir de los resultados observados impediría seguir considerando 2026 como una evaluación externa independiente.

En consecuencia, la utilidad del sistema debe expresarse de forma prudente: proporciona una señal anticipada que permite identificar y priorizar vuelos con mayor riesgo relativo de retraso, pero su limitada discriminación individual y la elevada proporción de falsas alertas impiden interpretarlo como un mecanismo de predicción de alta certeza.

### 7.3 Cierre del Notebook 04 y conexión con la fase de visualización

El análisis desarrollado en este notebook ha permitido trasladar la evaluación del modelo desde una perspectiva exclusivamente predictiva hacia una caracterización operacional del sistema de alerta temprana. A partir de las predicciones generadas sobre el período externo de enero a mayo de 2026, se ha estudiado la intensidad de las alertas, su composición, la distribución del score y la variabilidad del comportamiento entre períodos y segmentos operacionales.

Los resultados muestran que el sistema alcanza una cobertura elevada de los retrasos, pero esta capacidad se obtiene mediante una tasa de alerta también elevada y con una proporción considerable de falsas alertas. Esta relación constituye el principal condicionante para su posible utilización: el sistema aporta información para priorizar vuelos con mayor riesgo relativo, pero no proporciona una separación suficientemente elevada como para interpretar cada alerta como una predicción individual de alta certeza.

El análisis temporal y operacional ha permitido además identificar que este comportamiento no es uniforme. La cobertura y la selectividad varían entre meses y categorías operacionales, mientras que las franjas horarias muestran una asimetría especialmente relevante: los falsos negativos se concentran principalmente durante las primeras horas del día y, durante la tarde, el incremento de la cobertura se produce acompañado de una pérdida considerable de selectividad.

Estas evidencias no se han utilizado para modificar el modelo, el preprocesamiento ni el threshold. El período externo de 2026 mantiene así su función de evaluación independiente, mientras que los análisis segmentados realizados posteriormente deben entenderse como diagnósticos destinados a caracterizar el comportamiento del sistema y sus limitaciones.

Como resultado adicional, se han generado y validado productos específicos para la posterior fase de visualización. Estos productos recogen de forma separada los indicadores globales, temporales, horarios, por aerolínea, geográficos y de concentración de errores, manteniendo la granularidad correspondiente a cada nivel de análisis y evitando la necesidad de reprocesar el dataset analítico completo.

Con ello, el Notebook 04 queda cerrado con una caracterización operacional completa del sistema de alerta temprana. La siguiente fase del proyecto podrá centrarse en comunicar visualmente estos resultados, prestando especial atención al equilibrio entre cobertura y falsas alertas, a la variabilidad temporal y operacional y a las limitaciones que condicionan la interpretación práctica del sistema.